# FAIR-IRS: experiments for the referee reports (revision 10)

Uses only the files already cached on your Drive (scores of every trained seed, features, LDAP and answer files); nothing is downloaded. **Just choose *Runtime → Run all*** and accept the Google Drive prompt.

| Part | Question |
|---|---|
| N1 | Detection and burden for every retrained seed (seed means), a randomized global policy, convex costs, and the daily budget. |
| N2 | Timing attacks as unilateral deviations (one insider moves, everyone else keeps their behavior), with an oracle that searches every start day, over seeds; a sample checked with full simulations. |
| N3 | Farming over seeds with late and insider-level verdicts, earlier probes, a cooldown without backfill, and three defenses. |
| N4 | Per-employee-baseline detectors: burden, farming and timing. |
| N5 | Power of the permutation test. |
| N6 | Insiders whose episodes cross the training/test split. |

**Runtime type:** CPU, High-RAM if available. **Time:** about four to six hours. If Colab disconnects, choose *Run all* again: finished parts are reloaded.

**Output:** `MyDrive/FAIR-IRS_results/revision10/fairirs_revision10_results.zip`. Send that zip back.

In [ ]:
# ===== 1. Configuration (nothing to change) =====
import os, sys
IN_COLAB = 'google.colab' in sys.modules
OUT = os.environ.get('FAIRIRS_OUT', '/content/drive/MyDrive/FAIR-IRS_results')
if IN_COLAB and 'FAIRIRS_OUT' not in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
BUDGET = float(os.environ.get('FAIRIRS_BUDGET', 0.01))
DELTA = round(0.2 * BUDGET, 6)
MAIN_GROUP = 'g_functional_unit'
CACHE = os.path.join(OUT, 'cache')
REV = os.path.join(OUT, 'revision10')
MAIN, VERSION = 'r4.2', '4.2'
NEW, NEW_VERSION = 'r5.2', '5.2'
MIN_GROUP = int(os.environ.get('FAIRIRS_MIN_GROUP', 20))
ANSWERS = os.environ.get('FAIRIRS_ANSWERS', os.path.join(OUT, 'answers'))
RAW = os.environ.get('FAIRIRS_RAW', '/content/cert_raw')
os.makedirs(REV, exist_ok=True)
for f in ('scored_r4.2_v2.pkl', 'scored_r5.2_v2.pkl'):
    if not os.path.exists(os.path.join(CACHE, f)):
        raise FileNotFoundError(f'{f} not found in {CACHE}. Run the earlier notebooks first.')
print('Using cache in', CACHE, '| results ->', REV)

## Experiment code
These cells write the code files. You don't need to edit them.

In [ ]:
%%writefile fairirs.py
"""
fairirs.py -- Experiments for
the FAIR-IRS project: burden and evasion in
history-dependent insider-risk alerting

Pipeline
  1. load CERT release (r4.2 / r6.2 formats) + answers  -> user-day features, labels
  2. groups (LDAP structural groups, behavioural groups, semi-synthetic attribute)
  3. detectors (IF, AE, LR, DP-LR, GBM, per-group IF, plug-in scores)
  4. alert policies (global top-k, quota, FAIR-IRS allocation, + scheduling)
  5. metrics (detection, burden, exposure bias, Lemma 1 check, DP certificates)
  6. simulations (analyst noise, discount farming, exposure injection, kappa sweep)

Only numpy / pandas / scipy / scikit-learn / matplotlib are required.
"""
from __future__ import annotations

import csv
import glob
import math
import os
import re
import time
from dataclasses import dataclass, field
from functools import partial

import numpy as np
import pandas as pd
from scipy.special import gammaln, logsumexp
from scipy.stats import spearmanr

# =============================================================================
# 0. Utilities
# =============================================================================

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def find_files(root, name):
    """Case-insensitive recursive search for a file name."""
    out = []
    for dp, _, fns in os.walk(root):
        for fn in fns:
            if fn.lower() == name.lower():
                out.append(os.path.join(dp, fn))
    return sorted(out)


def find_dir(root, name):
    for dp, dns, _ in os.walk(root):
        for d in dns:
            if d.lower() == name.lower():
                return os.path.join(dp, d)
    return None


DATE_FMT = "%m/%d/%Y %H:%M:%S"
DATE_RE = re.compile(r"\d{1,2}/\d{1,2}/\d{4} \d{1,2}:\d{2}:\d{2}")

# Expected columns when a CERT file has no header (keyed by number of fields)
NOHEADER_SCHEMAS = {
    "logon": {5: ["id", "date", "user", "pc", "activity"]},
    "device": {5: ["id", "date", "user", "pc", "activity"],
               6: ["id", "date", "user", "pc", "file_tree", "activity"]},
    "file": {6: ["id", "date", "user", "pc", "filename", "content"],
             9: ["id", "date", "user", "pc", "filename", "activity",
                 "to_removable_media", "from_removable_media", "content"]},
    "email": {11: ["id", "date", "user", "pc", "to", "cc", "bcc", "from",
                   "size", "attachments", "content"],
              12: ["id", "date", "user", "pc", "to", "cc", "bcc", "from",
                   "activity", "size", "attachments", "content"]},
    "http": {6: ["id", "date", "user", "pc", "url", "content"],
             7: ["id", "date", "user", "pc", "url", "activity", "content"]},
}

# columns we actually need per source (others are skipped while parsing)
NEEDED = {
    "logon": ["date", "user", "pc", "activity"],
    "device": ["date", "user", "activity"],
    "file": ["date", "user", "filename", "activity", "to_removable_media"],
    "email": ["date", "user", "to", "cc", "bcc", "activity", "size", "attachments"],
    "http": ["date", "user", "url", "activity"],
}


def _sniff_columns(path, kind):
    with open(path, "r", errors="ignore", newline="") as f:
        first = next(csv.reader([f.readline()]))
    if first and first[0].strip().lower() == "id":
        return [c.strip().lower() for c in first], 0
    names = NOHEADER_SCHEMAS[kind].get(len(first))
    if names is None:
        raise ValueError(f"Cannot infer schema of {path} ({len(first)} fields)")
    return names, None


def read_chunks(path, kind, chunksize):
    names, header = _sniff_columns(path, kind)
    usecols = [c for c in NEEDED[kind] if c in names]
    kw = dict(usecols=usecols, dtype=str, chunksize=chunksize,
              on_bad_lines="skip", encoding_errors="ignore")
    if header is None:
        kw.update(header=None, names=names)
    else:
        kw.update(header=0)
        # normalise header case
        kw["usecols"] = lambda c: c.strip().lower() in usecols
    for ch in pd.read_csv(path, **kw):
        ch.columns = [c.strip().lower() for c in ch.columns]
        ch["ts"] = pd.to_datetime(ch["date"], format=DATE_FMT, errors="coerce")
        ch = ch.dropna(subset=["ts", "user"])
        ch["day"] = ch["ts"].dt.normalize()
        h = ch["ts"].dt.hour
        ch["ah"] = ((h < 8) | (h >= 18) | (ch["ts"].dt.dayofweek >= 5)).astype(np.int32)
        yield ch


# =============================================================================
# 1. Feature extraction (user-day)
# =============================================================================

JOB_RE = r"job|career|monster|indeed|simplyhired|hire|recruit|linkedin|resume"
LEAK_RE = r"wikileaks|leak"
CLOUD_RE = r"dropbox|drive\.google|box\.com|onedrive|mega\.|wetransfer|4shared|mediafire"

COUNT_FEATURES = [
    "logon_n", "logoff_n", "logon_ah", "pc_n",
    "dev_n", "dev_ah",
    "file_n", "file_ah", "file_exe", "file_zip", "file_doc", "file_pdf",
    "file_txt", "file_jpg", "file_usb",
    "email_n", "email_ah", "email_ext", "email_att", "email_bcc", "email_kb",
    "http_n", "http_ah", "http_job", "http_leak", "http_cloud", "http_upload",
]
VOLUME_FEATURES = ["logon_n", "logoff_n", "dev_n", "file_n", "email_n", "http_n"]


def _agg(df, cols):
    return df.groupby(["user", "day"])[cols].sum()


def feats_logon(ch, pcs_acc):
    a = ch["activity"].str.lower().fillna("")
    d = pd.DataFrame({"user": ch["user"], "day": ch["day"],
                      "logon_n": (a == "logon").astype(np.int32),
                      "logoff_n": (a == "logoff").astype(np.int32),
                      "logon_ah": ((a == "logon") & (ch["ah"] == 1)).astype(np.int32)})
    pcs_acc.append(ch[["user", "day", "pc"]].drop_duplicates())
    return _agg(d, ["logon_n", "logoff_n", "logon_ah"])


def feats_device(ch):
    a = ch["activity"].str.lower().fillna("") if "activity" in ch else pd.Series("connect", index=ch.index)
    c = (a == "connect").astype(np.int32)
    d = pd.DataFrame({"user": ch["user"], "day": ch["day"], "dev_n": c, "dev_ah": c * ch["ah"]})
    return _agg(d, ["dev_n", "dev_ah"])


def feats_file(ch):
    fn = ch["filename"].fillna("").str.lower()
    ext = fn.str.extract(r"\.([a-z0-9]{1,4})$", expand=False).fillna("")
    d = pd.DataFrame({"user": ch["user"], "day": ch["day"],
                      "file_n": 1, "file_ah": ch["ah"],
                      "file_exe": (ext == "exe").astype(np.int32),
                      "file_zip": ext.isin(["zip", "rar", "7z"]).astype(np.int32),
                      "file_doc": ext.isin(["doc", "docx"]).astype(np.int32),
                      "file_pdf": (ext == "pdf").astype(np.int32),
                      "file_txt": (ext == "txt").astype(np.int32),
                      "file_jpg": ext.isin(["jpg", "jpeg", "png"]).astype(np.int32)})
    if "to_removable_media" in ch:
        d["file_usb"] = (ch["to_removable_media"].fillna("").str.lower() == "true").astype(np.int32)
    else:
        d["file_usb"] = 0
    return _agg(d, ["file_n", "file_ah", "file_exe", "file_zip", "file_doc",
                    "file_pdf", "file_txt", "file_jpg", "file_usb"])


def feats_email(ch):
    if "activity" in ch:
        ch = ch[ch["activity"].fillna("").str.lower().str.contains("send")]
    col = lambda c: ch[c].fillna("") if c in ch else pd.Series("", index=ch.index)
    rec = col("to") + ";" + col("cc") + ";" + col("bcc")
    ext = rec.str.contains(r"@(?!dtaa\.com)", regex=True).astype(np.int32)
    att_raw = ch["attachments"].fillna("") if "attachments" in ch else pd.Series("", index=ch.index)
    att_num = pd.to_numeric(att_raw, errors="coerce")
    att_cnt = np.where(att_num.notna(), att_num.fillna(0),
                       np.where(att_raw.str.len() > 0, att_raw.str.count(";") + 1, 0))
    size = pd.to_numeric(ch["size"], errors="coerce").fillna(0) if "size" in ch else 0
    d = pd.DataFrame({"user": ch["user"], "day": ch["day"], "email_n": 1,
                      "email_ah": ch["ah"], "email_ext": ext,
                      "email_att": att_cnt.astype(np.float64),
                      "email_bcc": (col("bcc").str.len() > 0).astype(np.int32),
                      "email_kb": np.asarray(size) / 1000.0})
    return _agg(d, ["email_n", "email_ah", "email_ext", "email_att", "email_bcc", "email_kb"])


def feats_http(ch):
    url = ch["url"].fillna("").str.lower()
    d = pd.DataFrame({"user": ch["user"], "day": ch["day"], "http_n": 1,
                      "http_ah": ch["ah"],
                      "http_job": url.str.contains(JOB_RE, regex=True).astype(np.int32),
                      "http_leak": url.str.contains(LEAK_RE, regex=True).astype(np.int32),
                      "http_cloud": url.str.contains(CLOUD_RE, regex=True).astype(np.int32)})
    if "activity" in ch:
        d["http_upload"] = ch["activity"].fillna("").str.lower().str.contains("upload").astype(np.int32)
    else:
        d["http_upload"] = 0
    return _agg(d, ["http_n", "http_ah", "http_job", "http_leak", "http_cloud", "http_upload"])


def extract_features(release_dir, use_http=True, chunksize=2_000_000):
    """Stream every CERT log file of one release and aggregate per user-day."""
    parts = []
    pcs_acc = []
    sources = ["logon", "device", "file", "email"] + (["http"] if use_http else [])
    for kind in sources:
        paths = find_files(release_dir, f"{kind}.csv")
        if not paths:
            log(f"[warn] {kind}.csv not found under {release_dir}; skipping")
            continue
        path = paths[0]
        log(f"reading {kind} from {path} ({os.path.getsize(path)/1e9:.2f} GB)")
        acc = []
        for i, ch in enumerate(read_chunks(path, kind, chunksize)):
            if kind == "logon":
                acc.append(feats_logon(ch, pcs_acc))
            elif kind == "device":
                acc.append(feats_device(ch))
            elif kind == "file":
                acc.append(feats_file(ch))
            elif kind == "email":
                acc.append(feats_email(ch))
            else:
                acc.append(feats_http(ch))
            if i % 5 == 4:
                log(f"  {kind}: {(i+1)*chunksize:,} rows")
        if acc:
            parts.append(pd.concat(acc).groupby(level=[0, 1]).sum())
    if not parts:
        raise FileNotFoundError(f"No CERT log files (logon.csv, ...) found under {release_dir}")
    F = pd.concat(parts, axis=1).fillna(0)
    if pcs_acc:
        pcs = pd.concat(pcs_acc).drop_duplicates().groupby(["user", "day"]).size().rename("pc_n")
        F = F.join(pcs, how="left")
    for c in COUNT_FEATURES:
        if c not in F:
            F[c] = 0.0
    F = F[COUNT_FEATURES].fillna(0).astype(np.float32).reset_index()
    F["exposure"] = F[VOLUME_FEATURES].sum(axis=1)
    F["weekend"] = (F["day"].dt.dayofweek >= 5).astype(np.int8)
    return F


# =============================================================================
# 2. LDAP, psychometrics, labels
# =============================================================================

def load_ldap(release_dir):
    d = find_dir(release_dir, "LDAP")
    if d is None:
        raise FileNotFoundError(f"No LDAP folder under {release_dir}")
    rows = []
    for p in sorted(glob.glob(os.path.join(d, "*.csv"))):
        m = re.search(r"(\d{4}-\d{2})", os.path.basename(p))
        if not m:
            continue
        df = pd.read_csv(p, dtype=str)
        df.columns = [c.strip().lower() for c in df.columns]
        df["month"] = m.group(1)
        rows.append(df)
    L = pd.concat(rows, ignore_index=True).rename(columns={"user_id": "user"})
    for c in ["role", "business_unit", "functional_unit", "department", "team"]:
        if c not in L:
            L[c] = "NA"
        L[c] = L[c].fillna("NA").astype(str)
    return L[["user", "month", "role", "business_unit", "functional_unit", "department", "team"]]


def load_answers(answers_root, version):
    """Return (insiders table, set of malicious (user, day) pairs).

    Uses answers/insiders.csv. Malicious days are taken from the per-insider
    detail files when they can be found; otherwise every day between start
    and end is labelled malicious.
    """
    ins_paths = find_files(answers_root, "insiders.csv")
    if not ins_paths:
        raise FileNotFoundError(f"insiders.csv not found under {answers_root}")
    ins = pd.read_csv(ins_paths[0], dtype=str)
    ins.columns = [c.strip().lower() for c in ins.columns]
    v = float(str(version).lower().replace("r", ""))
    ins = ins[pd.to_numeric(ins["dataset"], errors="coerce").round(1) == round(v, 1)].copy()
    if ins.empty:
        raise ValueError(f"No insiders for dataset {version} in {ins_paths[0]}")
    ins["start"] = pd.to_datetime(ins["start"], format=DATE_FMT, errors="coerce")
    ins["end"] = pd.to_datetime(ins["end"], format=DATE_FMT, errors="coerce")
    ans_dir = os.path.dirname(ins_paths[0])
    # index all detail files once
    detail_index = {}
    for dp, _, fns in os.walk(ans_dir):
        for fn in fns:
            detail_index[fn.lower()] = os.path.join(dp, fn)
    mal = set()
    n_from_details = 0
    for _, r in ins.iterrows():
        p = detail_index.get(str(r.get("details", "")).lower())
        days = set()
        if p:
            with open(p, "r", errors="ignore", newline="") as f:
                for row in csv.reader(f):
                    for fld in row[:4]:
                        if DATE_RE.fullmatch(fld.strip()):
                            days.add(pd.Timestamp(pd.to_datetime(fld.strip(), format=DATE_FMT).normalize()))
                            break
        if days:
            n_from_details += 1
        elif pd.notna(r["start"]) and pd.notna(r["end"]):
            days = set(pd.date_range(r["start"].normalize(), r["end"].normalize(), freq="D"))
        for d in days:
            mal.add((r["user"], d))
    log(f"answers: {len(ins)} insiders for r{version}; {n_from_details} with activity-level detail files; "
        f"{len(mal)} malicious user-days")
    return ins, mal


@dataclass
class Dataset:
    name: str
    df: pd.DataFrame            # one row per active user-day
    insiders: pd.DataFrame
    split_day: pd.Timestamp
    group_cols: list = field(default_factory=list)


def build_dataset(name, release_dir, answers_root, version, cache_dir,
                  use_http=True, train_frac=0.5, min_group=20, chunksize=2_000_000,
                  seed=0):
    os.makedirs(cache_dir, exist_ok=True)
    fpath = os.path.join(cache_dir, f"features_{name}.pkl")
    if os.path.exists(fpath):
        log(f"loading cached features {fpath}")
        F = pd.read_pickle(fpath)
    else:
        F = extract_features(release_dir, use_http=use_http, chunksize=chunksize)
        F.to_pickle(fpath)
        log(f"cached features -> {fpath}")
    F = F[F["exposure"] > 0].copy()

    # labels
    ins, mal = load_answers(answers_root, version)
    key = pd.MultiIndex.from_arrays([F["user"], F["day"]])
    F["y"] = key.isin(list(mal)).astype(np.int8)

    # LDAP attributes per month (cached, so raw files can be deleted), fallback to user's modal value
    lpath = os.path.join(cache_dir, f"ldap_{name}.pkl")
    if os.path.exists(lpath):
        L = pd.read_pickle(lpath)
    else:
        L = load_ldap(release_dir)
        L.to_pickle(lpath)
    F["month"] = F["day"].dt.strftime("%Y-%m")
    F = F.merge(L, on=["user", "month"], how="left")
    mode = L.groupby("user")[["role", "business_unit", "functional_unit", "department", "team"]] \
            .agg(lambda s: s.value_counts().index[0])
    for c in mode.columns:
        F[c] = F[c].fillna(F["user"].map(mode[c])).fillna("NA")

    # temporal split
    days = np.sort(F["day"].unique())
    split_day = pd.Timestamp(days[int(len(days) * train_frac)])
    F["train"] = F["day"] < split_day

    # behavioural / derived groups, computed on the training period only
    tr = F[F["train"]]
    ah_share = (tr.groupby("user")["logon_ah"].sum() /
                tr.groupby("user")["logon_n"].sum().clip(lower=1))
    F["g_afterhours"] = np.where(F["user"].map(ah_share).fillna(0) > 0.2, "after-hours", "day")
    expo = tr.groupby("user")["exposure"].mean()
    terc = pd.qcut(expo.rank(method="first"), 3, labels=["low", "mid", "high"]).astype(str)
    F["g_exposure"] = F["user"].map(terc).fillna("mid")
    # structural groups, rare values collapsed
    for c in ["functional_unit", "department", "role"]:
        users_per = F.groupby(c)["user"].nunique()
        rare = users_per[users_per < min_group].index
        F["g_" + c] = np.where(F[c].isin(rare), "Other", F[c])
    scen = ins.drop_duplicates("user").set_index("user")["scenario"].astype(str)
    F["scenario"] = F["user"].map(scen).fillna("none")
    F = F.sort_values(["day", "user"]).reset_index(drop=True)
    group_cols = ["g_functional_unit", "g_department", "g_role", "g_afterhours", "g_exposure"]
    return Dataset(name, F, ins, split_day, group_cols)


def add_synthetic_attribute(ds: Dataset, kappa: float, seed=0, col="g_synthA"):
    """Semi-synthetic binary attribute A whose dependence on role is set by kappa.

    Roles are randomly ordered and the first roles covering ~50% of users get
    propensity 1, the rest 0. Then P(A=1 | role) = kappa * p_role + (1-kappa) * 0.5.
    kappa = 0: A independent of role; kappa = 1: A determined by role.
    """
    rng = np.random.default_rng(seed)
    users = ds.df.groupby("user")["role"].agg(lambda s: s.value_counts().index[0])
    roles = users.value_counts()
    order = rng.permutation(roles.index.values)
    cum, p_role, total = 0, {}, len(users)
    for r in order:
        p_role[r] = 1.0 if cum < total / 2 else 0.0
        cum += roles[r]
    p = users.map(p_role).values * kappa + (1 - kappa) * 0.5
    a = (rng.random(len(users)) < p).astype(int)
    amap = dict(zip(users.index, np.where(a == 1, "A=1", "A=0")))
    ds.df[col] = ds.df["user"].map(amap)
    return col


def dataset_summary(ds: Dataset):
    F = ds.df
    te = F[~F["train"]]
    tr = F[F["train"]]
    ins_users = set(F.loc[F["y"] == 1, "user"])
    return {
        "release": ds.name,
        "users": F["user"].nunique(),
        "insiders": len(ins_users),
        "insiders_train": tr.loc[tr["y"] == 1, "user"].nunique(),
        "insiders_test": te.loc[te["y"] == 1, "user"].nunique(),
        "days": F["day"].nunique(),
        "user_days": len(F),
        "malicious_user_days": int(F["y"].sum()),
        "base_rate": F["y"].mean(),
        "roles": F["role"].nunique(),
        "functional_units": F["functional_unit"].nunique(),
        "departments": F["department"].nunique(),
        "split_day": str(ds.split_day.date()),
    }


# =============================================================================
# 3. Feature transforms and detectors
# =============================================================================

class FeatureTransform:
    """'raw': log1p(counts) standardised.  'share': counts divided by daily
    exposure (removes volume -> exposure-normalisation baseline B2)."""

    def __init__(self, mode="raw"):
        self.mode = mode

    def _base(self, df):
        X = df[COUNT_FEATURES].to_numpy(np.float64)
        if self.mode == "share":
            e = df["exposure"].to_numpy(np.float64)[:, None]
            X = X / np.maximum(e, 1.0)
        else:
            X = np.log1p(X)
        return np.hstack([X, df[["weekend"]].to_numpy(np.float64)])

    def fit(self, df):
        X = self._base(df)
        self.mu = X.mean(0)
        self.sd = X.std(0) + 1e-6
        return self

    def transform(self, df):
        return ((self._base(df) - self.mu) / self.sd).astype(np.float32)


class IFDetector:
    name = "IF"
    supervised = False

    def __init__(self, seed=0, n_estimators=200, max_train=300_000):
        self.seed, self.n, self.max_train = seed, n_estimators, max_train

    def fit(self, X, y=None):
        from sklearn.ensemble import IsolationForest
        rng = np.random.default_rng(self.seed)
        if len(X) > self.max_train:
            X = X[rng.choice(len(X), self.max_train, replace=False)]
        self.m = IsolationForest(n_estimators=self.n, random_state=self.seed, n_jobs=-1).fit(X)
        return self

    def score(self, X):
        return -self.m.score_samples(X)


class AEDetector:
    """Under-complete MLP autoencoder; score = reconstruction MSE."""
    name = "AE"
    supervised = False

    def __init__(self, seed=0, hidden=(16, 6, 16), max_train=200_000, max_iter=60):
        self.seed, self.hidden, self.max_train, self.max_iter = seed, hidden, max_train, max_iter

    def fit(self, X, y=None):
        from sklearn.neural_network import MLPRegressor
        rng = np.random.default_rng(self.seed)
        if len(X) > self.max_train:
            X = X[rng.choice(len(X), self.max_train, replace=False)]
        self.m = MLPRegressor(hidden_layer_sizes=self.hidden, max_iter=self.max_iter,
                              early_stopping=True, random_state=self.seed).fit(X, X)
        return self

    def score(self, X):
        return ((self.m.predict(X) - X) ** 2).mean(1)


class LRDetector:
    name = "LR"
    supervised = True

    def fit(self, X, y):
        from sklearn.linear_model import LogisticRegression
        self.m = LogisticRegression(class_weight="balanced", max_iter=2000).fit(X, y)
        return self

    def score(self, X):
        return self.m.decision_function(X)


class GBMDetector:
    name = "GBM"
    supervised = True

    def __init__(self, seed=0):
        self.seed = seed

    def fit(self, X, y):
        from sklearn.ensemble import HistGradientBoostingClassifier
        self.m = HistGradientBoostingClassifier(class_weight="balanced", max_iter=300,
                                                learning_rate=0.05, random_state=self.seed).fit(X, y)
        return self

    def score(self, X):
        return self.m.decision_function(X)


# ---------- DP-SGD logistic regression with an RDP accountant ----------

def _rdp_sampled_gaussian(q, sigma, alpha):
    """RDP of the Poisson-subsampled Gaussian mechanism at integer order alpha
    (upper bound of Mironov, Talwar & Zhang 2019, as used by Abadi et al. 2016)."""
    if q >= 1.0:
        return alpha / (2 * sigma ** 2)
    k = np.arange(alpha + 1)
    logc = gammaln(alpha + 1) - gammaln(k + 1) - gammaln(alpha - k + 1)
    terms = logc + (alpha - k) * np.log1p(-q) + k * np.log(q) + (k * k - k) / (2 * sigma ** 2)
    return float(logsumexp(terms) / (alpha - 1))


def dp_epsilon(q, sigma, steps, delta, orders=range(2, 257)):
    return min(steps * _rdp_sampled_gaussian(q, sigma, a) + math.log(1 / delta) / (a - 1)
               for a in orders)


def calibrate_sigma(q, steps, eps, delta):
    lo, hi = 0.3, 200.0
    if dp_epsilon(q, hi, steps, delta) > eps:
        raise ValueError("epsilon too small for these settings")
    for _ in range(60):
        mid = (lo + hi) / 2
        if dp_epsilon(q, mid, steps, delta) > eps:
            lo = mid
        else:
            hi = mid
    return hi


class DPLRDetector:
    """Logistic regression trained with DP-SGD (per-example clipping + Gaussian
    noise, Poisson sampling). Guarantee is record-level (one user-day)."""
    supervised = True

    def __init__(self, eps=1.0, delta=None, epochs=10, q=0.01, clip=1.0, lr=0.5, seed=0):
        self.eps, self.delta, self.epochs, self.q = eps, delta, epochs, q
        self.clip, self.lr, self.seed = clip, lr, seed
        self.name = "SGD-LR (no noise)" if eps == np.inf else f"DP-LR(eps={eps:g})"

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        n, d = X.shape
        delta = self.delta or 1.0 / (10 * n)
        steps = int(self.epochs / self.q)
        self.sigma = 0.0 if self.eps == np.inf else calibrate_sigma(self.q, steps, self.eps, delta)
        Xb = np.hstack([X, np.ones((n, 1), np.float32)])
        pos = y.mean()
        w = np.where(y == 1, 0.5 / max(pos, 1e-9), 0.5 / max(1 - pos, 1e-9))
        w = w / w.max()                          # weights in (0,1]: clipping still bounds sensitivity
        theta = np.zeros(d + 1)
        # normalise by the expected *weighted* batch size; class counts are treated as public
        exp_batch = self.q * w.sum()
        for _ in range(steps):
            idx = np.nonzero(rng.random(n) < self.q)[0]
            g = np.zeros(d + 1)
            if len(idx):
                xb = Xb[idx]
                p = 1 / (1 + np.exp(-np.clip(xb @ theta, -30, 30)))
                G = (w[idx] * (p - y[idx]))[:, None] * xb
                norms = np.linalg.norm(G, axis=1, keepdims=True)
                G = G / np.maximum(1.0, norms / self.clip)
                g = G.sum(0)
            g = (g + rng.normal(0, self.sigma * self.clip, d + 1)) / exp_batch
            theta -= self.lr * g
        self.theta = theta
        self.eps_spent = np.inf if self.sigma == 0 else dp_epsilon(self.q, self.sigma, steps, delta)
        return self

    def score(self, X):
        return X @ self.theta[:-1] + self.theta[-1]


class LSTMDetector:
    """Sequence detector in the style of Tuor et al. (2017): an LSTM reads each
    employee's sequence of daily feature vectors and predicts the next day;
    the anomaly score of a user-day is the prediction error. Trained on
    benign training-period days only. Requires PyTorch (available on Colab)."""
    name = "LSTM"
    supervised = False
    sequence = True

    def __init__(self, seed=0, hidden=64, epochs=8, batch=64, lr=1e-3):
        self.seed, self.hidden, self.epochs, self.batch, self.lr = seed, hidden, epochs, batch, lr

    @staticmethod
    def _user_index(F):
        codes, uniq = pd.factorize(F["user"])
        order = np.lexsort((F["day"].to_numpy(), codes))
        bounds = np.flatnonzero(np.diff(codes[order])) + 1
        return np.split(order, bounds)

    def _pad(self, seqs, X, masks=None):
        L = max(len(q) for q in seqs)
        d = X.shape[1]
        xb = np.zeros((len(seqs), L, d), np.float32)
        mb = np.zeros((len(seqs), L), np.float32)
        for j, q in enumerate(seqs):
            xb[j, :len(q)] = X[q]
            mb[j, :len(q)] = 1.0 if masks is None else masks[q]
        return xb, mb

    def fit_seq(self, F, X, fit_mask, train_mask):
        import torch
        torch.manual_seed(self.seed)
        rng = np.random.default_rng(self.seed)
        d = X.shape[1]
        self.net = torch.nn.LSTM(d, self.hidden, batch_first=True)
        self.head = torch.nn.Linear(self.hidden, d)
        params = list(self.net.parameters()) + list(self.head.parameters())
        opt = torch.optim.Adam(params, lr=self.lr)
        seqs = [q[train_mask[q]] for q in self._user_index(F)]
        seqs = [q for q in seqs if len(q) >= 2]
        w = fit_mask.astype(np.float32)
        for ep in range(self.epochs):
            perm = rng.permutation(len(seqs))
            tot = 0.0
            for b0 in range(0, len(perm), self.batch):
                xb, mb = self._pad([seqs[j] for j in perm[b0:b0 + self.batch]], X, w)
                xb, mb = torch.from_numpy(xb), torch.from_numpy(mb)
                out, _ = self.net(xb[:, :-1])
                err = ((self.head(out) - xb[:, 1:]) ** 2).mean(-1)
                m = mb[:, 1:]
                loss = (err * m).sum() / m.sum().clamp(min=1.0)
                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                opt.step()
                tot += float(loss)
            log(f"  LSTM epoch {ep + 1}/{self.epochs}: loss {tot:.3f}")
        return self

    def score_seq(self, F, X):
        import torch
        raw = np.full(len(F), np.nan)
        seqs = self._user_index(F)
        with torch.no_grad():
            for b0 in range(0, len(seqs), self.batch):
                part = seqs[b0:b0 + self.batch]
                xb, _ = self._pad(part, X)
                xt = torch.from_numpy(xb)
                out, _ = self.net(xt[:, :-1])
                err = ((self.head(out) - xt[:, 1:]) ** 2).mean(-1).numpy()
                for j, q in enumerate(part):
                    if len(q) > 1:
                        raw[q[1:]] = err[j, :len(q) - 1]
        first = np.isnan(raw)
        raw[first] = np.nanmedian(raw)          # first active day of a user: no context
        return raw


class PerGroupDetector:
    """Peer-group modelling baseline (B3): one detector per group."""
    supervised = False

    def __init__(self, base_factory, groups_train, groups_score_fn, name="IF-peer"):
        self.base_factory, self.gtr, self.gfn, self.name = base_factory, groups_train, groups_score_fn, name

    def fit(self, X, y=None):
        self.models = {g: self.base_factory().fit(X[self.gtr == g]) for g in np.unique(self.gtr)}
        self.default = self.base_factory().fit(X)
        return self

    def score_groups(self, X, groups):
        s = np.empty(len(X))
        for g in np.unique(groups):
            m = self.models.get(g, self.default)
            idx = groups == g
            s[idx] = m.score(X[idx])
        return s


def to_quantile(scores, reference=None):
    """Map scores to their empirical CDF in [0,1] (detector-agnostic scale)."""
    ref = np.sort(scores if reference is None else reference)
    return np.searchsorted(ref, scores, side="right") / len(ref)


def to_quantile_ref(scores, reference_sorted):
    """Causal score scale: empirical CDF of the BENIGN TRAINING-PERIOD scores.
    Test scores above every training score would all map to 1.0; a tiny
    order-preserving term (< 1e-6) keeps them ranked by raw score."""
    scores = np.asarray(scores, float)
    q = np.searchsorted(reference_sorted, scores, side="right") / len(reference_sorted)
    order = np.argsort(np.argsort(scores, kind="stable"), kind="stable")
    return q + 1e-6 * (order + 1) / (len(scores) + 1)


def user_baseline(df, X, train_mask):
    """Mean transformed feature vector per user over the training period."""
    tmp = pd.DataFrame(X[train_mask])
    tmp["user"] = df.loc[train_mask, "user"].values
    return tmp.groupby("user").mean()


def explanations(df, X, base):
    """phi = deviation of a user-day from the user's own training baseline."""
    B = base.reindex(df["user"].values).fillna(0.0).to_numpy(np.float32)
    return X - B


@dataclass
class ScoredData:
    detector: str
    test: pd.DataFrame        # test rows with score columns
    phi: np.ndarray           # explanation vectors (aligned with test rows)
    model: object = None
    transform: object = None
    raw_ref: np.ndarray = None        # sorted benign training-period raw scores
    train_ref: pd.DataFrame = None    # benign training rows: raw score + group columns


def run_detector(ds: Dataset, det, feature_mode="raw", clean_train=True, peer_col=None,
                 phi_mode="population"):
    """phi_mode: 'population' -> explanation = standardised deviation from the
    population (training) baseline; 'self' -> deviation from the user's own
    training baseline."""
    F = ds.df
    tr_mask = F["train"].to_numpy()
    if clean_train and not det.supervised:
        tr_fit = tr_mask & (F["y"].to_numpy() == 0)
    else:
        tr_fit = tr_mask
    ft = FeatureTransform(feature_mode).fit(F[tr_fit])
    X = ft.transform(F)
    y = F["y"].to_numpy()
    if det.supervised and y[tr_fit].sum() == 0:
        log(f"[skip] {det.name}: no malicious user-days in the training period")
        return None
    t0 = time.time()
    tr_ben = tr_mask & (y == 0)
    if getattr(det, "sequence", False):
        det.fit_seq(F, X, tr_fit, tr_mask)
        raw_all = det.score_seq(F, X)
        raw, raw_tr = raw_all[~tr_mask], raw_all[tr_ben]
    elif isinstance(det, PerGroupDetector):
        det.gtr = F.loc[tr_fit, peer_col].to_numpy()
        det.fit(X[tr_fit])
        raw = det.score_groups(X[~tr_mask], F.loc[~tr_mask, peer_col].to_numpy())
        raw_tr = det.score_groups(X[tr_ben], F.loc[tr_ben, peer_col].to_numpy())
    else:
        det.fit(X[tr_fit], y[tr_fit])
        raw = det.score(X[~tr_mask])
        raw_tr = det.score(X[tr_ben])
    ref = np.sort(raw_tr)
    test = F[~tr_mask].copy().reset_index(drop=True)
    test["raw_score"] = raw
    test["score"] = to_quantile_ref(raw, ref)
    gcols = [c for c in F.columns if c.startswith("g_")]
    train_ref = F.loc[tr_ben, ["user"] + gcols].reset_index(drop=True)
    train_ref["raw_score"] = raw_tr
    if phi_mode == "self":
        phi = explanations(test, X[~tr_mask], user_baseline(F, X, tr_mask))
    else:
        phi = X[~tr_mask].copy()
    log(f"{det.name} [{feature_mode}] trained+scored in {time.time()-t0:.0f}s")
    return ScoredData(det.name if feature_mode == "raw" else f"{det.name}-share",
                      test, phi, det, ft, ref, train_ref)


def load_plugin_scores(ds: Dataset, csv_path, name="Plug-in"):
    """Use scores from your own model (e.g., the dynamic risk-indicator model).
    CSV columns: user, day (YYYY-MM-DD), score (higher = riskier)."""
    S = pd.read_csv(csv_path, dtype={"user": str})
    S["day"] = pd.to_datetime(S["day"])
    test = ds.df[~ds.df["train"]].merge(S[["user", "day", "score"]].rename(columns={"score": "raw_score"}),
                                        on=["user", "day"], how="left").reset_index(drop=True)
    test["raw_score"] = test["raw_score"].fillna(test["raw_score"].min())
    test["score"] = to_quantile(test["raw_score"].to_numpy())   # no training scores supplied
    phi = np.zeros((len(test), 1), np.float32)
    return ScoredData(name, test, phi)


# =============================================================================
# 4. Alert policies
# =============================================================================

def largest_remainder(n, k):
    exact = k * n / n.sum()
    kg = np.floor(exact).astype(int)
    rem = k - kg.sum()
    if rem > 0:
        kg[np.argsort(-(exact - kg))[:rem]] += 1
    return np.minimum(kg, n)


def fair_allocate(scores, gids, k, delta):
    """Exact FAIR-IRS allocation (Prop. 2): returns boolean alert mask.

    delta >= 1 -> global top-k;  delta == 'quota' -> proportional quota.
    Returns (mask, used_fallback).
    """
    N = len(scores)
    mask = np.zeros(N, bool)
    if k >= N:
        mask[:] = True
        return mask, False
    if delta != "quota" and delta >= 1:
        mask[np.argpartition(-scores, k - 1)[:k]] = True
        return mask, False
    labels, inv = np.unique(gids, return_inverse=True)
    G = len(labels)
    order = [np.nonzero(inv == g)[0] for g in range(G)]
    order = [o[np.argsort(-scores[o], kind="stable")] for o in order]
    sorted_s = [scores[o] for o in order]
    n = np.array([len(o) for o in order])

    def take(kg):
        for g in range(G):
            mask[order[g][:kg[g]]] = True
        return mask

    if delta == "quota":
        return take(largest_remainder(n, k)), False

    prefix = [np.concatenate([[0.0], np.cumsum(s)]) for s in sorted_s]
    rmax = k / N + 1e-12
    cands = {0.0, rmax}
    for g in range(G):
        for j in range(0, int(math.floor(rmax * n[g])) + 2):
            cands.add(j / n[g])
        for j in range(int(math.ceil(delta * n[g])), int(math.floor((rmax + delta) * n[g])) + 2):
            cands.add(j / n[g] - delta)
    cands = sorted(c for c in cands if 0 <= c <= rmax)
    mids = [(a + b) / 2 for a, b in zip(cands[:-1], cands[1:])]
    best_val, best_kg = -np.inf, None
    for r in cands + mids:
        L = np.ceil(r * n - 1e-9).astype(int)
        U = np.minimum(n, np.floor((r + delta) * n + 1e-9).astype(int))
        if L.sum() > k or U.sum() < k or np.any(L > U):
            continue
        val = sum(prefix[g][L[g]] for g in range(G))
        m = k - L.sum()
        kg = L.copy()
        if m > 0:
            pool = np.concatenate([sorted_s[g][L[g]:U[g]] for g in range(G)])
            pid = np.concatenate([np.full(U[g] - L[g], g) for g in range(G)])
            top = np.argpartition(-pool, m - 1)[:m]
            val += pool[top].sum()
            kg += np.bincount(pid[top], minlength=G)
        if val > best_val:
            best_val, best_kg = val, kg
    if best_kg is None:          # integrality makes this delta infeasible today
        return take(largest_remainder(n, k)), True
    return take(best_kg), False


def fair_allocate_cum(scores, gids, k, delta, A, C):
    """FAIR-IRS allocation with parity on CUMULATIVE alert rates.

    A[g], C[g]: alerts and active user-days of group g before today (dicts).
    Constraint: |(A_g + k_g)/(C_g + n_g) - (A_h + k_h)/(C_h + n_h)| <= delta.
    Same structure as fair_allocate (separable concave objective, box
    constraints for fixed r), hence exact. Returns (mask, used_fallback, labels, kg).
    """
    N = len(scores)
    mask = np.zeros(N, bool)
    labels, inv = np.unique(gids, return_inverse=True)
    G = len(labels)
    order = [np.nonzero(inv == g)[0] for g in range(G)]
    order = [o[np.argsort(-scores[o], kind="stable")] for o in order]
    sorted_s = [scores[o] for o in order]
    n = np.array([len(o) for o in order])
    a = np.array([A.get(l, 0) for l in labels], float)
    D = np.array([C.get(l, 0) for l in labels], float) + n
    k = min(k, N)

    def take(kg):
        for g in range(G):
            mask[order[g][:kg[g]]] = True
        return mask

    if delta != "quota" and delta >= 1:
        mask[np.argpartition(-scores, k - 1)[:k]] = True
        return mask, False
    prefix = [np.concatenate([[0.0], np.cumsum(s)]) for s in sorted_s]
    rmax = (k + a.sum()) / D.sum() + 1e-12
    cands = [np.array([0.0, rmax])]
    for g in range(G):
        jl = np.arange(0, n[g] + 1)
        cands.append((a[g] + jl) / D[g])
        cands.append((a[g] + jl) / D[g] - delta)
    c = np.unique(np.concatenate(cands))
    c = c[(c >= 0) & (c <= rmax)]
    r = np.unique(np.concatenate([c, (c[:-1] + c[1:]) / 2]))
    Lraw = np.ceil(r[:, None] * D - a - 1e-9)
    Uraw = np.floor((r[:, None] + delta) * D - a + 1e-9)
    ok = (Lraw <= n).all(1) & (Uraw >= 0).all(1)
    L = np.clip(Lraw, 0, n).astype(int)
    U = np.clip(Uraw, 0, n).astype(int)
    ok &= (L.sum(1) <= k) & (U.sum(1) >= k) & (L <= U).all(1)
    best_val, best_kg = -np.inf, None
    if ok.any():
        boxes = np.unique(np.hstack([L[ok], U[ok]]), axis=0)
        for row in boxes:
            Lb, Ub = row[:G], row[G:]
            val = sum(prefix[g][Lb[g]] for g in range(G))
            m = k - Lb.sum()
            kg = Lb.copy()
            if m > 0:
                pool = np.concatenate([sorted_s[g][Lb[g]:Ub[g]] for g in range(G)])
                pid = np.concatenate([np.full(Ub[g] - Lb[g], g) for g in range(G)])
                top = np.argpartition(-pool, m - 1)[:m]
                val += pool[top].sum()
                kg += np.bincount(pid[top], minlength=G)
            if val > best_val:
                best_val, best_kg = val, kg
    if best_kg is None:
        # deficit apportionment: give today's alerts to groups furthest below the
        # common cumulative rate (keeps cumulative rates within ~1/C_g)
        target = (k + a.sum()) * D / D.sum() - a
        e = np.clip(target, 0, n)
        if e.sum() <= 0:
            e = n.astype(float)
        exact = k * e / e.sum()
        kg = np.minimum(np.floor(exact).astype(int), n)
        rem = k - kg.sum()
        for g in np.argsort(-(exact - kg)):
            if rem <= 0:
                break
            if kg[g] < n[g]:
                kg[g] += 1
                rem -= 1
        while rem > 0:                          # capacity left elsewhere
            g = int(np.argmax(n - kg))
            kg[g] += 1
            rem -= 1
        return take(kg), True
    return take(best_kg), False


@dataclass
class PolicyConfig:
    budget: float = 0.01          # alerts per day as a fraction of active users
    delta: object = 1.0           # 1.0 = global top-k, 'quota', or a float in [0,1)
    parity: str = "cumulative"    # 'cumulative' (default) or 'daily' rates
    group_col: str = "g_functional_unit"
    schedule: bool = False
    lam: float = 0.01             # score discount per cleared similar alert
    gamma: float = 0.8            # explanation-similarity gate
    tau_ov: float = 0.999         # override threshold (score quantile)
    m_cap: int = 3                # max number of discounts
    horizon: int = 30             # days a cleared alert is remembered
    p_miss: float = 0.0           # analyst clears a truly malicious alert
    p_flag: float = 0.0           # analyst fails to clear a benign alert
    floor_mult: float = None      # only employees within the global top (floor_mult * k) may be
                                  # alerted; unfilled slots stay empty. 1.0 = leveling down.
    seed: int = 0


def simulate_policy(sd: ScoredData, cfg: PolicyConfig):
    """Run the alert policy day by day. Returns test df with columns
    alert, cleared, s_adj."""
    rng = np.random.default_rng(cfg.seed)
    T = sd.test
    days = T["day"].to_numpy()
    users = T["user"].to_numpy()
    y = T["y"].to_numpy()
    s = T["score"].to_numpy().astype(np.float64)
    g = T[cfg.group_col].to_numpy()
    phi = sd.phi
    phin = phi / (np.linalg.norm(phi, axis=1, keepdims=True) + 1e-9)
    alert = np.zeros(len(T), bool)
    cleared = np.zeros(len(T), bool)
    s_adj = s.copy()
    store = {}                                   # user -> list of (day, unit phi)
    fallbacks = 0
    n_discounts = 0
    cumA, cumC = {}, {}
    disp_ranks = []          # global rank / k of alerts outside the global top-k
    budget_total = 0
    uniq_days, starts = np.unique(days, return_index=True)
    bounds = list(starts) + [len(T)]
    for di, d in enumerate(uniq_days):
        idx = np.arange(bounds[di], bounds[di + 1])
        sd_day = s[idx].copy()
        if cfg.schedule:
            for j, i in enumerate(idx):
                hist = store.get(users[i])
                if not hist:
                    continue
                hist[:] = [(dd, v) for dd, v in hist if (d - dd) / np.timedelta64(1, "D") <= cfg.horizon]
                if not hist or sd_day[j] >= cfg.tau_ov:
                    continue
                sims = [float(phin[i] @ v) for _, v in hist]
                if max(sims) >= cfg.gamma:
                    sd_day[j] -= cfg.lam * min(len(hist), cfg.m_cap)
                    n_discounts += 1
        s_adj[idx] = sd_day
        k = max(1, int(round(cfg.budget * len(idx))))
        if cfg.parity == "cumulative" and cfg.delta != "quota":
            m, fb = fair_allocate_cum(sd_day, g[idx], k, cfg.delta, cumA, cumC)
        else:
            m, fb = fair_allocate(sd_day, g[idx], k, cfg.delta)
        fallbacks += fb
        rank = np.empty(len(idx), int)
        rank[np.argsort(-sd_day, kind="stable")] = np.arange(1, len(idx) + 1)
        if cfg.floor_mult is not None:
            m = m & (rank <= int(np.ceil(cfg.floor_mult * k)))
        disp_ranks.extend((rank[m & (rank > k)] / k).tolist())
        budget_total += k
        for lab, cnt in zip(*np.unique(g[idx], return_counts=True)):
            cumC[lab] = cumC.get(lab, 0) + cnt
        for lab, cnt in zip(*np.unique(g[idx][m], return_counts=True)):
            cumA[lab] = cumA.get(lab, 0) + cnt
        a_idx = idx[m]
        alert[a_idx] = True
        u = rng.random(len(a_idx))
        clr = np.where(y[a_idx] == 1, u < cfg.p_miss, u >= cfg.p_flag)
        cleared[a_idx[clr]] = True
        if cfg.schedule:
            for i in a_idx[clr]:
                store.setdefault(users[i], []).append((d, phin[i]))
    out = T[["user", "day", "y", "score", "exposure", "scenario", cfg.group_col]].copy()
    out["alert"], out["cleared"], out["s_adj"] = alert, cleared, s_adj
    out.attrs["fallback_days"] = fallbacks
    out.attrs["discounts"] = n_discounts
    out.attrs["n_days"] = len(uniq_days)
    out.attrs["disp_ranks"] = np.array(disp_ranks)
    out.attrs["budget_total"] = budget_total
    return out


# =============================================================================
# 5. Metrics
# =============================================================================

def gini(x):
    x = np.sort(np.asarray(x, float))
    if x.sum() == 0:
        return 0.0
    n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1) @ x / (n * x.sum()))


def exposure_bias(test, score_col="score", group_col="g_functional_unit", min_rows=30):
    """Within-group Spearman correlation between score and exposure (benign rows)."""
    b = test[test["y"] == 0]
    rs, ws = [], []
    for _, d in b.groupby(group_col):
        if len(d) >= min_rows and d["exposure"].nunique() > 1:
            r = spearmanr(d[score_col], d["exposure"]).statistic
            if np.isfinite(r):
                rs.append(r)
                ws.append(len(d))
    return float(np.average(rs, weights=ws)) if rs else float("nan")


def wilson(x, n, z=1.96):
    """95% Wilson score interval for a binomial proportion x/n."""
    if n == 0:
        return float("nan"), float("nan")
    p = x / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return max(0.0, c - h), min(1.0, c + h)


def policy_metrics(out, group_col, insiders=None, pi_assumed=0.05):
    res = {}
    mal = out[out["y"] == 1]
    ins_users = mal["user"].unique()
    det_users = mal.loc[mal["alert"], "user"].unique()
    res["insiders_test"] = len(ins_users)
    res["insiders_detected"] = len(det_users)
    res["recall_insider"] = len(det_users) / max(len(ins_users), 1)
    res["recall_lo"], res["recall_hi"] = wilson(len(det_users), len(ins_users))
    res["tpr_day"] = mal["alert"].mean() if len(mal) else float("nan")
    res["precision"] = out.loc[out["alert"], "y"].mean()
    # detection delay (days from first malicious test day to first alert on a malicious day)
    delays = []
    for u, d in mal.groupby("user"):
        hits = d.loc[d["alert"], "day"]
        if len(hits):
            delays.append((hits.min() - d["day"].min()).days)
    res["median_delay_days"] = float(np.median(delays)) if delays else float("nan")
    # group burden
    G = out.groupby(group_col)
    ar = G["alert"].mean()
    ben = out[out["y"] == 0]
    far = ben.groupby(group_col)["alert"].mean()
    pi = G["y"].mean()
    res["delta_AR"] = ar.max() - ar.min()
    res["delta_FAR"] = far.max() - far.min()
    res["ratio_FAR"] = far.max() / far.min() if far.min() > 0 else float("inf")
    res["pi_max"] = pi.max()
    res["lemma1_bound"] = res["delta_AR"] + 2 * res["pi_max"]
    res["lemma1_holds"] = bool(res["delta_FAR"] <= res["lemma1_bound"] + 1e-12)
    # audit from investigation outcomes: alerts confirmed as malicious are observed,
    # so P(a=1, y=0 | g) = AR_g - confirmed_g / n_g is known exactly; only the factor
    # 1/(1 - pi_g) is unknown. Interval for FAR_g with pi_g in [0, pi_assumed].
    conf = (out["alert"] & ~out["cleared"]).groupby(out[group_col]).sum() / G.size()
    lo = (ar - conf).clip(lower=0)
    hi = lo / (1 - pi_assumed)
    res["delta_FAR_hat"] = lo.max() - lo.min()
    res["audit_abs_err"] = abs(res["delta_FAR_hat"] - res["delta_FAR"])
    res["audit_bound"] = hi.max() - lo.min()
    res["audit_holds"] = bool(res["delta_FAR"] <= res["audit_bound"] + 1e-12)
    # longitudinal burden on never-malicious users
    benign_users = set(out["user"]) - set(ins_users)
    B = out[out["user"].isin(benign_users)].groupby("user")["cleared"].sum()
    res["gini_burden"] = gini(B.values)
    top = max(1, int(round(0.01 * len(B))))
    res["top1pct_share"] = B.sort_values(ascending=False).iloc[:top].sum() / max(B.sum(), 1)
    res["max_alerts_one_benign_user"] = int(B.max()) if len(B) else 0
    res["fallback_days"] = out.attrs.get("fallback_days", 0)
    res["discounts"] = out.attrs.get("discounts", 0)
    bt = out.attrs.get("budget_total", 0)
    res["alerts_used"] = out["alert"].sum() / bt if bt else float("nan")
    dr = out.attrs.get("disp_ranks", np.array([]))
    res["displaced_share"] = len(dr) / max(out["alert"].sum(), 1)
    res["displaced_rank_median"] = float(np.median(dr)) if len(dr) else float("nan")
    res["displaced_rank_p90"] = float(np.quantile(dr, 0.9)) if len(dr) else float("nan")
    return res


def lorenz(out):
    """Lorenz curve of cleared alerts over never-malicious employees."""
    ins = set(out.loc[out["y"] == 1, "user"])
    B = np.sort(out[~out["user"].isin(ins)].groupby("user")["cleared"].sum().to_numpy())
    x = np.concatenate([[0], np.arange(1, len(B) + 1) / len(B)])
    yv = np.concatenate([[0], np.cumsum(B) / max(B.sum(), 1)])
    return x, yv


def group_table(out, group_col):
    G = out.groupby(group_col)
    ben = out[out["y"] == 0].groupby(group_col)
    return pd.DataFrame({"users": G["user"].nunique(), "user_days": G.size(),
                         "pi": G["y"].mean(), "AR": G["alert"].mean(),
                         "FAR": ben["alert"].mean(), "TPR": out[out["y"] == 1].groupby(group_col)["alert"].mean()})


def dp_certificates(out, group_col, eps, beta=0.05, seed=0):
    """Per-day DP release of group alert counts and certificate soundness (Prop. 3)."""
    rng = np.random.default_rng(seed)
    rows = []
    for d, D in out.groupby("day"):
        G = D.groupby(group_col)
        n = G.size().to_numpy()
        if len(n) < 2:
            continue
        cnt = G["alert"].sum().to_numpy()
        noisy = cnt + rng.laplace(0, 1 / eps, len(cnt))
        ar, nar = cnt / n, noisy / n
        ben = D[D["y"] == 0].groupby(group_col)["alert"].mean()
        pi = G["y"].mean().max()
        slack = 2 * math.log(len(n) / beta) / (eps * n.min())
        bound = (nar.max() - nar.min()) + slack + 2 * pi
        dfar = ben.max() - ben.min()
        rows.append({"day": d, "true_dAR": ar.max() - ar.min(), "noisy_dAR": nar.max() - nar.min(),
                     "slack": slack, "bound": bound, "true_dFAR": dfar, "holds": dfar <= bound + 1e-12})
    R = pd.DataFrame(rows)
    return {"eps_per_day": eps, "eps_total_basic": eps * len(R), "days": len(R),
            "coverage": R["holds"].mean(), "mean_slack": R["slack"].mean(),
            "mean_abs_err_dAR": (R["noisy_dAR"] - R["true_dAR"]).abs().mean(),
            "mean_bound": R["bound"].mean(), "mean_true_dFAR": R["true_dFAR"].mean()}


def dp_certificates_window(out, group_col, eps, window=30, beta=0.05, seed=0, sizes="roster"):
    """Window-level DP burden report under ALERT-LOG semantics: the protected
    dataset is the alert log; neighbouring logs differ in one employee's rows.

    sizes='roster' (default): denominators are group rosters (every employee
      assigned to the group during the test period), which are public
      organisational information. Rates are the fraction of the roster alerted
      at least once in the window. Each employee adds at most 1 to one alert
      count, so the release of noisy counts has sensitivity 1 and is eps-DP.
      Slack (Prop. 3): 2 ln(|G|/beta) / (eps n_min).
    sizes='noisy': denominators are the employees ACTIVE in the window, which
      are not public; they are released with noise too, eps split evenly.
      Slack: 2 (b + b) / (n_min - b), b = ln(2|G|/beta) / (eps/2)."""
    rng = np.random.default_rng(seed)
    o = out.copy()
    d0 = o["day"].min()
    o["w"] = (o["day"] - d0).dt.days // window
    roster = o.groupby(group_col)["user"].nunique()
    U = o.groupby(["w", "user"]).agg(g=(group_col, "first"), alerted=("alert", "any"), mal=("y", "max")).reset_index()
    rows = []
    for w, D in U.groupby("w"):
        G = D.groupby("g")
        cnt = G["alerted"].sum()
        if sizes == "roster":
            n = roster.reindex(cnt.index).to_numpy().astype(float)
            mal = G["mal"].sum().to_numpy()
            ben_n = n - mal                               # benign roster members (inactive ones are benign)
            ben_alerted = (G["alerted"].sum() - D[D["mal"] == 1].groupby("g")["alerted"].sum().reindex(cnt.index).fillna(0)).to_numpy()
            far = ben_alerted / np.maximum(ben_n, 1)
            pi = (mal / n).max()
        else:
            n = G.size().to_numpy().astype(float)
            ben = D[D["mal"] == 0].groupby("g")["alerted"].mean().reindex(cnt.index)
            far = ben.to_numpy()
            pi = G["mal"].mean().max()
        if len(n) < 2:
            continue
        c = cnt.to_numpy().astype(float)
        ar = c / n
        if sizes == "roster":
            nar = (c + rng.laplace(0, 1 / eps, len(c))) / n
            slack = 2 * math.log(len(n) / beta) / (eps * n.min())
        else:
            e_half = eps / 2
            nn = n + rng.laplace(0, 1 / e_half, len(n))
            nar = (c + rng.laplace(0, 1 / e_half, len(c))) / np.maximum(nn, 1.0)
            bq = math.log(2 * len(n) / beta) / e_half
            slack = 2 * (bq + bq) / (n.min() - bq) if n.min() > bq else float("inf")
        bound = (nar.max() - nar.min()) + slack + 2 * pi
        dfar = np.nanmax(far) - np.nanmin(far)
        rows.append({"true_dAR": ar.max() - ar.min(), "noisy_dAR": nar.max() - nar.min(), "slack": slack,
                     "bound": bound, "true_dFAR": dfar, "holds": dfar <= bound + 1e-12, "n_min": n.min()})
    R = pd.DataFrame(rows)
    fin = R["slack"].replace(np.inf, np.nan)
    return {"window_days": window, "sizes": sizes, "eps_per_window": eps, "windows": len(R),
            "eps_total_basic": eps * len(R), "coverage": R["holds"].mean(),
            "median_slack": R["slack"].median(), "mean_slack": fin.mean(),
            "vacuous_share": float(np.isinf(R["slack"]).mean()),
            "median_abs_err_dAR": (R["noisy_dAR"] - R["true_dAR"]).abs().median(),
            "mean_true_dFAR": R["true_dFAR"].mean(), "median_min_group": R["n_min"].median()}


# =============================================================================
# 6. Simulations
# =============================================================================

def exposure_injection(ds: Dataset, factor, frac_roles=0.3, seed=0):
    """Return a copy of ds where benign activity counts of users in a random
    subset of roles are multiplied by `factor` (legitimately higher telemetry)."""
    rng = np.random.default_rng(seed)
    roles = ds.df["role"].unique()
    chosen = set(rng.choice(roles, max(1, int(len(roles) * frac_roles)), replace=False))
    D = ds.df.copy()
    sel = D["role"].isin(chosen).to_numpy()
    for c in COUNT_FEATURES:
        D.loc[sel, c] = np.round(D.loc[sel, c] * factor)
    D["exposure"] = D[VOLUME_FEATURES].sum(axis=1)
    D["g_injected"] = np.where(sel, "inflated", "normal")
    return Dataset(ds.name + f"-inj{factor}", D, ds.insiders, ds.split_day, ds.group_cols + ["g_injected"])


def discount_farming(sd: ScoredData, n_farm=3, alpha=1.0):
    """Insert farming days for each test insider: benign days just before the
    first malicious day are replaced with x_benign + alpha * (mean malicious -
    mean benign) so that they resemble the later malicious behaviour.
    Returns a new ScoredData and the list of farming insiders."""
    T = sd.test.copy()
    phi = sd.phi.copy()
    farmers = []
    for u, D in T[T["y"] == 1].groupby("user"):
        d0 = D["day"].min()
        ub = T[(T["user"] == u) & (T["y"] == 0)]
        prior = ub[ub["day"] < d0].sort_values("day").tail(n_farm)
        if prior.empty:
            continue
        delta = (D[COUNT_FEATURES].mean() - ub[COUNT_FEATURES].mean()).clip(lower=0)
        idx = prior.index
        T.loc[idx, COUNT_FEATURES] = (T.loc[idx, COUNT_FEATURES] + alpha * delta.values).round()
        T.loc[idx, "exposure"] = T.loc[idx, VOLUME_FEATURES].sum(axis=1)
        farmers.append(u)
    if not farmers:
        return None, []
    if isinstance(sd.model, PerGroupDetector) or sd.model is None:
        raise ValueError("discount_farming needs a single-model detector")
    X = sd.transform.transform(T)
    raw = sd.model.score(X)
    T["raw_score"] = raw
    T["score"] = to_quantile_ref(raw, sd.raw_ref)
    # recompute explanations for modified rows (baseline unchanged)
    Xo = sd.transform.transform(sd.test)
    base_shift = Xo - sd.phi
    phi = X - base_shift
    return ScoredData(sd.detector + "-farm", T, phi, sd.model, sd.transform, sd.raw_ref), farmers


# =============================================================================
# 7. Export helpers
# =============================================================================

PRETTY = {
    "recall_insider": "Recall", "precision": "Prec.", "median_delay_days": "Delay (d)",
    "delta_AR": r"$\Delta_{\mathrm{AR}}$", "delta_FAR": r"$\Delta_{\mathrm{FAR}}$",
    "ratio_FAR": r"$\rho_{\mathrm{FAR}}$", "gini_burden": r"Gini($B$)", "top1pct_share": "Top-1\\%",
    "exposure_bias": "Exp. bias", "max_alerts_one_benign_user": r"$\max B_i$", "pi_max": r"$\pi_{\max}$",
    "lemma1_bound": "Lemma 1 bound", "detector": "Detector", "grouping": "Grouping", "method": "Method",
    "delta": r"$\delta$", "lam": r"$\lambda$", "p_miss": r"$p_{\mathrm{miss}}$", "eps": r"$\varepsilon$",
    "eps_per_day": r"$\varepsilon$/day", "mean_slack": "Slack", "coverage": "Coverage",
    "mean_true_dFAR": r"True $\Delta_{\mathrm{FAR}}$", "n_groups": r"$|\mathcal{G}|$",
    "avg_min_group_size": r"$\bar{n}_{\min}$", "model": "Model", "kappa": r"$\kappa$",
    "factor": "Factor", "fallback_pct": "Fallback (\\%)", "release": "Release", "users": "Users",
    "insiders": "Insiders", "insiders_test": "Test ins.", "days": "Days", "user_days": "User-days",
    "base_rate": r"$\pi$", "roles": "Roles", "functional_units": "Func. units", "split_day": "Split",
    "farmer_recall": "Recall (farmers)", "alpha": r"$\alpha$", "gamma": r"$\gamma$",
    "tau_ov": r"$\tau_{\mathrm{ov}}$", "schedule": "Sched.",
    "window_days": "Window (d)", "avg_min_group_users": r"$\bar{n}_{\min}$", "eps_per_window": r"$\varepsilon$/window",
    "eps_total_basic": r"Total $\varepsilon$", "recall_no_farming": "Recall (no farming)", "recall_farming": "Recall (farming)",
    "tpr_day": "Day TPR", "recall_global": "Recall (global)", "recall_fair": "Recall (FAIR-IRS)",
    "fallback_days": "Fallback (days)", "lost_by_allocation": "Lost",
 "gained_by_allocation": "Gained", "mean_bound": "Bound", "setting": "Setting",
    "recall_ci": "Recall [95\\% CI]", "alerts_used": "Alerts used", "displaced_share": "Displaced",
    "displaced_rank_median": "Disp. rank (med.)", "displaced_rank_p90": "Disp. rank (p90)",
    "delta_FAR_hat": r"$\widehat{\Delta}_{\mathrm{FAR}}$", "audit_abs_err": "Audit error", "audit_bound": "Outcome bound",
    "floor": "Floor", "sizes": "Sizes", "median_slack": "Slack (median)", "vacuous_share": "Vacuous", "median_abs_err_dAR": "Error (median)",
    "median_min_group": r"$n_{\min}$ (median)", "weekdays_only": "Weekdays", "insiders_detected": "Detected",
    "policy": "Policy", "runs": "Runs", "budget": "Budget", "training": "Training", "baseline": "Expl. baseline",
    "recall_no_sched": "Recall (no sched.)", "recall_sched": "Recall (sched.)",
}
GROUP_PRETTY = {"g_functional_unit": "Func. unit", "g_department": "Department", "g_role": "Role",
                "g_afterhours": "After-hours", "g_exposure": "Exposure", "g_injected": "Injected",
                "g_synthA": "Synthetic $A$"}


def to_latex(df, path, caption, label, float_fmt="{:.3f}"):
    df = df.copy()
    for c in df.columns:
        if not (pd.api.types.is_numeric_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c])):
            df[c] = df[c].map(lambda v: GROUP_PRETTY.get(v, v))
    ONE_DECIMAL = {PRETTY.get(c, c) for c in ["median_delay_days", "fallback_pct", "avg_min_group_size",
                                               "max_alerts_one_benign_user"]}
    esc = lambda c: c if "$" in c or "\\" in c else c.replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")
    df.columns = [PRETTY.get(c, esc(str(c))) for c in df.columns]
    cols = list(df.columns)
    lines = [r"\begin{table}[!t]", rf"\caption{{{caption}\label{{{label}}}}}", r"\centering",
             r"\resizebox{\columnwidth}{!}{%", r"\begin{tabular}{" + "l" * 1 + "c" * (len(cols) - 1) + "}",
             r"\toprule", " & ".join(cols) + r" \\", r"\midrule"]
    for _, r in df.iterrows():
        cells = []
        for c in cols:
            v = r[c]
            if isinstance(v, (bool, np.bool_)):
                cells.append("yes" if v else "no")
            elif isinstance(v, (int, np.integer)):
                cells.append(f"{v:,}")
            elif isinstance(v, (float, np.floating)):
                fmt = "{:.1f}" if c in ONE_DECIMAL else float_fmt
                cells.append(r"$\infty$" if v == np.inf else ("--" if not np.isfinite(v) else fmt.format(v)))
            else:
                v = str(v)
                if "$" not in v:
                    v = v.replace("_", r"\_").replace("%", r"\%").replace("&", r"\&")
                cells.append(v)
        lines.append(" & ".join(cells) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}}", r"\end{table}"]
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")


# =============================================================================
# 8. Toy CERT-format generator (for testing the pipeline before real data)
# =============================================================================

def make_toy_cert(root, version="4.2", n_users=150, n_days=120, n_insiders=8, seed=0):
    """Write a small CERT-format release + answers folder for smoke tests."""
    rng = np.random.default_rng(seed)
    rel = os.path.join(root, f"r{version}")
    os.makedirs(os.path.join(rel, "LDAP"), exist_ok=True)
    ans = os.path.join(root, "answers")
    os.makedirs(ans, exist_ok=True)
    users = [f"{''.join(rng.choice(list('ABCDEFGHIJKLMNOPQRSTUVWXYZ'), 3))}{1000+i}" for i in range(n_users)]
    roles = rng.choice(["ITAdmin", "Salesman", "Engineer", "Manager", "Clerk", "Scientist"], n_users,
                       p=[.12, .3, .2, .1, .18, .1])
    units = {"ITAdmin": "IT", "Salesman": "Sales", "Engineer": "R&D", "Manager": "Admin",
             "Clerk": "Admin", "Scientist": "R&D"}
    act = {"ITAdmin": 3.0, "Salesman": 1.0, "Engineer": 1.5, "Manager": 0.8, "Clerk": 0.7, "Scientist": 1.2}
    start = pd.Timestamp("2010-01-04")
    days = pd.date_range(start, periods=n_days, freq="D")
    months = sorted({d.strftime("%Y-%m") for d in days})
    for m in months:
        pd.DataFrame({"employee_name": users, "user_id": users,
                      "email": [u.lower() + "@dtaa.com" for u in users], "role": roles,
                      "business_unit": "1", "functional_unit": [units[r] for r in roles],
                      "department": [units[r] + "-" + str(i % 2) for i, r in enumerate(roles)],
                      "team": "t", "supervisor": "x"}).to_csv(os.path.join(rel, "LDAP", f"{m}.csv"), index=False)
    night = set(rng.choice(users, n_users // 8, replace=False))
    insiders = rng.choice(users, n_insiders, replace=False)
    ins_rows = []
    mal_days = {}
    for j, u in enumerate(insiders):
        s = int(rng.integers(n_days * 0.3, n_days * 0.85))
        mal_days[u] = [days[s + t] for t in range(int(rng.integers(2, 6))) if s + t < n_days]
        ins_rows.append({"dataset": version, "scenario": 1 + j % 3,
                         "details": f"r{version}-{1 + j % 3}-{u}.csv", "user": u,
                         "start": mal_days[u][0].strftime(DATE_FMT),
                         "end": (mal_days[u][-1] + pd.Timedelta(hours=23)).strftime(DATE_FMT)})
    r6 = str(version).startswith("6")
    logon, device, file, email, http = [], [], [], [], []
    detail = {u: [] for u in insiders}
    cnt = 0
    for d in days:
        wk = d.dayofweek >= 5
        for i, u in enumerate(users):
            if wk and rng.random() > 0.1:
                continue
            a = act[roles[i]]
            h0 = 20 if u in night else 8
            is_mal = u in mal_days and d in mal_days[u]
            ts = lambda h: (d + pd.Timedelta(hours=h % 24, minutes=int(rng.integers(0, 59)))).strftime(DATE_FMT)
            cnt += 1
            logon.append([f"{{L{cnt}}}", ts(h0), u, f"PC-{i}", "Logon"])
            logon.append([f"{{F{cnt}}}", ts(h0 + 9), u, f"PC-{i}", "Logoff"])
            for _ in range(rng.poisson(0.3 * a + (4 if is_mal else 0))):
                h = 22 if is_mal else h0 + 2
                row = [f"{{D{cnt}}}", ts(h), u, f"PC-{i}"] + (["C:\\"] if r6 else []) + ["Connect"]
                device.append(row)
                if is_mal:
                    detail[u].append(["device"] + row)
            for _ in range(rng.poisson(5 * a + (10 if is_mal else 0))):
                ext = rng.choice(["doc", "pdf", "txt", "jpg", "zip", "exe"])
                row = [f"{{Fi{cnt}}}", ts(h0 + 3), u, f"PC-{i}", f"X{cnt}.{ext}"]
                row += (["File Write", str(bool(is_mal)), "False", "kw"] if r6 else ["kw"])
                file.append(row)
            for _ in range(rng.poisson(4 * a)):
                to = "a@dtaa.com" if rng.random() > 0.1 else "b@gmail.com"
                row = [f"{{E{cnt}}}", ts(h0 + 4), u, f"PC-{i}", to, "", "", u.lower() + "@dtaa.com"]
                row += (["Send"] if r6 else []) + [str(int(rng.integers(1000, 90000))),
                                                    ("f.doc(1)" if r6 else "1") if rng.random() < .2 else ("" if r6 else "0"),
                                                    "kw"]
                email.append(row)
            for _ in range(rng.poisson(12 * a)):
                url = rng.choice(["http://news.com/x", "http://shop.com/y", "http://monster.com/job"],
                                 p=[.6, .38, .02])
                if is_mal and rng.random() < 0.3:
                    url = "http://wikileaks.org/up"
                row = [f"{{H{cnt}}}", ts(h0 + 5), u, f"PC-{i}", url] + (["WWW Visit"] if r6 else []) + ["kw"]
                http.append(row)
    hdr = lambda names: names
    def write(name, rows, cols, header=True):
        df = pd.DataFrame(rows, columns=cols)
        df.to_csv(os.path.join(rel, name), index=False, header=header)
    write("logon.csv", logon, ["id", "date", "user", "pc", "activity"])
    write("device.csv", device, NOHEADER_SCHEMAS["device"][6 if r6 else 5])
    write("file.csv", file, NOHEADER_SCHEMAS["file"][9 if r6 else 6])
    write("email.csv", email, NOHEADER_SCHEMAS["email"][12 if r6 else 11])
    # http.csv without header, as in some CERT distributions
    write("http.csv", http, NOHEADER_SCHEMAS["http"][7 if r6 else 6], header=False)
    pd.DataFrame({"employee_name": users, "user_id": users, "O": 30, "C": 30, "E": 30, "A": 30, "N": 30}) \
        .to_csv(os.path.join(rel, "psychometric.csv"), index=False)
    ins_path = os.path.join(ans, "insiders.csv")
    old = pd.read_csv(ins_path, dtype=str) if os.path.exists(ins_path) else None
    new = pd.DataFrame(ins_rows)
    (pd.concat([old, new]) if old is not None else new).to_csv(ins_path, index=False)
    for j, u in enumerate(insiders):
        sub = os.path.join(ans, f"r{version}-{1 + j % 3}")
        os.makedirs(sub, exist_ok=True)
        with open(os.path.join(sub, f"r{version}-{1 + j % 3}-{u}.csv"), "w", newline="") as f:
            csv.writer(f).writerows(detail[u] or [["logon", "{x}", mal_days[u][0].strftime(DATE_FMT), u, "PC", "Logon"]])
    return rel, ans


# =============================================================================
# 9. Experiments (one function per research question / simulation)
# =============================================================================

def default_detectors(seed=0):
    return [IFDetector(seed), AEDetector(seed), LRDetector(), GBMDetector(seed)]


def score_all(ds: Dataset, detectors=None, extra_modes=True, peer_col="g_role", seed=0, with_lstm=True):
    """Train every detector once and return {name: ScoredData}."""
    detectors = detectors or default_detectors(seed)
    if with_lstm:
        try:
            import torch  # noqa: F401
            detectors = list(detectors) + [LSTMDetector(seed)]
        except ImportError:
            log("[skip] LSTM: PyTorch is not installed")
    scored = {}
    for det in detectors:
        try:
            sd = run_detector(ds, det)
        except Exception as ex:                      # one detector must not stop the run
            log(f"[skip] {det.name}: {type(ex).__name__}: {ex}")
            continue
        if sd is not None:
            scored[sd.detector] = sd
    if extra_modes:
        sd = run_detector(ds, IFDetector(seed), feature_mode="share")       # B2
        scored[sd.detector] = sd
        pg = PerGroupDetector(partial(IFDetector, seed), None, None, name="IF-peer")  # B3
        sd = run_detector(ds, pg, peer_col=peer_col)
        sd.detector = "IF-peer"
        scored["IF-peer"] = sd
    return scored


def rq1_audit(ds, scored, budget=0.01, detectors=None):
    """Burden of unmitigated detectors (global top-k) for every grouping."""
    rows, gtabs = [], {}
    for name in detectors or ["IF", "AE", "LR", "GBM", "LSTM"]:
        if name not in scored:
            continue
        sd = scored[name]
        for gc in ds.group_cols:
            out = simulate_policy(sd, PolicyConfig(budget=budget, delta=1.0, group_col=gc))
            m = policy_metrics(out, gc)
            m.update(detector=name, grouping=gc, exposure_bias=exposure_bias(sd.test, group_col=gc))
            rows.append(m)
            gtabs[(name, gc)] = group_table(out, gc)
    return pd.DataFrame(rows), gtabs


def rq2_price_of_fairness(ds, scored, group_col, budget=0.01,
                          deltas=(1.0, 0.01, 0.005, 0.002, 0.001, 0.0, "quota"),
                          methods=("IF", "AE", "LR", "GBM", "LSTM", "IF-share", "IF-peer")):
    rows = []
    for name in methods:
        if name not in scored:
            continue
        for d in deltas:
            if name in ("IF-share", "IF-peer") and d != 1.0:
                continue                      # baselines are evaluated without allocation
            out = simulate_policy(scored[name], PolicyConfig(budget=budget, delta=d, group_col=group_col))
            m = policy_metrics(out, group_col)
            m.update(method=name, delta=str(d), grouping=group_col)
            rows.append(m)
    return pd.DataFrame(rows)


def rq3_scheduling(scored, group_col, budget=0.01, delta=0.002, detectors=("IF", "GBM", "LSTM"),
                   lams=(0.0, 0.005, 0.01, 0.02), p_miss=(0.0, 0.1, 0.3)):
    rows = []
    for name in detectors:
        if name not in scored:
            continue
        for pm in p_miss:
            for lam in lams:
                cfg = PolicyConfig(budget=budget, delta=delta, group_col=group_col,
                                   schedule=lam > 0, lam=lam, p_miss=pm)
                out = simulate_policy(scored[name], cfg)
                m = policy_metrics(out, group_col)
                m.update(detector=name, lam=lam, p_miss=pm)
                rows.append(m)
    return pd.DataFrame(rows)


def rq4a_dp_training(ds, group_col, budget=0.01, eps_list=(0.5, 1, 2, 4, 8), seeds=(0, 1, 2, 3, 4)):
    """LR, clipped SGD without noise, and DP-SGD, each over several seeds.
    Returns (raw rows, summary with mean and 95% CI)."""
    rows = []
    sd = run_detector(ds, LRDetector())
    if sd is None:
        return pd.DataFrame(), pd.DataFrame()
    m = policy_metrics(simulate_policy(sd, PolicyConfig(budget=budget, group_col=group_col)), group_col)
    m.update(model="LR (non-private)", eps=float("inf"), seed=0)
    rows.append(m)
    for e in [np.inf] + list(eps_list):
        for sdseed in seeds:
            det = DPLRDetector(eps=e, seed=sdseed)
            sd = run_detector(ds, det)
            m = policy_metrics(simulate_policy(sd, PolicyConfig(budget=budget, group_col=group_col)), group_col)
            m.update(model="SGD-LR (clipped, no noise)" if e == np.inf else "DP-LR", eps=e, seed=sdseed)
            rows.append(m)
    raw = pd.DataFrame(rows)
    summ = []
    for (mod, e), d in raw.groupby(["model", "eps"], sort=False):
        r = {"model": mod, "eps": e, "runs": len(d)}
        for c in ["recall_insider", "precision", "delta_FAR", "ratio_FAR", "gini_burden"]:
            mu, h = _ci(d[c].replace(np.inf, np.nan))
            r[c] = f"{mu:.3f}" + ("" if not np.isfinite(h) else f" $\\pm$ {h:.3f}")
        summ.append(r)
    return raw, pd.DataFrame(summ)


def rq4b_certificates(ds, scored, detector="IF", budget=0.01, delta=0.002,
                      eps_list=(0.5, 1, 2, 4), windows=(1, 7, 30), beta=0.05,
                      groupings=("g_functional_unit", "g_department", "g_role")):
    """Roster denominators for every window; noisy active sizes reported as a variant."""
    rows = []
    for gc in groupings:
        out = simulate_policy(scored[detector], PolicyConfig(budget=budget, delta=delta, group_col=gc))
        for w in windows:
            for sz in ("roster", "noisy"):
                for e in eps_list:
                    c = dp_certificates_window(out, gc, e, window=w, beta=beta, sizes=sz)
                    c.update(grouping=gc, n_groups=out[gc].nunique())
                    rows.append(c)
    return pd.DataFrame(rows)


def rq5_evasion(scored, group_col, budget=0.01, delta=0.002, detectors=("IF", "GBM", "LSTM"),
                alphas=(0.5, 1.0), n_farm=3, gammas=(0.7, 0.8, 0.9), tau_ovs=(0.995, 0.999, 1.01)):
    """(a) threshold exploitation: insiders caught by global top-k but missed by
    FAIR-IRS allocation (and vice versa); (b) discount farming against the
    scheduler, compared with the same insiders when they do not farm."""
    thr, rows = [], []
    det = lambda o: set(o.loc[(o["y"] == 1) & o["alert"], "user"])
    for name in detectors:
        if name not in scored:
            continue
        sd = scored[name]
        base = simulate_policy(sd, PolicyConfig(budget=budget, delta=1.0, group_col=group_col))
        fair = simulate_policy(sd, PolicyConfig(budget=budget, delta=delta, group_col=group_col))
        thr.append({"detector": name, "recall_global": policy_metrics(base, group_col)["recall_insider"],
                    "recall_fair": policy_metrics(fair, group_col)["recall_insider"],
                    "lost_by_allocation": len(det(base) - det(fair)),
                    "gained_by_allocation": len(det(fair) - det(base)),
                    "lost_users": " ".join(sorted(det(base) - det(fair))),
                    "gained_users": " ".join(sorted(det(fair) - det(base)))})
        if getattr(sd.model, "sequence", False):
            log(f"[skip] discount farming for {name}: needs the full sequence model")
            continue
        for alpha in alphas:
            sdf, farmers = discount_farming(sd, n_farm=n_farm, alpha=alpha)
            if not farmers:
                continue
            def farmer_recall(s_, cfg):
                out = simulate_policy(s_, cfg)
                sub = out[out["user"].isin(farmers) & (out["y"] == 1)]
                return sub.groupby("user")["alert"].any().mean()
            for g in gammas:
                for t in tau_ovs:
                    for sched in (False, True):
                        cfg = PolicyConfig(budget=budget, delta=delta, group_col=group_col,
                                           schedule=sched, gamma=g, tau_ov=t)
                        rows.append({"detector": name, "alpha": alpha, "gamma": g, "tau_ov": t, "schedule": sched,
                                     "farmers": len(farmers),
                                     "recall_no_farming": farmer_recall(sd, cfg),
                                     "recall_farming": farmer_recall(sdf, cfg)})
    return pd.DataFrame(thr), pd.DataFrame(rows)


def sim_kappa(ds, kappas=(0.0, 0.25, 0.5, 0.75, 1.0), budget=0.01, delta=0.002,
              detectors=("IF", "GBM"), seed=0, scored=None):
    rows = []
    scored = scored or score_all(ds, extra_modes=False)
    for kappa in kappas:
        col = add_synthetic_attribute(ds, kappa, seed=seed)
        for name in detectors:
            if name not in scored:
                continue
            sd = scored[name]
            sd.test[col] = sd.test["user"].map(ds.df.drop_duplicates("user").set_index("user")[col])
            for d in (1.0, delta):
                out = simulate_policy(sd, PolicyConfig(budget=budget, delta=d, group_col=col))
                m = policy_metrics(out, col)
                m.update(kappa=kappa, detector=name, delta=str(d))
                rows.append(m)
    return pd.DataFrame(rows)


def sim_exposure_injection(ds, factors=(1.0, 1.5, 2.0, 3.0), budget=0.01, delta=0.002, seed=0):
    rows = []
    for f in factors:
        dsi = exposure_injection(ds, f, seed=seed)
        sd = run_detector(dsi, IFDetector(seed))
        sds = run_detector(dsi, IFDetector(seed), feature_mode="share")
        for label, s, d in [("IF global", sd, 1.0), ("IF + FAIR-IRS", sd, delta), ("IF-share", sds, 1.0)]:
            out = simulate_policy(s, PolicyConfig(budget=budget, delta=d, group_col="g_injected"))
            m = policy_metrics(out, "g_injected")
            m.update(factor=f, method=label, exposure_bias=exposure_bias(s.test, group_col="g_functional_unit"))
            rows.append(m)
    return pd.DataFrame(rows)


# =============================================================================
# 10. Figures
# =============================================================================

METHOD_LABEL = {"IF": "IF (D1)", "AE": "AE (D2)", "LR": "LR (D3)", "GBM": "GBM (D4)", "LSTM": "LSTM (D5)"}


def plot_pareto(df, path, title=None):
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 8, "font.family": "serif"})
    fig, ax = plt.subplots(figsize=(3.4, 2.5))
    marks = iter(["o", "s", "^", "D", "v", "P"])
    for name, d in df.groupby("method", sort=False):
        d = d.sort_values("delta_FAR")
        ax.plot(d["delta_FAR"], d["recall_insider"], marker=next(marks), ms=3.5, lw=1,
                label=METHOD_LABEL.get(name, name))
    ax.set_xlabel(r"False-alert disparity $\Delta_{\mathrm{FAR}}$")
    ax.set_ylabel("Insider recall")
    ax.set_ylim(0, 1.02)
    ax.legend(frameon=False, fontsize=7)
    ax.grid(alpha=.25, lw=.5)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def plot_lorenz(curves, path):
    """curves: list of (label, out) pairs; one panel per detector is drawn by the caller."""
    import matplotlib.pyplot as plt
    plt.rcParams.update({"font.size": 8, "font.family": "serif"})
    fig, ax = plt.subplots(figsize=(3.4, 2.5))
    for (label, out), ls in zip(curves, ["-", "--", ":", "-."]):
        x, yv = lorenz(out)
        ax.plot(x, yv, ls, lw=1.2, label=label)
    ax.plot([0, 1], [0, 1], color="0.6", lw=.6)
    ax.set_xlim(0.8, 1.0)
    ax.set_xlabel("Share of benign employees (sorted by false alerts)")
    ax.set_ylabel("Cumulative share of false alerts")
    ax.legend(frameon=False, fontsize=7, loc="upper left")
    ax.grid(alpha=.25, lw=.5)
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


def plot_lines(df, x, y, hue, path, xlabel=None, ylabel=None, logx=False):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4.2, 3.2))
    for name, d in df.groupby(hue):
        d = d.sort_values(x)
        ax.plot(d[x], d[y], "o-", label=str(name), ms=4)
    if logx:
        ax.set_xscale("log")
    ax.set_xlabel(xlabel or x)
    ax.set_ylabel(ylabel or y)
    ax.legend(fontsize=7)
    ax.grid(alpha=.3)
    fig.tight_layout()
    fig.savefig(path)
    plt.close(fig)


# =============================================================================
# 11. Additional experiments for the final paper
# =============================================================================

def add_intersection(df, a, b, col, min_users=20):
    """Intersectional group a x b; cells with fewer than min_users users -> 'Other'."""
    v = df[a].astype(str) + " | " + df[b].astype(str)
    size = df.assign(_v=v).groupby("_v")["user"].nunique()
    small = set(size[size < min_users].index)
    df[col] = np.where(v.isin(small), "Other", v)
    return col


def _ci(x):
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) < 2:
        return (x.mean() if len(x) else np.nan), np.nan
    from scipy.stats import t
    return x.mean(), t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))


def run_seeds(ds, seeds=(0, 1, 2, 3, 4), budget=0.01, delta=0.002, group_col="g_functional_unit",
              cache_dir=None, metrics=("recall_insider", "precision", "delta_FAR", "ratio_FAR", "gini_burden")):
    """Retrain every detector with several seeds; report mean and 95% CI for
    global top-k and FAIR-IRS. LR is deterministic and is run once."""
    import pickle
    rows = []
    for sd_seed in seeds:
        p = os.path.join(cache_dir, f"scored_{ds.name}_seed{sd_seed}_v2.pkl") if cache_dir else None
        if p and os.path.exists(p):
            with open(p, "rb") as f:
                sc = pickle.load(f)
        else:
            dets = [IFDetector(sd_seed), AEDetector(sd_seed), GBMDetector(sd_seed)] + ([LRDetector()] if sd_seed == seeds[0] else [])
            sc = {}
            for det in dets:
                s = run_detector(ds, det)
                if s is not None:
                    sc[s.detector] = s
            if p:
                with open(p, "wb") as f:
                    pickle.dump(sc, f)
        for name, s in sc.items():
            for d in (1.0, delta):
                out = simulate_policy(s, PolicyConfig(budget=budget, delta=d, group_col=group_col, seed=sd_seed))
                m = policy_metrics(out, group_col)
                m.update(detector=name, policy="global" if d == 1.0 else "FAIR-IRS", seed=sd_seed)
                rows.append(m)
    raw = pd.DataFrame(rows)
    summ = []
    for (det, pol), d in raw.groupby(["detector", "policy"], sort=False):
        r = {"detector": det, "policy": pol, "runs": len(d)}
        for m in metrics:
            mu, h = _ci(d[m])
            r[m] = f"{mu:.3f}" + ("" if not np.isfinite(h) else f" $\\pm$ {h:.3f}")
        summ.append(r)
    return raw, pd.DataFrame(summ)


def budget_sweep(sc, budgets=(0.005, 0.01, 0.02), delta_rel=0.2, group_col="g_functional_unit",
                 detectors=("IF", "AE", "LR", "GBM", "LSTM")):
    """delta is set relative to the budget: delta = delta_rel * budget."""
    rows = []
    for name in detectors:
        if name not in sc:
            continue
        for b in budgets:
            for d in (1.0, delta_rel * b):
                out = simulate_policy(sc[name], PolicyConfig(budget=b, delta=d, group_col=group_col))
                m = policy_metrics(out, group_col)
                m.update(detector=name, budget=b, policy="global" if d == 1.0 else "FAIR-IRS")
                rows.append(m)
    return pd.DataFrame(rows)


def contaminated_training(ds, budget=0.01, group_col="g_functional_unit", seed=0):
    """Unsupervised detectors trained on all training days (labels unused)."""
    rows = []
    for clean in (True, False):
        for det in (IFDetector(seed), AEDetector(seed)):
            s = run_detector(ds, det, clean_train=clean)
            out = simulate_policy(s, PolicyConfig(budget=budget, group_col=group_col))
            m = policy_metrics(out, group_col)
            m.update(detector=det.name, training="clean" if clean else "contaminated",
                     exposure_bias=exposure_bias(s.test, group_col=group_col))
            rows.append(m)
    return pd.DataFrame(rows)


def phi_comparison(ds, sc, budget=0.01, delta=0.002, group_col="g_functional_unit", lam=0.01, seed=0):
    """Scheduling with population- vs self-baseline explanations (IF)."""
    rows = []
    s_self = run_detector(ds, IFDetector(seed), phi_mode="self")
    for label, s in (("population", sc["IF"]), ("self", s_self)):
        for l in (0.0, lam):
            out = simulate_policy(s, PolicyConfig(budget=budget, delta=delta, group_col=group_col,
                                                  schedule=l > 0, lam=l))
            m = policy_metrics(out, group_col)
            m.update(baseline=label, lam=l)
            rows.append(m)
    return pd.DataFrame(rows)


def intersectional(sc, budget=0.01, delta=0.002, main_col="g_functional_unit", inter_col="g_inter",
                   detectors=("IF", "GBM")):
    """Parity enforced on the main grouping vs on the intersection; burden always
    measured on the intersection (fairness gerrymandering check)."""
    rows = []
    for name in detectors:
        if name not in sc:
            continue
        s = sc[name]
        for label, gcol, d in (("global", inter_col, 1.0), ("FAIR-IRS on main grouping", main_col, delta),
                               ("FAIR-IRS on intersection", inter_col, delta)):
            out = simulate_policy(s, PolicyConfig(budget=budget, delta=d, group_col=gcol))
            out[inter_col] = s.test[inter_col].values
            m = policy_metrics(out, inter_col)
            m.update(detector=name, policy=label)
            rows.append(m)
    return pd.DataFrame(rows)


# =============================================================================
# 12. Experiments added after review
# =============================================================================

def with_scores(sd: ScoredData, new_scores, name):
    """Copy of a ScoredData with a different score column (same explanations)."""
    T = sd.test.copy()
    T["score"] = new_scores
    return ScoredData(name, T, sd.phi, sd.model, sd.transform, sd.raw_ref, sd.train_ref)


def group_rank_scores(sd: ScoredData, group_col):
    """Baseline B6: rank scores within each group on each day (percentile), then
    take the global top-k. No memory across days."""
    T = sd.test
    return T.groupby(["day", T[group_col]])["score"].rank(pct=True, method="first").to_numpy() + 1e-9 * T["score"].to_numpy()


def simulate_group_thresholds(sd: ScoredData, group_col, budget=0.01):
    """Baseline B7 (Hardt-style): one threshold per group, tuned on the benign
    training period so that each group's training false-alert rate equals the
    budget rate; at test time everyone above their group's threshold is alerted,
    so the number of alerts per day is not fixed."""
    tr = sd.train_ref
    glob = np.quantile(tr["raw_score"], 1 - budget)
    thr = tr.groupby(group_col)["raw_score"].quantile(1 - budget)
    T = sd.test
    t = T[group_col].map(thr).fillna(glob).to_numpy()
    alert = T["raw_score"].to_numpy() > t
    out = T[["user", "day", "y", "score", "exposure", "scenario", group_col]].copy()
    out["alert"] = alert
    out["cleared"] = alert & (T["y"].to_numpy() == 0)
    out["s_adj"] = out["score"]
    k_total = sum(max(1, int(round(budget * n))) for n in T.groupby("day").size())
    out.attrs.update(fallback_days=0, discounts=0, n_days=T["day"].nunique(), disp_ranks=np.array([]),
                     budget_total=k_total)
    return out


def rq2_baselines(sc, group_col, budget=0.01, delta=0.002, detectors=("IF", "AE", "LR", "GBM", "LSTM")):
    """Global top-k, FAIR-IRS, B6 group-rank and B7 group thresholds for each detector."""
    rows = []
    for name in detectors:
        if name not in sc:
            continue
        sd = sc[name]
        runs = [("global (B0)", simulate_policy(sd, PolicyConfig(budget=budget, delta=1.0, group_col=group_col))),
                ("FAIR-IRS (B4)", simulate_policy(sd, PolicyConfig(budget=budget, delta=delta, group_col=group_col))),
                ("group rank (B6)", simulate_policy(with_scores(sd, group_rank_scores(sd, group_col), name),
                                                   PolicyConfig(budget=budget, delta=1.0, group_col=group_col))),
                ("group thresholds (B7)", simulate_group_thresholds(sd, group_col, budget))]
        for label, out in runs:
            m = policy_metrics(out, group_col)
            m.update(detector=name, policy=label)
            rows.append(m)
    return pd.DataFrame(rows)


def rq2_floor(sc, group_col, budget=0.01, delta=0.002, floors=(None, 2.0, 1.5, 1.0), detectors=("IF", "GBM", "LSTM")):
    """Weaker-evidence check: FAIR-IRS with a score floor. Only employees within the
    global top (floor * k) of the day may be alerted; unfilled slots stay empty.
    floor = 1 alerts only people the global policy would also have alerted (leveling down)."""
    rows = []
    for name in detectors:
        if name not in sc:
            continue
        base = simulate_policy(sc[name], PolicyConfig(budget=budget, delta=1.0, group_col=group_col))
        mb = policy_metrics(base, group_col)
        mb.update(detector=name, floor="global (B0)")
        rows.append(mb)
        for f in floors:
            out = simulate_policy(sc[name], PolicyConfig(budget=budget, delta=delta, group_col=group_col, floor_mult=f))
            m = policy_metrics(out, group_col)
            m.update(detector=name, floor="none" if f is None else f"{f:g}k")
            rows.append(m)
    return pd.DataFrame(rows)


def rq1_audit_outcomes(ds, sc, budget=0.01, delta=0.002, detectors=("IF", "AE", "LR", "GBM", "LSTM"), pi_assumed=0.05):
    """Label-free (Lemma 1) versus outcome-based audit, for global and FAIR-IRS policies."""
    rows = []
    for name in detectors:
        if name not in sc:
            continue
        for gc in ds.group_cols:
            for label, d in (("global", 1.0), ("FAIR-IRS", delta)):
                out = simulate_policy(sc[name], PolicyConfig(budget=budget, delta=d, group_col=gc))
                m = policy_metrics(out, gc, pi_assumed=pi_assumed)
                m.update(detector=name, grouping=gc, policy=label)
                rows.append(m)
    return pd.DataFrame(rows)


def recall_ci_str(df):
    """'0.961 [0.868, 0.989]' strings for tables."""
    return [f"{r:.3f} [{lo:.2f}, {hi:.2f}]" for r, lo, hi in zip(df["recall_insider"], df["recall_lo"], df["recall_hi"])]

In [ ]:
%%writefile fairirs_rev.py
"""Additional experiments requested in the review of the FAIR-IRS paper.

E1  Heterogeneous prevalence across groups (M2)
E2  Adaptive attacks on cumulative group allocation (M5)
E3  Sensitivity to role-level behavioural differences (M1)
E4  Several persistent benign behaviour patterns (M7)

All experiments reuse the cached detector scores of the main notebook.
"""
import time
import numpy as np
import pandas as pd
import fairirs as fi


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


# =============================================================================
# Allocation policy with configurable history (no scheduling)
# =============================================================================

def simulate_rev(sd, budget=0.01, delta=0.002, group_col="g_functional_unit", history="alerts",
                 p_miss=0.0, cap=None, record=False, seed=0, floor=None, schedule=False,
                 lam=0.01, gamma=0.8, tau_ov=0.999, m_cap=3, horizon=30):
    """Day-by-day alerting without scheduling.

    history='alerts'  : parity on cumulative alert rates (FAIR-IRS as in the paper)
    history='cleared' : parity on cumulative cleared-alert rates, i.e., alerts that
                        investigations found benign, over user-days not confirmed
                        malicious (both observable by the security team)
    cap               : each employee contributes at most `cap` alerts per calendar
                        month to its group's cumulative count (limits budget exhaustion)
    record            : also return per-day, per-group allocation (K_g, k_g)
    floor             : score floor theta (only the global top ceil(theta*k) may be alerted)
    schedule          : burden-aware scheduling, exactly as in fairirs.simulate_policy
    """
    rng = np.random.default_rng(seed)
    T = sd.test
    days = T["day"].to_numpy()
    users = T["user"].to_numpy()
    y = T["y"].to_numpy()
    s = T["score"].to_numpy().astype(np.float64)
    g = T[group_col].to_numpy()
    alert = np.zeros(len(T), bool)
    cleared = np.zeros(len(T), bool)
    cumA, cumC, cumConf, per_user = {}, {}, {}, {}
    fallbacks, budget_total, disp = 0, 0, []
    rec = []
    store, n_disc = {}, 0
    if schedule:
        phin = sd.phi / (np.linalg.norm(sd.phi, axis=1, keepdims=True) + 1e-9)
    uniq_days, starts = np.unique(days, return_index=True)
    bounds = list(starts) + [len(T)]
    for di, d in enumerate(uniq_days):
        idx = np.arange(bounds[di], bounds[di + 1])
        sc = s[idx].copy()
        if schedule:
            for j, i in enumerate(idx):
                hist = store.get(users[i])
                if not hist:
                    continue
                hist[:] = [(dd, v) for dd, v in hist if (d - dd) / np.timedelta64(1, "D") <= horizon]
                if not hist or sc[j] >= tau_ov:
                    continue
                if max(float(phin[i] @ v) for _, v in hist) >= gamma:
                    sc[j] -= lam * min(len(hist), m_cap)
                    n_disc += 1
        k = max(1, int(round(budget * len(idx))))
        if history == "cleared":
            Ceff = {lab: cumC.get(lab, 0) - cumConf.get(lab, 0) for lab in cumC}
            m, fb = fi.fair_allocate_cum(sc, g[idx], k, delta, cumA, Ceff)
        else:
            m, fb = fi.fair_allocate_cum(sc, g[idx], k, delta, cumA, cumC)
        fallbacks += fb
        rank = np.empty(len(idx), int)
        rank[np.argsort(-sc, kind="stable")] = np.arange(1, len(idx) + 1)
        if floor is not None:
            m = m & (rank <= int(np.ceil(floor * k)))
        disp.extend((rank[m & (rank > k)] / k).tolist())
        budget_total += k
        if record:
            top = rank <= k
            for lab in np.unique(g[idx]):
                sel = g[idx] == lab
                rec.append((d, lab, int(top[sel].sum()), int(m[sel].sum())))
        a_idx = idx[m]
        alert[a_idx] = True
        u = rng.random(len(a_idx))
        clr = np.where(y[a_idx] == 1, u < p_miss, True)
        cleared[a_idx[clr]] = True
        if schedule:
            for i in a_idx[clr]:
                store.setdefault(users[i], []).append((d, phin[i]))
        for lab, cnt in zip(*np.unique(g[idx], return_counts=True)):
            cumC[lab] = cumC.get(lab, 0) + cnt
        counted = a_idx[clr] if history == "cleared" else a_idx
        for i in counted:
            if cap is not None:
                key = (users[i], str(d)[:7])
                if per_user.get(key, 0) >= cap:
                    continue
                per_user[key] = per_user.get(key, 0) + 1
            cumA[g[i]] = cumA.get(g[i], 0) + 1
        conf = a_idx[~clr]
        for i in conf:
            cumConf[g[i]] = cumConf.get(g[i], 0) + 1
    out = T[["user", "day", "y", "score", "exposure", "scenario", group_col]].copy()
    out["alert"], out["cleared"] = alert, cleared
    out.attrs.update(fallback_days=fallbacks, discounts=n_disc, n_days=len(uniq_days),
                     disp_ranks=np.array(disp), budget_total=budget_total)
    if record:
        return out, pd.DataFrame(rec, columns=["day", "group", "K", "k"])
    return out


def _with_test(sd, T, name=None):
    return fi.ScoredData(name or sd.detector, T.reset_index(drop=True), sd.phi, sd.model,
                         sd.transform, sd.raw_ref, sd.train_ref)


def recall_of(out, users):
    users = set(users)
    mal = out[(out["y"] == 1) & out["user"].isin(users)]
    if mal.empty:
        return float("nan"), 0, 0
    det = mal.loc[mal["alert"], "user"].nunique()
    n = mal["user"].nunique()
    return det / n, det, n


# =============================================================================
# E1  Heterogeneous prevalence
# =============================================================================

def inject_prevalence(sd, group_col, target, pi_target, seed=0):
    """Raise the share of malicious user-days of group `target` to pi_target by
    turning episodes of benign members into synthetic insiders whose scores are
    drawn from the detector's scores on real malicious user-days."""
    rng = np.random.default_rng(seed)
    T = sd.test.copy()
    mal_scores = T.loc[T["y"] == 1, "score"].to_numpy()
    ep_len = T[T["y"] == 1].groupby("user").size().to_numpy()
    real_ins = set(T.loc[T["y"] == 1, "user"])
    in_g = T[group_col] == target
    n_g = int(in_g.sum())
    need = int(round(pi_target * n_g)) - int(T.loc[in_g, "y"].sum())
    synth = []
    if need <= 0:
        return _with_test(sd, T), synth
    cand = [u for u in T.loc[in_g, "user"].unique() if u not in real_ins]
    rng.shuffle(cand)
    for u in cand:
        if need <= 0:
            break
        rows = T.index[(T["user"] == u) & in_g].to_numpy()
        if len(rows) < 5:
            continue
        L = int(min(len(rows), max(1, rng.choice(ep_len)), need))
        start = rng.integers(0, len(rows) - L + 1)
        sel = rows[start:start + L]
        T.loc[sel, "y"] = 1
        T.loc[sel, "score"] = rng.choice(mal_scores, size=L) + rng.uniform(-1e-6, 1e-6, L)
        T.loc[sel, "scenario"] = "synthetic"
        synth.append(u)
        need -= L
    return _with_test(sd, T), synth


def e1_prevalence(sc, budget=0.01, delta=0.002, group_col="g_functional_unit",
                  detectors=("GBM", "LSTM", "IF"), pis=(0.01, 0.02, 0.05, 0.10), seeds=(0, 1, 2)):
    rows = []
    base = next(iter(sc.values())).test
    users_per = base.groupby(group_col)["user"].nunique().sort_values()
    targets = [users_per.index[0], users_per.index[-1]]
    log("E1 target groups:", targets, "users:", [int(users_per[t]) for t in targets])
    policies = [("B0 global", dict(delta=1.0)),
                ("B4 alert parity", dict(delta=delta)),
                ("B4c cleared-alert parity", dict(delta=delta, history="cleared")),
                ("B0 global, p_miss 0.3", dict(delta=1.0, p_miss=0.3)),
                ("B4 alert parity, p_miss 0.3", dict(delta=delta, p_miss=0.3))]
    for det in detectors:
        if det not in sc:
            log(f"[skip] E1 {det}: not in the scored cache")
            continue
        sd0 = sc[det]
        real = set(sd0.test.loc[sd0.test["y"] == 1, "user"])
        settings = [(None, 0.0, 0)] + [(t, p, s) for t in targets for p in pis for s in seeds]
        for target, pi_t, seed in settings:
            if target is None:
                sd, synth = sd0, []
            else:
                sd, synth = inject_prevalence(sd0, group_col, target, pi_t, seed)
            pi_g = sd.test.groupby(group_col)["y"].mean()
            for pol, kw in policies:
                out = simulate_rev(sd, budget=budget, group_col=group_col, seed=seed, **kw)
                m = fi.policy_metrics(out, group_col, pi_assumed=0.05)
                m15 = fi.policy_metrics(out, group_col, pi_assumed=0.15)
                ben = out[out["y"] == 0]
                far = ben.groupby(group_col)["alert"].mean()
                r_real = recall_of(out, real)
                r_syn = recall_of(out, synth)
                rows.append(dict(detector=det, target=target or "none", target_users=int(users_per.get(target, 0)) if target else 0,
                                 pi_target=pi_t, pi_actual=float(pi_g.get(target, np.nan)) if target else float(pi_g.max()),
                                 seed=seed, policy=pol,
                                 delta_AR=m["delta_AR"], delta_FAR=m["delta_FAR"], ratio_FAR=m["ratio_FAR"],
                                 delta_FAR_hat=m["delta_FAR_hat"], audit_abs_err=m["audit_abs_err"],
                                 audit_bound_5=m["audit_bound"], audit_holds_5=m["audit_holds"],
                                 audit_bound_15=m15["audit_bound"], audit_holds_15=m15["audit_holds"],
                                 lemma1_bound=m["lemma1_bound"],
                                 far_target=float(far.get(target, np.nan)) if target else np.nan,
                                 far_others_mean=float(far.drop(target).mean()) if target else float(far.mean()),
                                 recall_real=r_real[0], detected_real=r_real[1], n_real=r_real[2],
                                 recall_synth=r_syn[0], detected_synth=r_syn[1], n_synth=r_syn[2],
                                 precision=m["precision"], fallback_days=m["fallback_days"]))
        log(f"E1 {det} done")
    return pd.DataFrame(rows)


def e1_summary(df):
    keys = ["detector", "target", "target_users", "pi_target", "policy"]
    num = ["pi_actual", "delta_AR", "delta_FAR", "ratio_FAR", "delta_FAR_hat", "audit_abs_err",
           "audit_bound_5", "audit_bound_15", "lemma1_bound", "far_target", "far_others_mean",
           "recall_real", "recall_synth", "precision"]
    s = df.groupby(keys, sort=False)[num].mean().reset_index()
    h = df.groupby(keys, sort=False)[["audit_holds_5", "audit_holds_15"]].mean().reset_index()
    return s.merge(h, on=keys)


# =============================================================================
# E2  Adaptive attacks on cumulative allocation
# =============================================================================

def _episodes(T):
    ep = {}
    for u, D in T[T["y"] == 1].groupby("user"):
        ep[u] = D.index.to_numpy()
    return ep


def shift_episodes(sd, group_col, rec=None, mode="adaptive", seed=0):
    """Move each insider's malicious episode to other active days of the same
    insider (scores and labels are swapped with the benign days they replace).
    mode='adaptive': start where the insider's group loses the most alerts under
    FAIR-IRS (oracle knowledge of the unattacked allocation, an upper bound on
    attacker knowledge); mode='random': uniformly random start."""
    rng = np.random.default_rng(seed)
    T = sd.test.copy()
    cut = None
    if rec is not None:
        r = rec.copy()
        r["cut"] = (r["K"] - r["k"]).clip(lower=0)
        cut = r.set_index(["day", "group"])["cut"].to_dict()
    for u, mal_idx in _episodes(T).items():
        rows = T.index[T["user"] == u].to_numpy()          # sorted by day
        L = len(mal_idx)
        if len(rows) <= L:
            continue
        starts = np.arange(0, len(rows) - L + 1)
        if mode == "adaptive":
            gu = T.loc[rows, group_col].to_numpy()
            dd = T.loc[rows, "day"].to_numpy()
            c = np.array([cut.get((dd[j], gu[j]), 0) for j in range(len(rows))], float)
            win = np.convolve(c, np.ones(L), "valid")
            best = np.flatnonzero(win == win.max())
            st = int(rng.choice(best))
        else:
            st = int(rng.choice(starts))
        new = rows[st:st + L]
        old_scores = T.loc[mal_idx, "score"].to_numpy().copy()
        moved_out = [i for i in new if i not in set(mal_idx)]
        freed = [i for i in mal_idx if i not in set(new)]
        ben_scores = T.loc[moved_out, "score"].to_numpy().copy()
        T.loc[mal_idx, "y"] = 0
        T.loc[freed, "score"] = ben_scores
        T.loc[new, "score"] = old_scores
        T.loc[new, "y"] = 1
    return _with_test(sd, T)


def flood_group(sd, group_col, n_acc=3, window=10, seed=0):
    """Accomplices: n_acc benign colleagues of each insider (same group) make
    themselves maximally suspicious on their active days in the `window` days
    before the insider's first malicious day, so that the group's cumulative
    alert count rises before the attack."""
    rng = np.random.default_rng(seed)
    T = sd.test.copy()
    ins = set(T.loc[T["y"] == 1, "user"])
    acc_rows = set()
    for u, D in T[T["y"] == 1].groupby("user"):
        d0 = D["day"].min()
        gu = D[group_col].iloc[0]
        lo = d0 - pd.Timedelta(days=window)
        pool = T[(T[group_col] == gu) & (T["day"] >= lo) & (T["day"] < d0) & ~T["user"].isin(ins)]
        cands = pool["user"].unique()
        if len(cands) == 0:
            continue
        acc = rng.choice(cands, size=min(n_acc, len(cands)), replace=False)
        acc_rows |= set(pool.index[pool["user"].isin(acc)])
    acc_rows = np.array(sorted(acc_rows), int)
    T.loc[acc_rows, "score"] = 1.0 + rng.uniform(0, 1e-3, len(acc_rows))
    return _with_test(sd, T), len(acc_rows)


def e2_attacks(sc, budget=0.01, delta=0.002, group_col="g_functional_unit",
               detectors=("GBM", "LSTM", "IF"), seeds=(0, 1, 2, 3, 4)):
    rows = []
    pols = [("B0 global", dict(delta=1.0)), ("B4 FAIR-IRS", dict(delta=delta)),
            ("B4 FAIR-IRS, cap 3", dict(delta=delta, cap=3))]
    for det in detectors:
        if det not in sc:
            log(f"[skip] E2 {det}: not in the scored cache")
            continue
        sd0 = sc[det]
        _, rec = simulate_rev(sd0, budget, delta, group_col, record=True)
        scen = [("none", 0, sd0, 0)]
        scen.append(("timing, adaptive", 0, shift_episodes(sd0, group_col, rec, "adaptive", 0), 0))
        for s in seeds:
            scen.append(("timing, random", s, shift_episodes(sd0, group_col, None, "random", s), 0))
        for s in seeds:
            sdf, nacc = flood_group(sd0, group_col, seed=s)
            scen.append(("flooding (3 accomplices, 10 days)", s, sdf, nacc))
        for attack, s, sd, nacc in scen:
            ins = set(sd.test.loc[sd.test["y"] == 1, "user"])
            for pol, kw in pols:
                out = simulate_rev(sd, budget=budget, group_col=group_col, **kw)
                r = recall_of(out, ins)
                m = fi.policy_metrics(out, group_col)
                rows.append(dict(detector=det, attack=attack, seed=s, policy=pol, recall=r[0],
                                 detected=r[1], insiders=r[2], delta_FAR=m["delta_FAR"],
                                 ratio_FAR=m["ratio_FAR"], accomplice_days=nacc))
        log(f"E2 {det} done")
    return pd.DataFrame(rows)


def e2_summary(df):
    g = df.groupby(["detector", "attack", "policy"], sort=False)
    s = g[["recall", "detected", "delta_FAR", "ratio_FAR", "accomplice_days"]].mean()
    s["recall_sd"] = g["recall"].std().fillna(0)
    s["runs"] = g.size()
    return s.reset_index()


# =============================================================================
# E3  Role-contrast sensitivity
# =============================================================================

def make_role_transform(alpha):
    """Feature transform that keeps a fraction alpha of the between-role
    differences in mean (log) activity: alpha=1 reproduces the paper's
    features, alpha=0 removes role-level mean differences entirely."""
    class RoleTransform(fi.FeatureTransform):
        def __init__(self, mode="raw"):
            super().__init__("raw")

        def fit(self, df):
            Z = self._base(df)
            mu = Z.mean(0)
            roles = df["role"].astype(str).to_numpy()
            rm = pd.DataFrame(Z).groupby(roles).mean()
            sh = (1 - alpha) * (rm.to_numpy() - mu)
            sh[:, -1] = 0.0                                   # keep the weekend indicator
            self.shift = dict(zip(rm.index, sh))
            Zs = Z - self._shift(df)
            self.mu = Zs.mean(0)
            self.sd = Zs.std(0) + 1e-6
            return self

        def _shift(self, df):
            zero = np.zeros(len(COLS))
            return np.vstack([self.shift.get(r, zero) for r in df["role"].astype(str).to_numpy()])

        def transform(self, df):
            return (((self._base(df) - self._shift(df)) - self.mu) / self.sd).astype(np.float32)
    COLS = fi.COUNT_FEATURES + ["weekend"]
    return RoleTransform


def e3_role_contrast(ds, budget=0.01, delta=0.002, alphas=(1.0, 0.5, 0.0),
                     groupings=("g_functional_unit", "g_department", "g_role"), seed=0):
    rows = []
    orig = fi.FeatureTransform
    try:
        for a in alphas:
            fi.FeatureTransform = make_role_transform(a)
            for det in [fi.IFDetector(seed), fi.AEDetector(seed), fi.LRDetector(), fi.GBMDetector(seed)]:
                try:
                    sd = fi.run_detector(ds, det)
                except Exception as ex:
                    log(f"[skip] E3 {det.name} alpha={a}: {type(ex).__name__}: {ex}")
                    continue
                if sd is None:
                    continue
                eb = fi.exposure_bias(sd.test)
                for gc in groupings:
                    for pol, dl in [("B0 global", 1.0), ("B4 FAIR-IRS", delta)]:
                        out = fi.simulate_policy(sd, fi.PolicyConfig(budget=budget, delta=dl, group_col=gc))
                        m = fi.policy_metrics(out, gc)
                        rows.append(dict(alpha=a, detector=sd.detector, grouping=gc, policy=pol,
                                         recall=m["recall_insider"], detected=m["insiders_detected"],
                                         delta_FAR=m["delta_FAR"], ratio_FAR=m["ratio_FAR"],
                                         gini=m["gini_burden"], max_F=m["max_alerts_one_benign_user"],
                                         top1=m["top1pct_share"], exposure_bias=eb))
                log(f"E3 alpha={a} {sd.detector} done")
    finally:
        fi.FeatureTransform = orig
    return pd.DataFrame(rows)


# =============================================================================
# E4  Several persistent benign patterns
# =============================================================================

PATTERNS = {
    "after-hours work": ["logon_ah", "file_ah", "http_ah", "email_ah"],
    "removable media": ["dev_n", "dev_ah", "file_n"],
    "external e-mail": ["email_ext", "email_att", "email_kb"],
}


def inject_patterns(sd, per_pattern=10, day_share=0.7, strength=1.0, seed=0):
    """Give `per_pattern` benign employees per pattern a persistent, legitimate
    habit: on a random `day_share` of their test days, each feature of the
    pattern is increased by `strength` times its 99th percentile among benign
    test user-days. The detector re-scores the modified days."""
    if sd.model is None or getattr(sd.model, "sequence", False) or isinstance(sd.model, fi.PerGroupDetector):
        raise ValueError("pattern injection needs a single-model, non-sequence detector")
    rng = np.random.default_rng(seed)
    T = sd.test.copy()
    ins = set(T.loc[T["y"] == 1, "user"])
    ben = [u for u in T["user"].unique() if u not in ins]
    chosen = rng.choice(ben, size=min(len(ben), per_pattern * len(PATTERNS)), replace=False)
    who = {}
    for j, (pname, feats) in enumerate(PATTERNS.items()):
        feats = [f for f in feats if f in T.columns]
        q = T.loc[T["y"] == 0, feats].quantile(0.99).clip(lower=1).to_numpy()
        for u in chosen[j * per_pattern:(j + 1) * per_pattern]:
            rows = T.index[T["user"] == u].to_numpy()
            sel = rows[rng.random(len(rows)) < day_share]
            T.loc[sel, feats] = (T.loc[sel, feats].to_numpy() + strength * q).round()
            who[u] = pname
    T["exposure"] = T[fi.VOLUME_FEATURES].sum(axis=1)
    X = sd.transform.transform(T)
    raw = sd.model.score(X)
    T["raw_score"] = raw
    T["score"] = fi.to_quantile_ref(raw, sd.raw_ref)
    return fi.ScoredData(sd.detector + "-patterns", T, X.copy(), sd.model, sd.transform,
                         sd.raw_ref, sd.train_ref), who


def e4_patterns(sc, budget=0.01, delta=0.002, group_col="g_functional_unit",
                detectors=("GBM", "IF", "AE"), seeds=(0, 1, 2)):
    rows = []
    pols = [("B0 global", dict(delta=1.0)), ("B4 FAIR-IRS", dict(delta=delta)),
            ("B5 scheduling, lambda 0.01", dict(delta=delta, schedule=True, lam=0.01)),
            ("B5 scheduling, lambda 0.02", dict(delta=delta, schedule=True, lam=0.02))]
    for det in detectors:
        if det not in sc:
            log(f"[skip] E4 {det}: not in the scored cache")
            continue
        for s in seeds:
            try:
                sd, who = inject_patterns(sc[det], seed=s)
            except Exception as ex:
                log(f"[skip] E4 {det}: {type(ex).__name__}: {ex}")
                break
            for pol, kw in pols:
                out = fi.simulate_policy(sd, fi.PolicyConfig(budget=budget, group_col=group_col, **kw))
                m = fi.policy_metrics(out, group_col)
                ins = set(out.loc[out["y"] == 1, "user"])
                F = out[~out["user"].isin(ins)].groupby("user")["cleared"].sum()
                inj = F[F.index.isin(who.keys())]
                oth = F[~F.index.isin(who.keys())]
                row = dict(detector=det, seed=s, policy=pol, recall=m["recall_insider"],
                           detected=m["insiders_detected"], tpr_day=m["tpr_day"],
                           delta_FAR=m["delta_FAR"], gini=m["gini_burden"], top1=m["top1pct_share"],
                           max_F_all=int(F.max()), mean_F_injected=float(inj.mean()), max_F_injected=int(inj.max()),
                           max_F_others=int(oth.max()), n_F_ge_10=int((F >= 10).sum()),
                           n_alerted_benign=int((F > 0).sum()), false_alerts=int(F.sum()))
                for pname in PATTERNS:
                    us = [u for u, p in who.items() if p == pname]
                    row[f"mean_F_{pname}"] = float(F.reindex(us).fillna(0).mean())
                rows.append(row)
        log(f"E4 {det} done")
    return pd.DataFrame(rows)


def e4_summary(df):
    num = [c for c in df.columns if c not in ("detector", "seed", "policy")]
    return df.groupby(["detector", "policy"], sort=False)[num].mean().reset_index()


# =============================================================================
# F1  Parity on cleared alerts: main results for every detector
# =============================================================================

def f1_cleared(sc, budget=0.01, delta=0.002, group_col="g_functional_unit",
               deltas=(1.0, 0.01, 0.005, 0.002, 0.001, 0.0)):
    """Policies with parity on alert rates (B4, as in the paper) and on cleared-alert
    rates (B4c), with delta sweep, score floors and scheduling."""
    rows = []
    dets = [d for d in ("IF", "AE", "LR", "GBM", "LSTM") if d in sc]
    for det in dets:
        sd = sc[det]
        settings = []
        for dl in deltas:
            for h in ("alerts", "cleared"):
                if dl >= 1 and h == "cleared":
                    continue
                settings.append((h, dl, None, False, 0.0))
        for h in ("alerts", "cleared"):
            for th in (2.0, 1.5, 1.0):
                settings.append((h, delta, th, False, 0.0))
            for lam in (0.01, 0.02):
                settings.append((h, delta, None, True, lam))
        for h, dl, th, sch, lam in settings:
            out = simulate_rev(sd, budget=budget, delta=dl, group_col=group_col, history=h,
                               floor=th, schedule=sch, lam=lam if sch else 0.01)
            m = fi.policy_metrics(out, group_col)
            m.update(detector=det, history=h, delta=dl, floor=th if th else "none",
                     schedule=sch, lam=lam)
            rows.append(m)
        log(f"F1 {det} done")
    return pd.DataFrame(rows)


def f1_groupings(sc, budget=0.01, delta=0.002, groupings=("g_functional_unit", "g_department", "g_role",
                                                          "g_afterhours", "g_exposure")):
    rows = []
    for det in [d for d in ("IF", "AE", "LR", "GBM", "LSTM") if d in sc]:
        for gc in groupings:
            if gc not in sc[det].test.columns:
                continue
            for h, dl in (("global", 1.0), ("alerts", delta), ("cleared", delta)):
                out = simulate_rev(sc[det], budget=budget, delta=dl, group_col=gc,
                                   history="alerts" if h == "global" else h)
                m = fi.policy_metrics(out, gc)
                m.update(detector=det, grouping=gc, policy=h)
                rows.append(m)
        log(f"F1 groupings {det} done")
    return pd.DataFrame(rows)


def f1_release(sc, release, budget=0.01, delta=0.002, group_col="g_functional_unit"):
    rows = []
    for det in [d for d in ("IF", "AE", "LR", "GBM", "LSTM") if d in sc]:
        for h, dl in (("global", 1.0), ("alerts", delta), ("cleared", delta)):
            out = simulate_rev(sc[det], budget=budget, delta=dl, group_col=group_col,
                               history="alerts" if h == "global" else h)
            m = fi.policy_metrics(out, group_col)
            m.update(release=release, detector=det, policy=h)
            rows.append(m)
    return pd.DataFrame(rows)


# =============================================================================
# F2  Timing attacks against several policies and attacker knowledge levels
# =============================================================================

def report_targets(out, group_col):
    """Monthly alert rate of each group relative to the mean over groups, as a
    burden report would publish it (without noise: an upper bound on what the
    attacker learns)."""
    o = out.copy()
    o["month"] = o["day"].dt.strftime("%Y-%m")
    ar = o.groupby(["month", group_col])["alert"].mean().unstack()
    return ar.sub(ar.mean(axis=1), axis=0)


def shift_to(sd, group_col, chooser, seed=0):
    """Move each insider's malicious episode; chooser(u, rows, T) returns the
    start position among the insider's active test days."""
    rng = np.random.default_rng(seed)
    T = sd.test.copy()
    for u, mal_idx in _episodes(T).items():
        rows = T.index[T["user"] == u].to_numpy()
        L = len(mal_idx)
        if len(rows) <= L:
            continue
        st = chooser(u, rows, T, L, rng)
        st = int(min(max(st, 0), len(rows) - L))
        new = rows[st:st + L]
        mal_set, new_set = set(mal_idx), set(new)
        old_scores = T.loc[mal_idx, "score"].to_numpy().copy()
        moved_out = [i for i in new if i not in mal_set]
        freed = [i for i in mal_idx if i not in new_set]
        ben_scores = T.loc[moved_out, "score"].to_numpy().copy()
        T.loc[mal_idx, "y"] = 0
        T.loc[freed, "score"] = ben_scores
        T.loc[new, "score"] = old_scores
        T.loc[new, "y"] = 1
    return _with_test(sd, T)


def f2_timing(sc, budget=0.01, delta=0.002, group_col="g_functional_unit",
              detectors=("GBM", "LSTM", "IF"), seeds=(0, 1, 2, 3, 4)):
    policies = [("B0 global", dict(delta=1.0)),
                ("B4 alert parity, delta 0.002", dict(delta=delta)),
                ("B4 alert parity, delta 0.005", dict(delta=0.005)),
                ("B4 alert parity, delta 0.01", dict(delta=0.01)),
                ("B4 alert parity, floor 2", dict(delta=delta, floor=2.0)),
                ("B4c cleared parity, delta 0.002", dict(delta=delta, history="cleared"))]
    rows = []
    for det in detectors:
        if det not in sc:
            log(f"[skip] F2 {det}: not in the scored cache")
            continue
        sd0 = sc[det]
        for pol, kw in policies:
            out0, rec = simulate_rev(sd0, budget=budget, group_col=group_col, record=True, **kw)
            r = rec.copy()
            r["cut"] = (r["K"] - r["k"]).clip(lower=0)
            cut = r.set_index(["day", "group"])["cut"].to_dict()
            rep = report_targets(out0, group_col)

            def oracle(u, rows_, T, L, rng):
                gu = T.loc[rows_, group_col].to_numpy(); dd = T.loc[rows_, "day"].to_numpy()
                c = np.array([cut.get((dd[j], gu[j]), 0) for j in range(len(rows_))], float)
                win = np.convolve(c, np.ones(L), "valid")
                return int(rng.choice(np.flatnonzero(win == win.max())))

            def informed(u, rows_, T, L, rng):
                # start in the month after the published report in which the
                # insider's group was most over-alerted
                gu = T.loc[rows_[0], group_col]
                months = T.loc[rows_, "day"].dt.strftime("%Y-%m").to_numpy()
                if gu not in rep.columns:
                    return int(rng.integers(0, len(rows_) - L + 1))
                ex = rep[gu].dropna()
                order = list(ex.sort_values(ascending=False).index)
                allm = sorted(rep.index)
                for mth in order:
                    nxt = allm.index(mth) + 1
                    if nxt >= len(allm):
                        continue
                    pos = np.flatnonzero(months == allm[nxt])
                    if len(pos) and pos[0] <= len(rows_) - L:
                        return int(pos[0])
                return int(rng.integers(0, len(rows_) - L + 1))

            def rand(u, rows_, T, L, rng):
                return int(rng.integers(0, len(rows_) - L + 1))

            scen = [("none", 0, sd0),
                    ("oracle (knows the allocation)", 0, shift_to(sd0, group_col, oracle, 0)),
                    ("report-informed (monthly group rates)", 0, shift_to(sd0, group_col, informed, 0))]
            scen += [("random", s, shift_to(sd0, group_col, rand, s)) for s in seeds]
            for attack, s, sd in scen:
                out = out0 if attack == "none" else simulate_rev(sd, budget=budget, group_col=group_col, **kw)
                ins = set(sd.test.loc[sd.test["y"] == 1, "user"])
                rr = recall_of(out, ins)
                m = fi.policy_metrics(out, group_col)
                rows.append(dict(detector=det, policy=pol, attack=attack, seed=s, recall=rr[0],
                                 detected=rr[1], insiders=rr[2], delta_FAR=m["delta_FAR"],
                                 ratio_FAR=m["ratio_FAR"], cut_alerts=int(r["cut"].sum())))
        log(f"F2 {det} done")
    return pd.DataFrame(rows)


def f2_summary(df):
    g = df.groupby(["detector", "policy", "attack"], sort=False)
    s = g[["recall", "detected", "delta_FAR", "ratio_FAR", "cut_alerts"]].mean()
    s["detected_sd"] = g["detected"].std().fillna(0)
    s["runs"] = g.size()
    return s.reset_index()

In [ ]:
%%writefile fairirs_rev3.py
"""Experiments for the second review of the FAIR-IRS paper.

G1  Realistic investigation feedback: delays, missing, revised and group-dependent verdicts (M2)
G2  Total harm: distinct employees investigated and convex repeat costs (M4)
G3  Sequential attackers and interactions with floors and randomized tolerance (M5)
G4  Stronger fairness-aware baselines: daily parity without memory, FA*IR-style bands (M7)
G5  Disparity over time and during fallback (M3)
G6  Synthetic stress tests independent of CERT (M1)
"""
import time
import numpy as np
import pandas as pd
from scipy.stats import binom
import fairirs as fi
import fairirs_rev as fr


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


# =============================================================================
# General day-by-day simulator
# =============================================================================

def _fairstar(sc, gids, k, alpha=0.1):
    """Multi-group adaptation of FA*IR to alerting: each group receives between
    the alpha/2 and 1-alpha/2 binomial quantiles of its proportional share of the
    k alerts; within the bands, alerts follow the global score order."""
    labs, inv = np.unique(gids, return_inverse=True)
    n = np.bincount(inv, minlength=len(labs))
    p = n / n.sum()
    lo = np.minimum(binom.ppf(alpha / 2, k, p).astype(int), n)
    hi = np.minimum(np.maximum(binom.ppf(1 - alpha / 2, k, p).astype(int), lo), n)
    if lo.sum() > k:                                    # keep the minima feasible
        lo = np.floor(lo * k / lo.sum()).astype(int)
    order = np.argsort(-sc, kind="stable")
    m = np.zeros(len(sc), bool)
    cnt = np.zeros(len(labs), int)
    for g in range(len(labs)):                          # minimum representation
        if lo[g] > 0:
            idx = order[inv[order] == g][:lo[g]]
            m[idx] = True
            cnt[g] += len(idx)
    total = int(m.sum())
    for i in order:                                     # fill within the caps
        if total >= k:
            break
        if not m[i] and cnt[inv[i]] < hi[inv[i]]:
            m[i] = True
            cnt[inv[i]] += 1
            total += 1
    if total < k:                                       # caps too tight: global order
        for i in order:
            if total >= k:
                break
            if not m[i]:
                m[i] = True
                total += 1
    return m


def simulate_g(sd, budget=0.01, delta=0.002, group_col="g_functional_unit", history="alerts",
               allocator="fair", floor=None, delta_jitter=False,
               delay=0, missing=0.0, revise=0.0, revise_days=30,
               p_miss=0.0, p_fc=0.0, err_group=None, p_miss_g=0.0, p_fc_g=0.0,
               schedule=False, lam=0.01, gamma=0.8, tau_ov=0.999, m_cap=3, horizon=30,
               record=False, seed=0, protect=0.0, eta=0.0, dual_mode="prop"):
    """history: 'global' (top-k), 'alerts' (label-free), 'cleared' (outcome-aware),
    'none' (parity enforced on each day alone, no memory).
    Investigation feedback (affects the outcome-aware history and the final verdicts):
      delay        verdicts become known `delay` days after the alert
      missing      share of verdicts that never arrive (alert stays counted as cleared)
      revise       share of malicious alerts first cleared, then confirmed after revise_days
      p_miss       share of malicious alerts wrongly cleared for good
      p_fc         share of benign alerts wrongly confirmed as malicious
      err_group    group with its own error rates p_miss_g, p_fc_g
    Until its verdict is known, an alert counts as cleared (provisional clearance).
    protect      the global top ceil(protect*k) are always alerted; parity applies to the rest
    allocator='dual': online dual prices; each day the top k of score - mu_g are alerted,
                 with mu_g = eta*(r_g - r) ('prop') or mu_g += eta*(r_g - r) ('ascent'),
                 r_g the group's cumulative (alert or cleared-alert) rate and r the overall rate."""
    rng = np.random.default_rng(seed)
    T = sd.test
    days = T["day"].to_numpy()
    users = T["user"].to_numpy()
    y = T["y"].to_numpy()
    s = T["score"].to_numpy().astype(np.float64)
    g = T[group_col].to_numpy()
    alert = np.zeros(len(T), bool)
    cleared = np.zeros(len(T), bool)
    cumA, cumC, cumConf = {}, {}, {}
    pending = []                                        # (reveal day index, group)
    fb_days, budget_total, disp, rec = [], 0, [], []
    store, n_disc = {}, 0
    mu = {}
    if schedule:
        phin = sd.phi / (np.linalg.norm(sd.phi, axis=1, keepdims=True) + 1e-9)
    uniq_days, starts = np.unique(days, return_index=True)
    bounds = list(starts) + [len(T)]
    for di, d in enumerate(uniq_days):
        idx = np.arange(bounds[di], bounds[di + 1])
        # verdicts revealed by today
        keep = []
        for (rd, lab) in pending:
            if rd <= di:
                cumA[lab] = cumA.get(lab, 0) - 1
                cumConf[lab] = cumConf.get(lab, 0) + 1
            else:
                keep.append((rd, lab))
        pending = keep
        sc = s[idx].copy()
        if schedule:
            for j, i in enumerate(idx):
                hist = store.get(users[i])
                if not hist:
                    continue
                hist[:] = [(dd, v) for dd, v in hist if (d - dd) / np.timedelta64(1, "D") <= horizon]
                if not hist or sc[j] >= tau_ov:
                    continue
                if max(float(phin[i] @ v) for _, v in hist) >= gamma:
                    sc[j] -= lam * min(len(hist), m_cap)
                    n_disc += 1
        k = max(1, int(round(budget * len(idx))))
        dl = delta * rng.uniform(0, 2) if delta_jitter else delta
        if allocator == "dual" and history in ("alerts", "cleared") and dl < 1:
            adj = sc - np.array([mu.get(lab, 0.0) for lab in g[idx]])
            m = np.zeros(len(idx), bool)
            m[np.argsort(-adj, kind="stable")[:k]] = True
            fb = 0
        elif protect > 0 and allocator == "fair" and history in ("alerts", "cleared", "none") and dl < 1:
            order = np.argsort(-sc, kind="stable")
            P = order[:min(k, int(np.ceil(protect * k)))]
            rest = np.setdiff1d(np.arange(len(idx)), P)
            if history == "none":
                A0, C0 = {}, {}
            elif history == "cleared":
                A0 = dict(cumA); C0 = {lab: cumC.get(lab, 0) - cumConf.get(lab, 0) for lab in cumC}
            else:
                A0, C0 = dict(cumA), dict(cumC)
            for j in P:
                lab = g[idx][j]
                A0[lab] = A0.get(lab, 0) + 1
                C0[lab] = C0.get(lab, 0) + 1
            m = np.zeros(len(idx), bool)
            m[P] = True
            fb = 0
            if k - len(P) > 0 and len(rest):
                mr, fb = fi.fair_allocate_cum(sc[rest], g[idx][rest], k - len(P), dl, A0, C0)
                m[rest[mr]] = True
        elif history == "global" or dl >= 1:
            m = np.zeros(len(idx), bool)
            m[np.argsort(-sc, kind="stable")[:k]] = True
            fb = 0
        elif allocator == "fairstar":
            m, fb = _fairstar(sc, g[idx], k), 0
        elif history == "none":
            m, fb = fi.fair_allocate_cum(sc, g[idx], k, dl, {}, {})
        elif history == "cleared":
            Ceff = {lab: cumC.get(lab, 0) - cumConf.get(lab, 0) for lab in cumC}
            m, fb = fi.fair_allocate_cum(sc, g[idx], k, dl, cumA, Ceff)
        else:
            m, fb = fi.fair_allocate_cum(sc, g[idx], k, dl, cumA, cumC)
        if fb:
            fb_days.append(di)
        rank = np.empty(len(idx), int)
        rank[np.argsort(-sc, kind="stable")] = np.arange(1, len(idx) + 1)
        if floor is not None:
            m = m & (rank <= int(np.ceil(floor * k)))
        disp.extend((rank[m & (rank > k)] / k).tolist())
        budget_total += k
        if record:
            top = rank <= k
            for lab in np.unique(g[idx]):
                sel = g[idx] == lab
                rec.append((d, lab, int(top[sel].sum()), int(m[sel].sum()), int(sel.sum())))
        a_idx = idx[m]
        alert[a_idx] = True
        for lab, cnt in zip(*np.unique(g[idx], return_counts=True)):
            cumC[lab] = cumC.get(lab, 0) + cnt
        for i in a_idx:
            lab = g[i]
            pm = p_miss_g if (err_group is not None and lab == err_group) else p_miss
            pf = p_fc_g if (err_group is not None and lab == err_group) else p_fc
            if y[i] == 1:
                final_clear = rng.random() < pm
            else:
                final_clear = not (rng.random() < pf)
            cleared[i] = final_clear
            if history == "alerts" or history == "none" or history == "global":
                cumA[lab] = cumA.get(lab, 0) + 1
                if schedule and final_clear:
                    store.setdefault(users[i], []).append((d, phin[i]))
                continue
            cumA[lab] = cumA.get(lab, 0) + 1            # provisional clearance
            if schedule and final_clear:
                store.setdefault(users[i], []).append((d, phin[i]))
            if final_clear or rng.random() < missing:
                continue
            if y[i] == 1 and rng.random() < revise:
                pending.append((di + max(revise_days, delay), lab))
            else:
                pending.append((di + delay, lab))
        if allocator == "dual":
            if history == "cleared":
                den = {lab: cumC[lab] - cumConf.get(lab, 0) for lab in cumC}
            else:
                den = dict(cumC)
            rates = {lab: cumA.get(lab, 0) / max(den[lab], 1) for lab in den}
            rbar = sum(cumA.get(lab, 0) for lab in den) / max(sum(den.values()), 1)
            for lab, r in rates.items():
                if dual_mode == "ascent":
                    mu[lab] = mu.get(lab, 0.0) + eta * (r - rbar)
                else:
                    mu[lab] = eta * (r - rbar)
    out = T[["user", "day", "y", "score", "exposure", "scenario", group_col]].copy()
    out["alert"], out["cleared"] = alert, cleared
    out.attrs.update(fallback_days=len(fb_days), fallback_list=fb_days, discounts=n_disc,
                     n_days=len(uniq_days), disp_ranks=np.array(disp), budget_total=budget_total)
    if record:
        return out, pd.DataFrame(rec, columns=["day", "group", "K", "k", "n"])
    return out


def harm(out):
    """Burden on benign employees beyond disparity."""
    ins = set(out.loc[out["y"] == 1, "user"])
    ben = out[~out["user"].isin(ins)]
    F = ben.groupby("user")["alert"].sum()
    Fc = ben.groupby("user")["cleared"].sum()
    return dict(distinct_alerted=int((F > 0).sum()), employees_F_ge2=int((F >= 2).sum()),
                employees_F_ge5=int((F >= 5).sum()), employees_F_ge10=int((F >= 10).sum()),
                false_alerts=int(F.sum()), cost_squared=float((F ** 2).sum()),
                cost_repeat=float((F * (F + 1) / 2).sum()), max_F=int(F.max()) if len(F) else 0,
                benign_employees=int(len(F)), wrongly_confirmed=int((ben["alert"] & ~ben["cleared"]).sum()))


def metrics(out, group_col, **extra):
    m = fi.policy_metrics(out, group_col)
    m.update(harm(out))
    m.update(extra)
    return m


# =============================================================================
# G1  Investigation feedback (M2)
# =============================================================================

def g1_feedback(sc, budget=0.01, delta=0.002, group_col="g_functional_unit",
                detectors=("GBM", "LSTM", "IF"), seeds=(0, 1, 2)):
    rows = []
    base = next(iter(sc.values())).test
    users_per = base.groupby(group_col)["user"].nunique().sort_values()
    small, large = users_per.index[0], users_per.index[-1]
    variants = [("correct, immediate", {})]
    variants += [(f"delay {d} days", dict(delay=d)) for d in (3, 7, 14, 30)]
    variants += [(f"missing {int(100*m)}%", dict(missing=m)) for m in (0.2, 0.5)]
    variants += [("revised after 30 days (30%)", dict(revise=0.3, revise_days=30)),
                 ("missed 30%, all groups", dict(p_miss=0.3)),
                 ("false confirmation 5%, all groups", dict(p_fc=0.05))]
    for tg, lab in ((small, "smallest"), (large, "largest")):
        variants += [(f"missed 30%, {lab} group only", dict(err_group=tg, p_miss_g=0.3)),
                     (f"false confirmation 5%, {lab} group only", dict(err_group=tg, p_fc_g=0.05)),
                     (f"false confirmation 20%, {lab} group only", dict(err_group=tg, p_fc_g=0.2))]
    variants += [("delay 7 days + missing 20% + missed 30%", dict(delay=7, missing=0.2, p_miss=0.3))]
    for det in detectors:
        if det not in sc:
            continue
        sd = sc[det]
        for pol, h in (("B0 global", "global"), ("B4 label-free", "alerts")):
            out = simulate_g(sd, budget, delta, group_col, history=h)
            rows.append(metrics(out, group_col, detector=det, policy=pol, feedback="(not used)", seed=0,
                                far_small=float(out[(out.y == 0) & (out[group_col] == small)]["alert"].mean()),
                                far_large=float(out[(out.y == 0) & (out[group_col] == large)]["alert"].mean())))
        for name, kw in variants:
            for s in seeds:
                out = simulate_g(sd, budget, delta, group_col, history="cleared", seed=s, **kw)
                rows.append(metrics(out, group_col, detector=det, policy="B4c outcome-aware", feedback=name, seed=s,
                                    far_small=float(out[(out.y == 0) & (out[group_col] == small)]["alert"].mean()),
                                    far_large=float(out[(out.y == 0) & (out[group_col] == large)]["alert"].mean())))
                if not kw:
                    break
        log(f"G1 {det} done")
    df = pd.DataFrame(rows)
    df.attrs["groups"] = (str(small), str(large))
    return df


# =============================================================================
# G2  Total harm (M4)
# =============================================================================

def g2_harm(sc, budget=0.01, delta=0.002, group_col="g_functional_unit", lams=(0.0, 0.01, 0.02, 0.05)):
    rows = []
    for det in [d for d in ("IF", "AE", "LR", "GBM", "LSTM") if d in sc]:
        sd = sc[det]
        out = simulate_g(sd, budget, delta, group_col, history="global")
        rows.append(metrics(out, group_col, detector=det, policy="B0 global", lam=0.0))
        for h, name in (("alerts", "label-free"), ("cleared", "outcome-aware")):
            for lam in lams:
                out = simulate_g(sd, budget, delta, group_col, history=h, schedule=lam > 0, lam=max(lam, 1e-9))
                rows.append(metrics(out, group_col, detector=det, policy=name, lam=lam))
        log(f"G2 {det} done")
    return pd.DataFrame(rows)


# =============================================================================
# G3  Sequential attackers (M5)
# =============================================================================

def g3_attacks(sc, budget=0.01, delta=0.002, group_col="g_functional_unit",
               detectors=("GBM", "LSTM"), seeds=(0, 1, 2, 3, 4), window=30):
    policies = [("B0 global", dict(history="global")),
                ("B4 label-free", dict(history="alerts")),
                ("B4c outcome-aware", dict(history="cleared")),
                ("B4 label-free, floor 2", dict(history="alerts", floor=2.0)),
                ("B4c outcome-aware, floor 2", dict(history="cleared", floor=2.0)),
                ("B4c outcome-aware, delta 0.005", dict(history="cleared", delta=0.005)),
                ("B4c outcome-aware, randomized delta", dict(history="cleared", delta_jitter=True))]
    rows = []
    for det in detectors:
        if det not in sc:
            continue
        sd0 = sc[det]
        for pol, kw in policies:
            kw = dict(kw)
            dl = kw.pop("delta", delta)
            out0, rec = simulate_g(sd0, budget, dl, group_col, record=True, **kw)
            r = rec.copy()
            r["cut"] = (r["K"] - r["k"]).clip(lower=0)
            cut = r.set_index(["day", "group"])["cut"].to_dict()
            # what an employee can observe: colleagues investigated in the past `window` days
            r = r.sort_values("day")
            obs = {}
            for lab, D in r.groupby("group"):
                D = D.sort_values("day")
                a = D["k"].rolling(window, min_periods=1).sum().shift(1).fillna(0)
                n = D["n"].rolling(window, min_periods=1).sum().shift(1).fillna(1)
                for dd, v in zip(D["day"], (a / n).to_numpy()):
                    obs[(dd, lab)] = v

            def oracle(u, rows_, T, L, rng):
                gu = T.loc[rows_, group_col].to_numpy(); dd = T.loc[rows_, "day"].to_numpy()
                c = np.array([cut.get((dd[j], gu[j]), 0) for j in range(len(rows_))], float)
                win = np.convolve(c, np.ones(L), "valid")
                return int(rng.choice(np.flatnonzero(win == win.max())))

            def observing(u, rows_, T, L, rng):
                # acts when colleagues were investigated most often in the past window
                gu = T.loc[rows_, group_col].to_numpy(); dd = T.loc[rows_, "day"].to_numpy()
                c = np.array([obs.get((dd[j], gu[j]), 0) for j in range(len(rows_) - L + 1)], float)
                return int(rng.choice(np.flatnonzero(c == c.max())))

            def rand(u, rows_, T, L, rng):
                return int(rng.integers(0, len(rows_) - L + 1))

            scen = [("none", 0, sd0), ("knows allocation", 0, fr.shift_to(sd0, group_col, oracle, 0)),
                    ("observes colleagues", 0, fr.shift_to(sd0, group_col, observing, 0))]
            scen += [("random", s, fr.shift_to(sd0, group_col, rand, s)) for s in seeds]
            for attack, s, sd in scen:
                out = out0 if attack == "none" else simulate_g(sd, budget, dl, group_col, **kw)
                ins = set(sd.test.loc[sd.test["y"] == 1, "user"])
                rr = fr.recall_of(out, ins)
                m = fi.policy_metrics(out, group_col)
                rows.append(dict(detector=det, policy=pol, attack=attack, seed=s, detected=rr[1],
                                 insiders=rr[2], delta_FAR=m["delta_FAR"], ratio_FAR=m["ratio_FAR"],
                                 alerts_used=m["alerts_used"], fallback_days=m["fallback_days"]))
        log(f"G3 {det} done")
    return pd.DataFrame(rows)


def g3_summary(df):
    g = df.groupby(["detector", "policy", "attack"], sort=False)
    s = g[["detected", "delta_FAR", "alerts_used", "fallback_days"]].mean()
    s["detected_sd"] = g["detected"].std().fillna(0)
    return s.reset_index()


# =============================================================================
# G4  Stronger baselines (M7)
# =============================================================================

def g4_baselines(sc, budget=0.01, delta=0.002, group_col="g_functional_unit"):
    pols = [("B0 global", dict(history="global")),
            ("daily parity, no memory (label-free)", dict(history="none")),
            ("FA*IR-style bands, alpha 0.1", dict(allocator="fairstar", history="alerts")),
            ("B4 label-free (cumulative)", dict(history="alerts")),
            ("B4c outcome-aware (cumulative)", dict(history="cleared"))]
    rows = []
    for det in [d for d in ("IF", "AE", "LR", "GBM", "LSTM") if d in sc]:
        for pol, kw in pols:
            out = simulate_g(sc[det], budget, delta, group_col, **kw)
            rows.append(metrics(out, group_col, detector=det, policy=pol))
        log(f"G4 {det} done")
    return pd.DataFrame(rows)


# =============================================================================
# G5  Disparity over time (M3)
# =============================================================================

def trajectory(out, group_col):
    o = out.copy()
    o["ben"] = (o["y"] == 0).astype(int)
    o["fa"] = (o["alert"] & (o["y"] == 0)).astype(int)
    o["cl"] = (o["alert"] & o["cleared"]).astype(int)
    o["conf"] = (o["alert"] & ~o["cleared"]).astype(int)
    o["n"] = 1
    D = o.groupby(["day", group_col])[["n", "alert", "ben", "fa", "cl", "conf"]].sum().groupby(level=1).cumsum()
    D = D.reset_index()
    D["AR"] = D["alert"] / D["n"]
    D["FAR"] = D["fa"] / D["ben"].clip(lower=1)
    D["CR"] = D["cl"] / (D["n"] - D["conf"]).clip(lower=1)
    t = D.groupby("day").agg(dAR=("AR", lambda v: v.max() - v.min()), dFAR=("FAR", lambda v: v.max() - v.min()),
                             dCR=("CR", lambda v: v.max() - v.min()))
    return t.reset_index()


def g5_over_time(sc, budget=0.01, delta=0.002, group_col="g_functional_unit", burn_in=30):
    pols = [("B0 global", dict(history="global")), ("B4 label-free", dict(history="alerts")),
            ("B4c outcome-aware", dict(history="cleared")),
            ("B4 label-free, floor 2", dict(history="alerts", floor=2.0)),
            ("B4 label-free, floor 1", dict(history="alerts", floor=1.0)),
            ("B4c outcome-aware, floor 2", dict(history="cleared", floor=2.0)),
            ("B4c outcome-aware, floor 1", dict(history="cleared", floor=1.0))]
    rows, traj = [], []
    for det in [d for d in ("IF", "AE", "LR", "GBM", "LSTM") if d in sc]:
        for pol, kw in pols:
            out = simulate_g(sc[det], budget, delta, group_col, **kw)
            t = trajectory(out, group_col)
            t["detector"], t["policy"] = det, pol
            traj.append(t)
            after = t.iloc[burn_in:]
            fbl = out.attrs["fallback_list"]
            fb_after = [i for i in fbl if i >= burn_in]
            target = "dCR" if kw.get("history") == "cleared" else "dAR"
            rows.append(dict(detector=det, policy=pol, constrained=target,
                             final=float(t[target].iloc[-1]), max_after_burn_in=float(after[target].max()),
                             share_days_above_delta=float((after[target] > delta + 1e-12).mean()),
                             max_excess=float(max(0.0, after[target].max() - delta)),
                             fallback_days=len(fbl), fallback_days_after_burn_in=len(fb_after),
                             final_true_dFAR=float(t["dFAR"].iloc[-1]),
                             max_true_dFAR_after_burn_in=float(after["dFAR"].max()),
                             alerts_used=out["alert"].sum() / out.attrs["budget_total"]))
        log(f"G5 {det} done")
    return pd.DataFrame(rows), pd.concat(traj, ignore_index=True)


# =============================================================================
# G6  Synthetic stress tests (M1)
# =============================================================================

def synth_data(sizes=(100, 100, 100, 100, 100), act_gap=0.0, risk_ratio=1.0, persistence=0.5,
               base_rate=0.004, n_days=250, episode=10, boost=2.5, seed=0):
    """Fully synthetic scores, independent of CERT.
    Benign score  = mu_g + u_i + e_it, with mu_g spread over [0, act_gap] across groups,
                    user effect u_i ~ N(0, persistence^2) and daily noise e_it ~ N(0, 1).
    Insiders      : prevalence proportional to risk_ratio^(g/(G-1)) (risk increases with
                    group activity when risk_ratio > 1), mean base_rate of user-days; each
                    insider has one episode of `episode` days with a score boost `boost`."""
    rng = np.random.default_rng(seed)
    G = len(sizes)
    mu = np.linspace(0, act_gap, G)
    w = risk_ratio ** (np.arange(G) / max(G - 1, 1))
    users, groups, ueff = [], [], []
    for gi, n in enumerate(sizes):
        for j in range(n):
            users.append(f"g{gi}_u{j}"); groups.append(f"G{gi}")
            ueff.append(mu[gi] + rng.normal(0, persistence))
    U = len(users)
    n_ins_total = max(1, int(round(base_rate * U * n_days / episode)))
    probs = np.repeat(w, sizes); probs = probs / probs.sum()
    ins_idx = rng.choice(U, size=min(n_ins_total, U), replace=False, p=probs)
    start = {i: rng.integers(n_days // 2, n_days - episode) for i in ins_idx}
    days = pd.date_range("2020-01-01", periods=n_days, freq="D")
    rows_u = np.repeat(np.arange(U), n_days)
    rows_d = np.tile(np.arange(n_days), U)
    score = np.array(ueff)[rows_u] + rng.normal(0, 1, U * n_days)
    y = np.zeros(U * n_days, int)
    for i, st in start.items():
        sel = (rows_u == i) & (rows_d >= st) & (rows_d < st + episode)
        y[sel] = 1
        score[sel] += boost
    T = pd.DataFrame({"user": np.array(users)[rows_u], "day": days[rows_d], "y": y, "score": score,
                      "exposure": np.array(ueff)[rows_u], "scenario": np.where(y == 1, "synthetic", ""),
                      "g_group": np.array(groups)[rows_u]})
    T = T[T["day"] >= days[n_days // 2]].sort_values(["day", "user"]).reset_index(drop=True)
    return fi.ScoredData("synthetic", T, None, None, None, None, None)


def g6_stress(budget=0.01, delta=0.002, seeds=(0, 1, 2)):
    settings = []
    for a in (0.0, 0.25, 0.5, 1.0):
        settings.append(("activity gap", a, dict(act_gap=a)))
    for name, sz in (("balanced", (100,) * 5), ("skewed", (20, 40, 80, 160, 200)), ("one tiny group", (10, 120, 120, 125, 125))):
        settings.append(("group sizes", name, dict(sizes=sz, act_gap=0.5)))
    for rr in (1.0, 3.0, 10.0):
        settings.append(("risk ratio", rr, dict(risk_ratio=rr, act_gap=0.5)))
    for p in (0.0, 0.5, 1.0):
        settings.append(("persistence", p, dict(persistence=p, act_gap=0.5)))
    pols = [("B0 global", dict(history="global")), ("daily parity, no memory", dict(history="none")),
            ("FA*IR-style bands", dict(allocator="fairstar", history="alerts")),
            ("B4 label-free", dict(history="alerts")), ("B4c outcome-aware", dict(history="cleared"))]
    rows = []
    for factor, level, kw in settings:
        for s in seeds:
            sd = synth_data(seed=s, **kw)
            for pol, pk in pols:
                out = simulate_g(sd, budget, delta, "g_group", **pk)
                m = metrics(out, "g_group")
                m.update(factor=factor, level=str(level), seed=s, policy=pol)
                rows.append(m)
        log(f"G6 {factor}={level} done")
    return pd.DataFrame(rows)


def summarize(df, keys, cols):
    return df.groupby(keys, sort=False)[cols].mean().reset_index()

In [ ]:
%%writefile fairirs_rev5.py
"""Experiments for the fifth review of the FAIR-IRS paper.

I1  LSTM retrained with several seeds (cached)
I2  Non-inferiority of detection, seed-pooled, for three budgets
I3  Disparity over rolling 30- and 90-day windows; windowed and discounted histories;
    deficit apportionment as a baseline
I4  Employee-clustered bootstrap intervals, removal of the most-flagged employees,
    reference Gini under random allocation
I5  Insiders removed after their first confirmed alert
I6  Missed insiders and false confirmations together, with a fallible re-checker;
    joint (Bonferroni) confidence level for the bound of Corollary 1
I7  Timing attackers over many draws, compared with the global policy on the SAME
    relocated episodes; deficit-estimating attacker; randomized displacement
I8  Tolerance chosen on a validation period (first half of the test period)
I9  Score ties and group counts
I10 Shadow month and prior warm start
I11 Power of a test on private monthly reports
"""
import math
import time
import numpy as np
import pandas as pd
from scipy.stats import beta as beta_dist
import fairirs as fi
import fairirs_rev as fr
import fairirs_rev3 as g3


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


# =============================================================================
# Simulator (extends fairirs_rev3.simulate_g; defaults reproduce it)
# =============================================================================

def _deficit(sc, gids, k, A, C):
    """Deficit apportionment used alone: today's alerts go to groups in proportion to
    their deficit with respect to the common cumulative rate (largest remainder);
    within a group, the highest scores are alerted."""
    labels, inv = np.unique(gids, return_inverse=True)
    G = len(labels)
    order = [np.nonzero(inv == g)[0] for g in range(G)]
    order = [o[np.argsort(-sc[o], kind="stable")] for o in order]
    n = np.array([len(o) for o in order])
    a = np.array([A.get(l, 0) for l in labels], float)
    D = np.array([C.get(l, 0) for l in labels], float) + n
    k = min(k, int(n.sum()))
    target = (k + a.sum()) * D / D.sum() - a
    e = np.clip(target, 0, n)
    if e.sum() <= 0:
        e = n.astype(float)
    exact = k * e / e.sum()
    kg = np.minimum(np.floor(exact).astype(int), n)
    rem = k - kg.sum()
    for g in np.argsort(-(exact - kg)):
        if rem <= 0:
            break
        if kg[g] < n[g]:
            kg[g] += 1
            rem -= 1
    while rem > 0:
        g = int(np.argmax(n - kg))
        kg[g] += 1
        rem -= 1
    m = np.zeros(len(sc), bool)
    for g in range(G):
        m[order[g][:kg[g]]] = True
    return m


def _randround(sc, gids, k, rng):
    """Memoryless quota: each group's expected share k*n_g/N, rounded by systematic
    sampling so that the shares sum to k and each group receives its share in expectation."""
    labels, inv = np.unique(gids, return_inverse=True)
    n = np.bincount(inv, minlength=len(labels)).astype(float)
    exact = k * n / n.sum()
    cum = np.concatenate([[0.0], np.cumsum(exact)])
    u = rng.random()
    pts = u + np.arange(k)
    kg = np.array([int(((pts >= cum[g]) & (pts < cum[g + 1])).sum()) for g in range(len(labels))])
    kg = np.minimum(kg, n.astype(int))
    m = np.zeros(len(sc), bool)
    for g in range(len(labels)):
        o = np.nonzero(inv == g)[0]
        o = o[np.argsort(-sc[o], kind="stable")]
        m[o[:kg[g]]] = True
    return m


def simulate_h(sd, budget=0.01, delta=0.002, group_col="g_functional_unit", history="alerts",
               allocator="fair", floor=None, delay=0, missing=0.0,
               p_miss=0.0, p_fc=0.0, err_group=None, p_miss_g=0.0, p_fc_g=0.0,
               schedule=False, lam=0.01, gamma=0.8, tau_ov=0.999, m_cap=3, horizon=30,
               record=False, seed=0, eta=0.0,
               hist_window=None, hist_decay=None, warm=None, warm_days=30,
               remove_confirmed=False, displace="lowest", eval_from=None, cooldown=0,
               p_inconclusive=0.0, p_miss_low=None, p_fc_high=None, score_split=0.999, fc_col=None,
               cd_delay=0, cd_backfill=True, cd_recheck=0.0, cd_recent_days=0, cd_recent_tau=None, cd_high_days=None, cd_high_split=0.99, state_log=None):
    """history: 'global', 'alerts' (label-free), 'cleared' (outcome-aware), 'none' (no memory).
    allocator: 'fair' (FAIR-IRS), 'deficit' (deficit apportionment alone), 'fairstar', 'dual'.
    hist_window  only the last `hist_window` days count in the group histories
    hist_decay   day weights hist_decay**age in the group histories
    warm         None; 'shadow' (global top-k for the first `warm_days` days, histories
                 accumulated, parity enforced afterwards); 'prior' (histories start with
                 `warm_days` pseudo-days per group at the budget rate)
    remove_confirmed  an insider leaves the population once an alert on them is confirmed
    displace     'lowest' (FAIR-IRS: a group whose alerts are cut keeps its highest scores) or
                 'random' (a cut group keeps k_g alerts drawn uniformly from its global top K_g)
    cooldown     an employee whose alert was cleared is not alerted again for `cooldown` days
                 unless their score reaches tau_ov (ordinary SOC practice)
    allocator 'randround': memoryless proportional quota with randomized (systematic) rounding
    p_inconclusive  share of benign alerts closed as inconclusive: they count as burden but start
                    no cooldown and are not stored by the scheduler
    p_miss_low      if set, malicious alerts scored below score_split are missed with this
                    probability (p_miss applies above it)
    p_fc_high       if set, benign alerts scored at or above score_split are wrongly confirmed
                    with this probability (p_fc applies below it)
    cd_delay        days before a clearance verdict is known (the cooldown starts then)
    cd_backfill     False: a suppressed alert is dropped and its slot is not given to anyone else
    cd_recheck      probability that a suppressed alert is investigated anyway (random re-check)
    cd_recent_days, cd_recent_tau
                    farming-signature monitor: within cd_recent_days of a clearance, a score of at
                    least cd_recent_tau is not suppressed
    cd_high_days    cooldown length after clearing an alert scored at least cd_high_split
                    (score-dependent cooldown); `cooldown` applies otherwise
    fc_col          optional column of per-row probabilities that a benign alert is confirmed
                    (NaN: the default applies); used for farming probes that may not be cleared
    """
    rng = np.random.default_rng(seed)
    T = sd.test
    days = T["day"].to_numpy()
    users = T["user"].to_numpy()
    y = T["y"].to_numpy()
    s = T["score"].to_numpy().astype(np.float64)
    fcv = T[fc_col].to_numpy(np.float64) if fc_col is not None and fc_col in T else None
    g = T[group_col].to_numpy()
    alert = np.zeros(len(T), bool)
    cleared = np.zeros(len(T), bool)
    removed = np.zeros(len(T), bool)
    gone = {}                                            # user -> day index from which removed
    cumA, cumC, cumConf = {}, {}, {}
    ledger = hist_window is not None or hist_decay is not None
    dA, dC, dF = [], [], []                              # per-day group counts (ledger)
    pending = []                                         # (reveal day, group, alert day, user)
    fb_days, budget_total, disp, rec = [], 0, [], []
    store, n_disc = {}, 0
    mu = {}
    last_clear = {}
    if schedule:
        phin = sd.phi / (np.linalg.norm(sd.phi, axis=1, keepdims=True) + 1e-9)
    uniq_days, starts = np.unique(days, return_index=True)
    bounds = list(starts) + [len(T)]
    if warm == "prior":
        first = np.arange(bounds[0], bounds[1])
        for lab, cnt in zip(*np.unique(g[first], return_counts=True)):
            cumC[lab] = cnt * warm_days
            cumA[lab] = budget * cnt * warm_days
    for di, d in enumerate(uniq_days):
        idx = np.arange(bounds[di], bounds[di + 1])
        if gone:
            keep_mask = np.array([gone.get(users[i], 10 ** 9) > di for i in idx])
            removed[idx[~keep_mask]] = True
            idx = idx[keep_mask]
            if len(idx) == 0:
                continue
        if ledger:
            dA.append({}); dC.append({}); dF.append({})
        keep = []
        for (rd, lab, ad, u) in pending:
            if rd <= di:
                if history == "cleared":
                    cumA[lab] = cumA.get(lab, 0) - 1
                    cumConf[lab] = cumConf.get(lab, 0) + 1
                    if ledger:
                        dA[ad][lab] = dA[ad].get(lab, 0) - 1
                        dF[ad][lab] = dF[ad].get(lab, 0) + 1
                if remove_confirmed and u is not None:
                    gone.setdefault(u, di)
            else:
                keep.append((rd, lab, ad, u))
        pending = keep
        sc = s[idx].copy()
        if schedule:
            for j, i in enumerate(idx):
                hist = store.get(users[i])
                if not hist:
                    continue
                hist[:] = [(dd, v) for dd, v in hist if (d - dd) / np.timedelta64(1, "D") <= horizon]
                if not hist or sc[j] >= tau_ov:
                    continue
                if max(float(phin[i] @ v) for _, v in hist) >= gamma:
                    sc[j] -= lam * min(len(hist), m_cap)
                    n_disc += 1
        supp = []
        if cooldown > 0 and last_clear:
            for j, i in enumerate(idx):
                lc = last_clear.get(users[i])
                if lc is None:
                    continue
                lday, lsc = lc
                if lday > di:
                    continue
                length = cooldown if (cd_high_days is None or lsc < cd_high_split) else cd_high_days
                if di - lday <= length and sc[j] < tau_ov:
                    if cd_recent_tau is not None and di - lday <= cd_recent_days and sc[j] >= cd_recent_tau:
                        continue
                    if cd_recheck > 0 and rng.random() < cd_recheck:
                        continue
                    if cd_backfill:
                        sc[j] = -1.0
                    else:
                        supp.append(i)
        k = max(1, int(round(budget * len(idx))))
        gi = g[idx]
        # histories seen by the allocator
        if ledger:
            A0, C0 = {}, {}
            now = len(dA) - 1
            for dd in range(now):
                age = now - dd
                if hist_window is not None and age > hist_window:
                    continue
                w = 1.0 if hist_decay is None else hist_decay ** age
                for lab, v in dA[dd].items():
                    A0[lab] = A0.get(lab, 0.0) + w * v
                for lab, v in dC[dd].items():
                    C0[lab] = C0.get(lab, 0.0) + w * v
                if history == "cleared":
                    for lab, v in dF[dd].items():
                        C0[lab] = C0.get(lab, 0.0) - w * v
        elif history == "cleared":
            A0 = cumA
            C0 = {lab: cumC.get(lab, 0) - cumConf.get(lab, 0) for lab in cumC}
        else:
            A0, C0 = cumA, cumC
        use_global = history == "global" or delta >= 1 or (warm == "shadow" and di < warm_days)
        if state_log is not None:
            state_log[d] = (dict(A0), dict(C0), k, use_global)
        fb = 0
        if use_global:
            m = np.zeros(len(idx), bool)
            m[np.argsort(-sc, kind="stable")[:k]] = True
        elif allocator == "dual":
            adj = sc - np.array([mu.get(lab, 0.0) for lab in gi])
            m = np.zeros(len(idx), bool)
            m[np.argsort(-adj, kind="stable")[:k]] = True
        elif allocator == "fairstar":
            m = g3._fairstar(sc, gi, k)
        elif allocator == "deficit":
            m = _deficit(sc, gi, k, A0, C0)
        elif allocator == "randround":
            m = _randround(sc, gi, k, rng)
        elif history == "none":
            m, fb = fi.fair_allocate_cum(sc, gi, k, delta, {}, {})
        else:
            m, fb = fi.fair_allocate_cum(sc, gi, k, delta, A0, C0)
        rank = np.empty(len(idx), int)
        rank[np.argsort(-sc, kind="stable")] = np.arange(1, len(idx) + 1)
        top = rank <= k
        if displace == "random" and not use_global:
            m2 = m.copy()
            for lab in np.unique(gi):
                sel = gi == lab
                Kg, kg = int((top & sel).sum()), int((m & sel).sum())
                if kg < Kg:
                    cand = np.flatnonzero(top & sel)
                    m2[sel] = False
                    if kg > 0:
                        m2[rng.choice(cand, size=kg, replace=False)] = True
            m = m2
        if fb:
            fb_days.append(di)
        if floor is not None:
            m = m & (rank <= int(np.ceil(floor * k)))
        disp.extend((rank[m & ~top] / k).tolist())
        budget_total += k
        if record:
            for lab in np.unique(gi):
                sel = gi == lab
                rec.append((d, lab, int(top[sel].sum()), int(m[sel].sum()), int(sel.sum())))
        a_idx = idx[m]
        if supp:
            a_idx = np.setdiff1d(a_idx, np.asarray(supp), assume_unique=False)
        alert[a_idx] = True
        for lab, cnt in zip(*np.unique(gi, return_counts=True)):
            cumC[lab] = cumC.get(lab, 0) + cnt
            if ledger:
                dC[-1][lab] = dC[-1].get(lab, 0) + cnt
        for i in a_idx:
            lab = g[i]
            pm = p_miss_g if (err_group is not None and lab == err_group) else p_miss
            pf = p_fc_g if (err_group is not None and lab == err_group) else p_fc
            if p_miss_low is not None and y[i] == 1 and s[i] < score_split:
                pm = p_miss_low
            if p_fc_high is not None and y[i] == 0 and s[i] >= score_split:
                pf = p_fc_high
            if fcv is not None and y[i] == 0 and not np.isnan(fcv[i]):
                pf = fcv[i]
            if y[i] == 1:
                final_clear = rng.random() < pm
            else:
                final_clear = not (rng.random() < pf)
            cleared[i] = final_clear
            inconclusive = p_inconclusive > 0 and y[i] == 0 and final_clear and rng.random() < p_inconclusive
            if final_clear and not inconclusive:
                last_clear[users[i]] = (di + cd_delay, s[i])
            cumA[lab] = cumA.get(lab, 0) + 1
            if ledger:
                dA[-1][lab] = dA[-1].get(lab, 0) + 1
            if schedule and final_clear and not inconclusive:
                store.setdefault(users[i], []).append((d, phin[i]))
            if final_clear or rng.random() < missing:
                continue
            if history == "cleared" and y[i] == 1:
                rng.random()                             # keeps the random stream of simulate_g (revise=0)
            u = users[i] if (remove_confirmed and y[i] == 1) else None
            if history == "cleared" or u is not None:
                pending.append((di + max(delay, 1 if history != "cleared" else 0), lab, len(dA) - 1 if ledger else 0, u))
        if allocator == "dual":
            den = ({lab: cumC[lab] - cumConf.get(lab, 0) for lab in cumC} if history == "cleared" else dict(cumC))
            rates = {lab: cumA.get(lab, 0) / max(den[lab], 1) for lab in den}
            rbar = sum(cumA.get(lab, 0) for lab in den) / max(sum(den.values()), 1)
            for lab, r in rates.items():
                mu[lab] = eta * (r - rbar)
    out = T[["user", "day", "y", "score", "exposure", "scenario", group_col]].copy()
    out["alert"], out["cleared"] = alert, cleared
    if remove_confirmed:
        out = out[~removed].copy()
    if eval_from is not None:
        out = out[out["day"] >= eval_from].copy()
    out.attrs.update(fallback_days=len(fb_days), fallback_list=fb_days, discounts=n_disc,
                     n_days=len(uniq_days), disp_ranks=np.array(disp), budget_total=budget_total,
                     removed_rows=int(removed.sum()))
    if record:
        return out, pd.DataFrame(rec, columns=["day", "group", "K", "k", "n"])
    return out


def metrics(out, group_col, **extra):
    m = fi.policy_metrics(out, group_col)
    m.update(g3.harm(out))
    m.update(extra)
    return m


def insider_hits(out):
    mal = out[out["y"] == 1]
    return mal.groupby("user")["alert"].any().astype(int)


DETS = ("IF", "AE", "LR", "GBM", "LSTM")


# =============================================================================
# I1  LSTM seeds
# =============================================================================

def lstm_seeds(ds, seeds, cache_dir):
    """Retrain the LSTM with each seed (cached on Drive). Returns {seed: ScoredData}."""
    import os
    import pickle
    res = {}
    for s in seeds:
        p = os.path.join(cache_dir, f"scored_{ds.name}_lstm_seed{s}_v2.pkl")
        if os.path.exists(p):
            with open(p, "rb") as f:
                res[s] = pickle.load(f)
            log(f"LSTM seed {s}: reloaded")
            continue
        sd = fi.run_detector(ds, fi.LSTMDetector(s))
        with open(p, "wb") as f:
            pickle.dump(sd, f)
        res[s] = sd
        log(f"LSTM seed {s}: trained and cached")
    return res


# =============================================================================
# I2  Non-inferiority
# =============================================================================

def _boot_mean(x, n_boot=10000, seed=0):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(x), size=(n_boot, len(x)))
    return x[idx].mean(axis=1)


def i2_noninferiority(runs, budgets=(0.005, 0.01, 0.02), delta_rel=0.2, margin=0.05,
                      group_col="g_functional_unit", pols=None, sched_budget=0.01):
    """runs: {detector: {run_name: ScoredData}}. For each detector, budget and policy:
    per run, insiders lost and gained against the global policy; seed-pooled, the
    per-insider detection averaged over runs, and a paired bootstrap over insiders of
    the recall difference. Non-inferiority: one-sided 95% lower bound > -margin."""
    pols = pols or [("B4 label-free", dict(history="alerts")),
                    ("B4c outcome-aware", dict(history="cleared")),
                    ("B5c outcome-aware + scheduling", dict(history="cleared", schedule=True))]
    per_run, pooled = [], []
    for det, rr in runs.items():
        for b in budgets:
            dl = delta_rel * b
            ref = {}
            for rn, sd in rr.items():
                ref[rn] = insider_hits(simulate_h(sd, b, 1.0, group_col, history="global"))
            for pol, kw in pols:
                if kw.get("schedule") and sched_budget is not None and abs(b - sched_budget) > 1e-12:
                    continue
                hits = {}
                for rn, sd in rr.items():
                    if kw.get("schedule") and sd.phi is None:
                        continue
                    h = insider_hits(simulate_h(sd, b, dl, group_col, **kw))
                    a, c = ref[rn].align(h, join="inner")
                    hits[rn] = (a, c)
                    per_run.append(dict(detector=det, budget=b, delta=dl, policy=pol, run=rn,
                                        insiders=len(a), detected_ref=int(a.sum()), detected=int(c.sum()),
                                        lost=int(((a == 1) & (c == 0)).sum()), gained=int(((a == 0) & (c == 1)).sum())))
                if not hits:
                    continue
                A = pd.concat([v[0] for v in hits.values()], axis=1).mean(axis=1)
                Cc = pd.concat([v[1] for v in hits.values()], axis=1).mean(axis=1)
                A, Cc = A.align(Cc, join="inner")
                diff = (Cc - A).to_numpy()
                bs = _boot_mean(diff)
                lo95_one = float(np.percentile(bs, 5))
                lo, hi = np.percentile(bs, [2.5, 97.5])
                pooled.append(dict(detector=det, budget=b, delta=dl, policy=pol, runs=len(hits), insiders=len(diff),
                                   recall_ref=float(A.mean()), recall=float(Cc.mean()), diff=float(diff.mean()),
                                   diff_insiders_per_run=float(diff.sum()), ci_lo=float(lo), ci_hi=float(hi),
                                   one_sided_lo=lo95_one, margin=margin, noninferior=bool(lo95_one > -margin)))
            log(f"I2 {det} budget {b} done")
    return pd.DataFrame(per_run), pd.DataFrame(pooled)


# =============================================================================
# I3  Rolling windows, windowed histories, deficit apportionment
# =============================================================================

def rolling_disparity(out, group_col, windows=(30, 90)):
    o = out.copy()
    o["ben"] = (o["y"] == 0).astype(int)
    o["fa"] = (o["alert"] & (o["y"] == 0)).astype(int)
    o["n"] = 1
    D = o.groupby(["day", group_col])[["n", "alert", "ben", "fa"]].sum().unstack(fill_value=0)
    res = {}
    for W in windows:
        R = D.rolling(W, min_periods=W).sum().dropna()
        if R.empty:
            res.update({f"roll{W}_{q}": float("nan") for q in ("dFAR_median", "dFAR_p90", "dFAR_max", "dAR_median", "dAR_max")})
            continue
        far = R["fa"] / R["ben"].where(R["ben"] > 0)
        ar = R["alert"] / R["n"].where(R["n"] > 0)
        dfar = far.max(axis=1) - far.min(axis=1)
        dar = ar.max(axis=1) - ar.min(axis=1)
        res.update({f"roll{W}_dFAR_median": float(dfar.median()), f"roll{W}_dFAR_p90": float(dfar.quantile(0.9)),
                    f"roll{W}_dFAR_max": float(dfar.max()), f"roll{W}_dAR_median": float(dar.median()),
                    f"roll{W}_dAR_max": float(dar.max())})
    # employee level, non-overlapping 30-day windows (the quantity of the private reports)
    d0 = o["day"].min()
    o["w"] = (o["day"] - d0).dt.days // 30
    ins = set(o.loc[o["y"] == 1, "user"])
    U = o[~o["user"].isin(ins)].groupby(["w", "user"]).agg(gr=(group_col, "first"), al=("alert", "any")).reset_index()
    roster = o[~o["user"].isin(ins)].groupby(group_col)["user"].nunique()
    vals = []
    for w, Dw in U.groupby("w"):
        cnt = Dw.groupby("gr")["al"].sum().reindex(roster.index).fillna(0)
        r = cnt / roster
        vals.append(r.max() - r.min())
    res["employee_month_dFAR_mean"] = float(np.mean(vals)) if vals else float("nan")
    res["employee_month_dFAR_max"] = float(np.max(vals)) if vals else float("nan")
    return res


I3_POLICIES = [
    ("B0 global", dict(history="global")),
    ("B8 daily parity, no memory", dict(history="none")),
    ("B9 FA*IR bands", dict(allocator="fairstar", history="alerts")),
    ("B10 dual prices (eta 10), label-free", dict(allocator="dual", history="alerts", eta=10.0)),
    ("B11 deficit apportionment, label-free", dict(allocator="deficit", history="alerts")),
    ("B11c deficit apportionment, outcome-aware", dict(allocator="deficit", history="cleared")),
    ("B4 label-free", dict(history="alerts")),
    ("B4 label-free, 30-day history", dict(history="alerts", hist_window=30)),
    ("B4 label-free, 90-day history", dict(history="alerts", hist_window=90)),
    ("B4c outcome-aware", dict(history="cleared")),
    ("B4c outcome-aware, 30-day history", dict(history="cleared", hist_window=30)),
    ("B4c outcome-aware, 90-day history", dict(history="cleared", hist_window=90)),
    ("B4c outcome-aware, discounted history (half-life 30 days)", dict(history="cleared", hist_decay=0.5 ** (1 / 30))),
]


def i3_windows(sc, budget=0.01, delta=0.002, group_col="g_functional_unit"):
    rows = []
    for det in [d for d in DETS if d in sc]:
        for pol, kw in I3_POLICIES:
            out = simulate_h(sc[det], budget, delta, group_col, **kw)
            m = metrics(out, group_col, detector=det, policy=pol)
            m.update(rolling_disparity(out, group_col))
            rows.append(m)
        log(f"I3 {det} done")
    return pd.DataFrame(rows)


# =============================================================================
# I4  Uncertainty and individual concentration
# =============================================================================

def _far_by_group(ben, group_col):
    G = ben.groupby(group_col)
    return G["fa"].sum() / G["n"].sum()


def i4_uncertainty(sc, budget=0.01, delta=0.002, group_col="g_functional_unit", n_boot=2000, n_random=20):
    pols = [("B0 global", dict(history="global")), ("B4 label-free", dict(history="alerts")),
            ("B4c outcome-aware", dict(history="cleared"))]
    rows = []
    for det in [d for d in DETS if d in sc]:
        for pol, kw in pols:
            out = simulate_h(sc[det], budget, delta, group_col, **kw)
            ben = out[out["y"] == 0]
            E = ben.groupby("user").agg(gr=(group_col, "first"), fa=("alert", "sum"), n=("alert", "size")).reset_index()
            far = _far_by_group(E, "gr")
            dfar, rho = far.max() - far.min(), (far.max() / far.min() if far.min() > 0 else np.inf)
            rng = np.random.default_rng(0)
            groups = {lab: D for lab, D in E.groupby("gr")}
            labs = list(groups)
            imax, imin = labs.index(far.idxmax()), labs.index(far.idxmin())
            bd, br, bp = [], [], []
            for _ in range(n_boot):
                fa, nn = [], []
                for lab, D in groups.items():
                    j = rng.integers(0, len(D), len(D))
                    fa.append(D["fa"].to_numpy()[j].sum()); nn.append(D["n"].to_numpy()[j].sum())
                f = np.array(fa) / np.array(nn)
                bd.append(f.max() - f.min()); br.append(f.max() / f.min() if f.min() > 0 else np.inf)
                bp.append(f[imax] - f[imin])
            bd, br, bp = np.array(bd), np.array(br), np.array(bp)
            # most-flagged benign employees removed (never-malicious employees only)
            ins = set(out.loc[out["y"] == 1, "user"])
            Eb = E[~E["user"].isin(ins)]
            res = dict(detector=det, policy=pol, delta_FAR=dfar, ratio_FAR=rho,
                       dFAR_lo=float(np.percentile(bd, 2.5)), dFAR_hi=float(np.percentile(bd, 97.5)),
                       pair_lo=float(np.percentile(bp, 2.5)), pair_hi=float(np.percentile(bp, 97.5)),
                       rho_lo=float(np.percentile(br, 2.5, method="lower")),
                       rho_hi=float(np.percentile(br, 97.5, method="higher")),
                       rho_inf_share=float(np.isinf(br).mean()))
            for top in (1, 10):
                drop = set(Eb.sort_values("fa", ascending=False)["user"].iloc[:top])
                f2 = _far_by_group(E[~E["user"].isin(drop)], "gr")
                res[f"dFAR_without_top{top}"] = float(f2.max() - f2.min())
                res[f"rho_without_top{top}"] = float(f2.max() / f2.min()) if f2.min() > 0 else np.inf
            # share of the max-min gap due to the single most-flagged employee of the max group
            gmax = far.idxmax()
            Dg = E[E["gr"] == gmax]
            res["max_group_top_employee_share_of_rate"] = float(Dg["fa"].max() / max(Dg["fa"].sum(), 1))
            # Gini of false alerts per benign employee, actual and under random allocation
            F = out[~out["user"].isin(ins)].groupby("user")["alert"].sum()
            res["gini_false_alerts"] = fi.gini(F.values)
            gr = []
            T = out[["day", "user"]].reset_index(drop=True)
            dayidx = T.groupby("day").indices
            for r in range(n_random):
                rg = np.random.default_rng(100 + r)
                a = np.zeros(len(T), bool)
                for dd, ii in dayidx.items():
                    kk = max(1, int(round(budget * len(ii))))
                    a[rg.choice(ii, kk, replace=False)] = True
                Fr = pd.Series(a, index=T["user"]).groupby(level=0).sum()
                Fr = Fr[~Fr.index.isin(ins)]
                gr.append(fi.gini(Fr.values))
            res["gini_random_allocation"] = float(np.mean(gr))
            rows.append(res)
        log(f"I4 {det} done")
    return pd.DataFrame(rows)


# =============================================================================
# I5  Insiders removed after their first confirmed alert
# =============================================================================

def i5_removal(sc, budget=0.01, delta=0.002, group_col="g_functional_unit"):
    pols = [("B0 global", dict(history="global")), ("B4 label-free", dict(history="alerts")),
            ("B4c outcome-aware", dict(history="cleared"))]
    rows = []
    for det in [d for d in DETS if d in sc]:
        for pol, kw in pols:
            for rm in (False, True):
                out = simulate_h(sc[det], budget, delta, group_col, remove_confirmed=rm, **kw)
                rows.append(metrics(out, group_col, detector=det, policy=pol, insiders_removed=rm,
                                    removed_rows=out.attrs["removed_rows"]))
        log(f"I5 {det} done")
    return pd.DataFrame(rows)


# =============================================================================
# I6  Joint verdict errors, fallible re-checker, joint confidence
# =============================================================================

def cp_upper(x, m, alpha):
    """One-sided Clopper-Pearson upper bound at level 1 - alpha."""
    if m == 0:
        return 1.0
    if x >= m:
        return 1.0
    return float(beta_dist.ppf(1 - alpha, x + 1, m - x))


def group_stats(out, group_col):
    """Per group: active user-days, cleared alerts, labels of confirmed alerts."""
    st = {}
    g = out[group_col].to_numpy()
    a = out["alert"].to_numpy(); c = out["cleared"].to_numpy(); yy = out["y"].to_numpy()
    for lab in np.unique(g):
        sel = g == lab
        st[lab] = (int(sel.sum()), int((sel & a & c).sum()), yy[sel & a & ~c])
    ben = out[out["y"] == 0]
    far = ben.groupby(group_col)["alert"].mean()
    return st, float(far.max() - far.min())


def corollary_bound(stats, frac=0.1, recheck_err=0.0, alpha=0.05, bonferroni=True, pi_bar=0.05, rng=None):
    """Bound of Corollary 1 from one re-check sample of each group's confirmed alerts.
    The re-checker misses a false confirmation with probability recheck_err.
    Returns (bound, plain outcome-based audit)."""
    rng = rng or np.random.default_rng(0)
    a_g = alpha / len(stats) if bonferroni else alpha
    lo, up = [], []
    for lab, (n, cl, conf_y) in stats.items():
        C = len(conf_y)
        msz = int(math.ceil(frac * C))
        if msz > 0:
            samp = conf_y[rng.choice(C, msz, replace=False)]
            found = (samp == 0) & (rng.random(msz) >= recheck_err)
            q = cp_upper(int(found.sum()), msz, a_g)
        else:
            q = 0.0
        fbar = q * C / (cl + q * C) if (cl + q * C) > 0 else 0.0
        base = cl / n
        lo.append(base)
        up.append(base / ((1 - pi_bar) * (1 - fbar)) if fbar < 1 else np.inf)
    return max(up) - min(lo), max(lo) - min(lo)


def i6_joint_errors(sc, budget=0.01, delta=0.002, group_col="g_functional_unit",
                    detectors=("GBM", "LSTM", "IF"), seeds=(0, 1, 2), n_samples=20):
    base = next(iter(sc.values())).test
    users_per = base.groupby(group_col)["user"].nunique().sort_values()
    small = users_per.index[0]
    settings = []
    for pm in (0.0, 0.3):
        settings.append((pm, "--", 0.0, {}))
        for where in ("all groups", "smallest group"):
            for f in (0.1, 0.2, 0.3):
                kw = dict(p_fc=f) if where == "all groups" else dict(err_group=small, p_fc_g=f)
                settings.append((pm, where, f, kw))
    rows = []
    for det in [d for d in detectors if d in sc]:
        for pm, where, f, kw in settings:
            for s in seeds:
                out = simulate_h(sc[det], budget, delta, group_col, history="cleared", p_miss=pm, seed=s, **kw)
                stats, tru = group_stats(out, group_col)
                for frac in (0.1, 0.2):
                    for rerr in (0.0, 0.1, 0.2):
                        for bonf in (False, True):
                            rng = np.random.default_rng(1000 * s + 7)
                            res = [corollary_bound(stats, frac, rerr, bonferroni=bonf, rng=rng)
                                   for _ in range(n_samples)]
                            b = np.array([r[0] for r in res])
                            rows.append(dict(detector=det, missed=pm, where=where, fc_rate=f, seed=s, frac=frac,
                                             recheck_error=rerr, bonferroni=bonf, true_dFAR=tru,
                                             audit=res[0][1], bound_mean=float(b.mean()),
                                             coverage=float((tru <= b + 1e-12).mean())))
        log(f"I6 {det} done")
    return pd.DataFrame(rows)


# =============================================================================
# I7  Timing attackers: many draws, paired with the global policy, randomized displacement
# =============================================================================

def _monthly_noisy(rec, eps, rng):
    """Published monthly noisy cumulative alert counts per group (roster-free version:
    counts of alerts and active user-days, Laplace noise on the alert counts)."""
    r = rec.copy()
    r["month"] = pd.to_datetime(r["day"]).dt.strftime("%Y-%m")
    M = r.groupby(["month", "group"])[["k", "n"]].sum().reset_index()
    if eps is not None:
        M["k"] = M["k"] + rng.laplace(0, 1 / eps, len(M))
    return M


def i7_attacks(sc, budget=0.01, delta=0.002, group_col="g_functional_unit", detectors=("GBM", "LSTM"),
               det_draws=5, rand_draws=10, window=30, eps=1.0, n_boot=5000, policies=None, attackers_only=None,
               shifter=None, collect=None):
    """For each FAIR-IRS policy P, attackers choose when to act from P's unattacked run
    (allocation, colleagues' investigations, monthly reports, deficit estimate).
    The relocated episodes are scored under P AND under the global policy, so the
    allocation's effect is isolated from the effect of relocation itself."""
    policies = policies or [("B4 label-free", dict(history="alerts")),
                            ("B4c outcome-aware", dict(history="cleared")),
                            ("B4 label-free, randomized displacement", dict(history="alerts", displace="random")),
                            ("B4c outcome-aware, randomized displacement", dict(history="cleared", displace="random"))]
    rows, per_ins = [], []
    for det in [d for d in detectors if d in sc]:
        sd0 = sc[det]
        out_g0 = simulate_h(sd0, budget, 1.0, group_col, history="global")
        hit_g0 = insider_hits(out_g0)
        for pol, kw in policies:
            out0, rec = simulate_h(sd0, budget, delta, group_col, record=True, seed=0, **kw)
            hit0 = insider_hits(out0)
            r = rec.copy()
            r["cut"] = (r["K"] - r["k"]).clip(lower=0)
            r["cfrac"] = np.where(r["K"] > 0, r["cut"] / r["K"].clip(lower=1), 0.0)
            cut = r.set_index(["day", "group"])["cut"].to_dict()
            obs = {}
            for lab, D in r.sort_values("day").groupby("group"):
                a = D["k"].rolling(window, min_periods=1).sum().shift(1).fillna(0)
                n = D["n"].rolling(window, min_periods=1).sum().shift(1).fillna(1)
                for dd, v in zip(D["day"], (a / n).to_numpy()):
                    obs[(dd, lab)] = v
            rep = fr.report_targets(out0, group_col)

            def make_deficit(seed):
                """Estimated cumulative excess of the group's alert rate over the overall rate:
                noisy published monthly counts up to the last report, plus the colleagues'
                investigations observed since then (own group only)."""
                M = _monthly_noisy(rec, eps, np.random.default_rng(500 + seed))
                months = sorted(M["month"].unique())
                cumk = M.pivot(index="month", columns="group", values="k").fillna(0).cumsum()
                cumn = M.pivot(index="month", columns="group", values="n").fillna(0).cumsum()
                overall = cumk.sum(axis=1) / cumn.sum(axis=1)
                R = r.copy()
                R["month"] = pd.to_datetime(R["day"]).dt.strftime("%Y-%m")
                est = {}
                for lab, D in R.sort_values("day").groupby("group"):
                    D = D.copy()
                    D["prev"] = D["month"].map(lambda mth: months[months.index(mth) - 1] if months.index(mth) > 0 else None)
                    D["k_since"] = D.groupby("month")["k"].cumsum().shift(1).fillna(0)
                    D.loc[D["month"] != D["month"].shift(1), "k_since"] = 0
                    D["n_since"] = D.groupby("month")["n"].cumsum().shift(1).fillna(0)
                    D.loc[D["month"] != D["month"].shift(1), "n_since"] = 0
                    for dd, pm, ks, ns in zip(D["day"], D["prev"], D["k_since"], D["n_since"]):
                        if pm is None or pd.isna(pm) or lab not in cumk.columns:
                            est[(dd, lab)] = 0.0
                            continue
                        ck, cn = cumk.loc[pm, lab] + ks, cumn.loc[pm, lab] + ns
                        est[(dd, lab)] = ck / max(cn, 1) - overall.loc[pm]
                return est

            def oracle(u, rows_, T, L, rng):
                gu = T.loc[rows_, group_col].to_numpy(); dd = T.loc[rows_, "day"].to_numpy()
                c = np.array([cut.get((dd[j], gu[j]), 0) for j in range(len(rows_))], float)
                win = np.convolve(c, np.ones(L), "valid")
                return int(rng.choice(np.flatnonzero(win == win.max())))

            def observing(u, rows_, T, L, rng):
                gu = T.loc[rows_, group_col].to_numpy(); dd = T.loc[rows_, "day"].to_numpy()
                c = np.array([obs.get((dd[j], gu[j]), 0) for j in range(len(rows_) - L + 1)], float)
                return int(rng.choice(np.flatnonzero(c == c.max())))

            def reads_reports(u, rows_, T, L, rng):
                gu = T.loc[rows_[0], group_col]
                months = T.loc[rows_, "day"].dt.strftime("%Y-%m").to_numpy()
                if gu not in rep.columns:
                    return int(rng.integers(0, len(rows_) - L + 1))
                ex = rep[gu].dropna()
                allm = sorted(rep.index)
                for mth in list(ex.sort_values(ascending=False).index):
                    nxt = allm.index(mth) + 1
                    if nxt >= len(allm):
                        continue
                    pos = np.flatnonzero(months == allm[nxt])
                    pos = pos[pos <= len(rows_) - L]
                    if len(pos):
                        return int(rng.choice(pos[:5]))
                return int(rng.integers(0, len(rows_) - L + 1))

            def rand(u, rows_, T, L, rng):
                return int(rng.integers(0, len(rows_) - L + 1))

            attackers = [("knows allocation", oracle, det_draws), ("observes colleagues", observing, det_draws),
                         ("reads reports", reads_reports, det_draws), ("estimates deficit", None, det_draws),
                         ("random", rand, rand_draws)]
            if attackers_only:
                attackers = [a for a in attackers if a[0] in attackers_only]
            for name, fn, nd in attackers:
                for dr in range(nd):
                    if name == "estimates deficit":
                        est = make_deficit(dr)

                        def fn(u, rows_, T, L, rng, est=est):
                            gu = T.loc[rows_, group_col].to_numpy(); dd = T.loc[rows_, "day"].to_numpy()
                            c = np.array([est.get((dd[j], gu[j]), 0) for j in range(len(rows_) - L + 1)], float)
                            return int(rng.choice(np.flatnonzero(c == c.max())))
                    sd = fr.shift_to(sd0, group_col, fn, dr) if shifter is None else shifter(det, sd0, group_col, fn, dr)
                    outP, recP = simulate_h(sd, budget, delta, group_col, record=True, seed=dr + 1, **kw)
                    outG = simulate_h(sd, budget, 1.0, group_col, history="global")
                    hP, hG = insider_hits(outP), insider_hits(outG)
                    # per insider: allocation effect under attack minus allocation effect without attack
                    D = pd.concat([hit0.rename("P0"), hit_g0.rename("G0"), hP.rename("PA"), hG.rename("GA")], axis=1).fillna(0)
                    dd_ = ((D["PA"] - D["GA"]) - (D["P0"] - D["G0"])).to_numpy()
                    if collect is not None:
                        collect.extend(dict(detector=det, policy=pol, attack=name, draw=dr, user=u, d=float(v))
                                       for u, v in zip(D.index, dd_))
                    bs = _boot_mean(dd_, n_boot, seed=dr)
                    row = dict(detector=det, policy=pol, attack=name, draw=dr, insiders=len(D),
                               none_P=int(D["P0"].sum()), none_G=int(D["G0"].sum()),
                               attacked_P=int(D["PA"].sum()), attacked_G=int(D["GA"].sum()),
                               excess_loss=float(-dd_.sum()), excess_lo=float(-np.percentile(bs, 97.5) * len(D)),
                               excess_hi=float(-np.percentile(bs, 2.5) * len(D)),
                               delta_FAR=fi.policy_metrics(outP, group_col)["delta_FAR"])
                    if kw.get("displace") == "random":
                        # bound of the randomized-displacement proposition, per insider:
                        # P(miss | caught by global) <= prod over the insider's global-top-k days of c_{g,t}
                        Rp = recP.copy()
                        Rp["cfrac"] = np.where(Rp["K"] > 0, (Rp["K"] - Rp["k"]).clip(lower=0) / Rp["K"].clip(lower=1), 0.0)
                        cf = Rp.set_index(["day", "group"])["cfrac"].to_dict()
                        topG = outG[(outG["y"] == 1) & outG["alert"]]
                        bnd = topG.groupby("user").apply(lambda Q: float(np.prod([cf.get((a, b), 0.0) for a, b in zip(Q["day"], Q[group_col])])))
                        caughtG = bnd.index
                        row["bound_expected_misses"] = float(bnd.sum())
                        row["observed_misses_among_caught_by_global"] = int((hP.reindex(caughtG).fillna(0) == 0).sum())
                        row["max_cut_fraction"] = float(Rp["cfrac"].max())
                        row["mean_cut_fraction_on_cut_days"] = float(Rp.loc[Rp["cfrac"] > 0, "cfrac"].mean()) if (Rp["cfrac"] > 0).any() else 0.0
                    rows.append(row)
            log(f"I7 {det} {pol} done")
    return pd.DataFrame(rows)


def i7_summary(df):
    g = df.groupby(["detector", "policy", "attack"], sort=False)
    s = g[["none_P", "none_G", "attacked_P", "attacked_G", "excess_loss", "excess_lo", "excess_hi"]].mean()
    s["draws"] = g.size()
    s["excess_loss_min"] = g["excess_loss"].min()
    s["excess_loss_max"] = g["excess_loss"].max()
    for c in ("bound_expected_misses", "observed_misses_among_caught_by_global", "max_cut_fraction"):
        if c in df.columns:
            s[c] = g[c].mean()
    return s.reset_index()


# =============================================================================
# I8  Tolerance chosen on a validation period
# =============================================================================

def i8_validation(sc, budget=0.01, group_col="g_functional_unit", deltas=(0.001, 0.002, 0.005, 0.01),
                  margin=0.05):
    """The first half of the test period is the validation period. For each detector
    and mode, delta* = smallest tolerance whose validation-period recall difference
    with the global policy has a one-sided 95% lower bound above -margin. Reported on
    the second half (alerts and histories run over the whole period)."""
    rows = []
    for det in [d for d in DETS if d in sc]:
        sd = sc[det]
        dd = np.sort(sd.test["day"].unique())
        mid = pd.Timestamp(dd[len(dd) // 2])
        outG = simulate_h(sd, budget, 1.0, group_col, history="global")
        for mode in ("alerts", "cleared"):
            res = {}
            for dl in deltas:
                out = simulate_h(sd, budget, dl, group_col, history=mode)
                vG, vP = insider_hits(outG[outG["day"] < mid]), insider_hits(out[out["day"] < mid])
                a, c = vG.align(vP, join="inner")
                lo = float(np.percentile(_boot_mean((c - a).to_numpy().astype(float)), 5)) if len(a) else 0.0
                tG, tP = outG[outG["day"] >= mid], out[out["day"] >= mid]
                mT = fi.policy_metrics(tP, group_col)
                mG = fi.policy_metrics(tG, group_col)
                res[dl] = dict(val_insiders=len(a), val_lost=int(((a == 1) & (c == 0)).sum()),
                               val_gained=int(((a == 0) & (c == 1)).sum()), val_lo=lo,
                               test_detected=mT["insiders_detected"], test_detected_global=mG["insiders_detected"],
                               test_insiders=mT["insiders_test"], test_dFAR=mT["delta_FAR"], test_dFAR_global=mG["delta_FAR"])
            ok = [dl for dl in deltas if res[dl]["val_lo"] > -margin]
            chosen = min(ok) if ok else max(deltas)
            for dl in deltas:
                rows.append(dict(detector=det, mode=mode, delta=dl, chosen=dl == chosen, **res[dl]))
        log(f"I8 {det} done")
    return pd.DataFrame(rows)


# =============================================================================
# I9  Score ties and group counts
# =============================================================================

def i9_ties(sc, budget=0.01):
    rows = []
    for det in [d for d in DETS if d in sc]:
        T = sc[det].test
        tied_top, n_top, sat = 0, 0, 0
        sc_top = []
        for dd, D in T.groupby("day"):
            k = max(1, int(round(budget * len(D))))
            s = np.sort(D["score"].to_numpy())[::-1]
            thr = s[k - 1]
            tied_top += int((s[:k] == thr).sum() > 1 or (k < len(s) and s[k] == thr))
            n_top += 1
            sat += int((s[:k] >= 0.999).sum())
            sc_top.extend(s[:k].tolist())
        sc_top = np.array(sc_top)
        rows.append(dict(detector=det, days=n_top, days_with_tie_at_threshold=tied_top,
                         share_alerts_score_ge_0999=sat / max(len(sc_top), 1),
                         alerted_score_min=float(sc_top.min()), alerted_score_median=float(np.median(sc_top)),
                         distinct_scores_share=float(T["score"].nunique() / len(T))))
    return pd.DataFrame(rows)


def i9_groups(ds):
    rows = []
    te = ds.df[~ds.df["train"]]
    for gc in ds.group_cols:
        u = te.groupby(gc)["user"].nunique()
        rows.append(dict(grouping=gc, groups=len(u), smallest=int(u.min()), largest=int(u.max()),
                         median=float(u.median()), sizes=";".join(f"{k}:{v}" for k, v in u.sort_values().items())))
    return pd.DataFrame(rows)


# =============================================================================
# I10 Shadow month and prior warm start
# =============================================================================

def i10_warm(sc, budget=0.01, delta=0.002, group_col="g_functional_unit", burn_in=30):
    rows = []
    for det in [d for d in DETS if d in sc]:
        for mode in ("alerts", "cleared"):
            for warm in (None, "shadow", "prior"):
                out = simulate_h(sc[det], budget, delta, group_col, history=mode, warm=warm)
                t = g3.trajectory(out, group_col)
                target = "dCR" if mode == "cleared" else "dAR"
                after = t.iloc[burn_in:]
                m = metrics(out, group_col, detector=det, mode=mode, warm=str(warm))
                m.update(fallback_list=";".join(map(str, out.attrs["fallback_list"])),
                         max_constrained_after_burn_in=float(after[target].max()),
                         share_days_above_delta_after_burn_in=float((after[target] > delta + 1e-12).mean()),
                         max_constrained_first_30=float(t[target].iloc[:burn_in].max()))
                rows.append(m)
        log(f"I10 {det} done")
    return pd.DataFrame(rows)


# =============================================================================
# I11 Power of a test on private monthly reports
# =============================================================================

def i11_power(ds, sc_det, budget=0.01, delta=0.002, group_col="g_functional_unit",
              eps_list=(0.5, 1.0, 2.0, 4.0, None), gaps=(0.01, 0.02, 0.05, 0.1, 0.2), n_sim=4000,
              groupings=("g_functional_unit", "g_department", "g_role")):
    """Monthly roster-based reports (share of each group's roster alerted at least once).
    For the smallest and the largest group of each grouping, the oversight body tests
    H0: equal rates against a gap, with the Laplace noise and the binomial sampling of
    employees. Base rate: the median monthly employee-level alert rate under global top-k.
    Returns the power at alpha = 0.05 (two-sided) and the minimal detectable gap at 80% power."""
    rows = []
    te = ds.df[~ds.df["train"]]
    for gc in groupings:
        out = simulate_h(sc_det, budget, 1.0, gc, history="global")
        o = out.copy()
        o["w"] = (o["day"] - o["day"].min()).dt.days // 30
        U = o.groupby(["w", "user"])["alert"].any().reset_index()
        p0 = float(U.groupby("w")["alert"].mean().median())
        u = te.groupby(gc)["user"].nunique()
        n1, n2 = int(u.min()), int(u.max())
        rng = np.random.default_rng(0)
        for eps in eps_list:
            def stat(gap, size):
                x1 = rng.binomial(n1, min(1, p0 + gap), size)
                x2 = rng.binomial(n2, p0, size)
                if eps is not None:
                    x1 = x1 + rng.laplace(0, 1 / eps, size)
                    x2 = x2 + rng.laplace(0, 1 / eps, size)
                return x1 / n1 - x2 / n2
            null = np.abs(stat(0.0, n_sim))
            crit = np.quantile(null, 0.95)
            pw = {f"power_gap_{gp}": float((np.abs(stat(gp, n_sim)) > crit).mean()) for gp in gaps}
            grid = np.linspace(0.0, 0.6, 121)
            mdg = next((float(gp) for gp in grid if (np.abs(stat(gp, 2000)) > crit).mean() >= 0.8), float("nan"))
            rows.append(dict(grouping=gc, smallest=n1, largest=n2, base_rate=p0,
                             eps=("none" if eps is None else eps), min_detectable_gap_80=mdg, **pw))
    return pd.DataFrame(rows)

In [ ]:
%%writefile fairirs_rev6.py
"""Experiments for the sixth and seventh reviews of the FAIR-IRS paper.

J1  Permutation test of group disparity (group labels shuffled across employees)
J2  Scheduling and cooldown on top of the global policy; memoryless quota with randomized
    rounding; deficit apportionment with histories (the new default)
J3  Detectors on per-employee baselines
J4  Timing attacks with delayed and missing verdicts; deficit apportionment under attack
J5  Discount farming against the outcome-aware mode and deficit apportionment
J6  Non-inferiority of the new default and of the new baselines
J7  Score calibration: share of malicious user-days by score band
J8  CERT r5.2: burden, parity rules, non-inferiority, attacks, permutation test
"""
import time
import numpy as np
import pandas as pd
import fairirs as fi
import fairirs_rev3 as g3
import fairirs_rev5 as h5


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


DETS = ("IF", "AE", "LR", "GBM", "LSTM")
HALF30 = 0.5 ** (1 / 30)

DEFAULT = ("B11c deficit apportionment (outcome-aware)", dict(allocator="deficit", history="cleared"))
DEFAULT_DISC = ("B11c deficit apportionment, discounted", dict(allocator="deficit", history="cleared", hist_decay=HALF30))


# =============================================================================
# J1  Permutation test
# =============================================================================

def _user_table(out, group_col):
    ben = out[out["y"] == 0]
    E = ben.groupby("user").agg(gr=(group_col, lambda s: s.mode().iloc[0]), fa=("alert", "sum"),
                                n=("alert", "size")).reset_index()
    return E


def _dfar(fa, n, codes, G):
    FA = np.bincount(codes, weights=fa, minlength=G)
    N = np.bincount(codes, weights=n, minlength=G)
    r = FA / np.maximum(N, 1)
    return r.max() - r.min()


def j1_permutation(sc, budget=0.01, delta=0.002, group_col="g_functional_unit", n_perm=5000, release="r4.2"):
    """Null: group labels exchangeable across employees (group sizes kept). Statistic: the
    max-min difference of group false-alert rates, which is biased upward under the null too;
    the permutation distribution accounts for that bias."""
    pols = [("B0 global", dict(history="global")), ("B4c exact rule", dict(history="cleared")), DEFAULT]
    rows = []
    for det in [d for d in DETS if d in sc]:
        for pol, kw in pols:
            out = h5.simulate_h(sc[det], budget, delta, group_col, **kw)
            E = _user_table(out, group_col)
            ins = set(out.loc[out["y"] == 1, "user"])
            for drop in (0, 10):
                Ed = E
                if drop:
                    top = set(E[~E["user"].isin(ins)].sort_values("fa", ascending=False)["user"].iloc[:drop])
                    Ed = E[~E["user"].isin(top)]
                codes, labs = pd.factorize(Ed["gr"])
                G = len(labs)
                fa, n = Ed["fa"].to_numpy(float), Ed["n"].to_numpy(float)
                obs = _dfar(fa, n, codes, G)
                rng = np.random.default_rng(0)
                perm = np.array([_dfar(fa, n, rng.permutation(codes), G) for _ in range(n_perm)])
                rows.append(dict(release=release, detector=det, policy=pol, dropped_top=drop, delta_FAR=obs,
                                 null_median=float(np.median(perm)), null_p95=float(np.quantile(perm, 0.95)),
                                 p_value=float((1 + (perm >= obs - 1e-15).sum()) / (1 + n_perm))))
        log(f"J1 {release} {det} done")
    return pd.DataFrame(rows)


# =============================================================================
# J2  Baselines: scheduling and cooldown on global, randomized rounding, default
# =============================================================================

J2_POLICIES = [
    ("B0 global", dict(history="global")),
    ("B0 + scheduling", dict(history="global", schedule=True)),
    ("B0 + cooldown 7 days", dict(history="global", cooldown=7)),
    ("B0 + cooldown 30 days", dict(history="global", cooldown=30)),
    ("B1r quota, randomized rounding", dict(allocator="randround", history="alerts")),
    ("B4c exact rule", dict(history="cleared")),
    ("B4c exact rule, discounted", dict(history="cleared", hist_decay=HALF30)),
    ("B4c + scheduling", dict(history="cleared", schedule=True)),
    ("B4c + cooldown 30 days", dict(history="cleared", cooldown=30)),
    DEFAULT,
    DEFAULT_DISC,
    ("B11c + scheduling", dict(allocator="deficit", history="cleared", schedule=True)),
    ("B11c + cooldown 30 days", dict(allocator="deficit", history="cleared", cooldown=30)),
]


def j2_baselines(sc, budget=0.01, delta=0.002, group_col="g_functional_unit", release="r4.2", seeds=(0, 1, 2)):
    rows = []
    for det in [d for d in DETS if d in sc]:
        for pol, kw in J2_POLICIES:
            if kw.get("schedule") and sc[det].phi is None:
                continue
            ss = seeds if kw.get("allocator") == "randround" else (0,)
            for s in ss:
                out = h5.simulate_h(sc[det], budget, delta, group_col, seed=s, **kw)
                m = h5.metrics(out, group_col, release=release, detector=det, policy=pol, seed=s)
                m.update(h5.rolling_disparity(out, group_col))
                rows.append(m)
        log(f"J2 {release} {det} done")
    return pd.DataFrame(rows)


# =============================================================================
# J3  Detectors on per-employee baselines
# =============================================================================

def run_detector_user(ds, det):
    """Same pipeline as fairirs.run_detector, but every user-day is expressed as its
    deviation from the employee's own mean over the training period."""
    F = ds.df
    tr_mask = F["train"].to_numpy()
    y = F["y"].to_numpy()
    tr_fit = tr_mask & (y == 0) if not det.supervised else tr_mask
    ft = fi.FeatureTransform("raw").fit(F[tr_fit])
    X0 = ft.transform(F)
    base = fi.user_baseline(F, X0, tr_mask & (y == 0))
    X = fi.explanations(F, X0, base).astype(np.float32)
    det.fit(X[tr_fit], y[tr_fit])
    raw = det.score(X[~tr_mask])
    raw_tr = det.score(X[tr_mask & (y == 0)])
    ref = np.sort(raw_tr)
    test = F[~tr_mask].copy().reset_index(drop=True)
    test["raw_score"] = raw
    test["score"] = fi.to_quantile_ref(raw, ref)
    return fi.ScoredData(det.name + "-user", test, X[~tr_mask].copy(), det, ft, ref, None)


def j3_user_baselines(ds, budget=0.01, delta=0.002, group_col="g_functional_unit", seed=0):
    rows, scs = [], {}
    for det in (fi.IFDetector(seed), fi.AEDetector(seed), fi.GBMDetector(seed)):
        try:
            sd = run_detector_user(ds, det)
        except Exception as ex:
            log(f"[skip] {det.name}-user: {type(ex).__name__}: {ex}")
            continue
        scs[sd.detector] = sd
        for pol, kw in [("B0 global", dict(history="global")), ("B0 + scheduling", dict(history="global", schedule=True)),
                        ("B4c exact rule", dict(history="cleared")), DEFAULT]:
            out = h5.simulate_h(sd, budget, delta, group_col, **kw)
            m = h5.metrics(out, group_col, detector=sd.detector, policy=pol)
            m["exposure_bias"] = fi.exposure_bias(sd.test, group_col=group_col)
            rows.append(m)
        log(f"J3 {sd.detector} done")
    return pd.DataFrame(rows), scs


# =============================================================================
# J4  Timing attacks with delayed verdicts; deficit apportionment under attack
# =============================================================================

J4_POLICIES = [
    ("B4c, verdicts after 7 days", dict(history="cleared", delay=7)),
    ("B4c, verdicts after 30 days", dict(history="cleared", delay=30)),
    ("B4c, 20% verdicts missing + 7-day delay", dict(history="cleared", delay=7, missing=0.2)),
    DEFAULT,
    DEFAULT_DISC,
    ("B11c, verdicts after 7 days", dict(allocator="deficit", history="cleared", delay=7)),
]


def j4_attacks(sc, budget=0.01, delta=0.002, group_col="g_functional_unit", detectors=("GBM", "LSTM"),
               det_draws=5, rand_draws=5, release="r4.2"):
    df = h5.i7_attacks(sc, budget, delta, group_col, detectors=detectors, det_draws=det_draws,
                       rand_draws=rand_draws, policies=J4_POLICIES)
    df["release"] = release
    return df


# =============================================================================
# J5  Discount farming against outcome-aware accounting
# =============================================================================

def j5_farming(sc, budget=0.01, delta=0.002, group_col="g_functional_unit", detectors=("GBM", "IF"),
               alphas=(0.5, 1.0)):
    pols = [("B0 global", dict(history="global")), ("B4 label-free", dict(history="alerts")),
            ("B4c exact rule", dict(history="cleared")), DEFAULT]
    rows = []
    for det in [d for d in detectors if d in sc]:
        sd = sc[det]
        for alpha in alphas:
            sdf, farmers = fi.discount_farming(sd, n_farm=3, alpha=alpha)
            if not farmers:
                continue
            for pol, kw in pols:
                for sched in (False, True):
                    res = {}
                    for lab, s_ in (("no farming", sd), ("farming", sdf)):
                        out = h5.simulate_h(s_, budget, delta, group_col, schedule=sched, **kw)
                        sub = out[out["user"].isin(farmers) & (out["y"] == 1)]
                        res[lab] = sub.groupby("user")["alert"].any()
                        if lab == "farming":
                            fa = out[out["user"].isin(farmers) & (out["y"] == 0)]
                            farm_alerts = int(fa["alert"].sum())
                    a, b = res["no farming"].align(res["farming"], join="inner")
                    rows.append(dict(detector=det, alpha=alpha, policy=pol, schedule=sched, farmers=len(farmers),
                                     detected_no_farming=int(a.sum()), detected_farming=int(b.sum()),
                                     lost=int((a & ~b).sum()), gained=int((~a & b).sum()),
                                     benign_alerts_on_farmers=farm_alerts))
        log(f"J5 {det} done")
    return pd.DataFrame(rows)


# =============================================================================
# J7  Score calibration
# =============================================================================

def j7_calibration(sc, budget=0.01, release="r4.2"):
    bands = [(0.0, 0.95), (0.95, 0.99), (0.99, 0.995), (0.995, 0.999), (0.999, 1.01)]
    rows = []
    for det in [d for d in DETS if d in sc]:
        T = sc[det].test
        s, y = T["score"].to_numpy(), T["y"].to_numpy()
        for lo, hi in bands:
            m = (s >= lo) & (s < hi)
            rows.append(dict(release=release, detector=det, band=f"[{lo:g}, {min(hi, 1):g}{']' if hi > 1 else ')'}",
                             user_days=int(m.sum()), malicious=int(y[m].sum()),
                             share_malicious=float(y[m].mean()) if m.any() else float("nan")))
        # does the score mass of the day's alerts track the number of true alerts?
        out = h5.simulate_h(sc[det], budget, 1.0, "g_functional_unit", history="global")
        D = out[out["alert"]].groupby("day").agg(mass=("score", "sum"), hits=("y", "sum"))
        from scipy.stats import spearmanr
        rho = spearmanr(D["mass"], D["hits"]).correlation if D["hits"].std() > 0 else float("nan")
        rows.append(dict(release=release, detector=det, band="daily score mass vs. true alerts (Spearman)",
                         user_days=len(D), malicious=int(D["hits"].sum()), share_malicious=float(rho)))
    return pd.DataFrame(rows)


# =============================================================================
# J8  r5.2 summary of burden and parity rules
# =============================================================================

def j8_rules(sc, budget=0.01, delta=0.002, group_col="g_functional_unit", release="r5.2"):
    pols = [("B0 global", dict(history="global")), ("B8 daily parity, no memory", dict(history="none")),
            ("B1r quota, randomized rounding", dict(allocator="randround", history="alerts")),
            ("B4 exact rule, label-free", dict(history="alerts")), ("B4c exact rule", dict(history="cleared")),
            ("B4c exact rule, discounted", dict(history="cleared", hist_decay=HALF30)), DEFAULT, DEFAULT_DISC,
            ("B0 + scheduling", dict(history="global", schedule=True)),
            ("B11c + scheduling", dict(allocator="deficit", history="cleared", schedule=True))]
    rows = []
    for det in [d for d in DETS if d in sc]:
        for pol, kw in pols:
            if kw.get("schedule") and sc[det].phi is None:
                continue
            out = h5.simulate_h(sc[det], budget, delta, group_col, **kw)
            m = h5.metrics(out, group_col, release=release, detector=det, policy=pol)
            m.update(h5.rolling_disparity(out, group_col))
            if pol == "B0 global":
                m["exposure_bias"] = fi.exposure_bias(sc[det].test, group_col=group_col)
            rows.append(m)
        log(f"J8 {release} {det} done")
    return pd.DataFrame(rows)

In [ ]:
%%writefile fairirs_rev7.py
"""Experiments for the repositioned paper (history-dependent alerting as fairness tool and attack surface).

K1  Discount farming against a cooldown after clearance (with and without the override), on r4.2 and r5.2
K2  Cooldown and scheduling on r5.2 (burden, detection, non-inferiority over seeds)
K3  Timing attacks against parity combined with a cooldown
"""
import time
import numpy as np
import pandas as pd
import fairirs as fi
import fairirs_rev5 as h5
import fairirs_rev6 as j6


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


SUPPRESSION = [
    ("B0 global", dict(history="global")),
    ("B0 + scheduling", dict(history="global", schedule=True)),
    ("B0 + cooldown 7 days", dict(history="global", cooldown=7)),
    ("B0 + cooldown 30 days", dict(history="global", cooldown=30)),
    ("B0 + cooldown 30 days, no override", dict(history="global", cooldown=30, tau_ov=1.01)),
    ("B0 + cooldown 30 days, override 0.995", dict(history="global", cooldown=30, tau_ov=0.995)),
    ("B11c + cooldown 30 days", dict(allocator="deficit", history="cleared", cooldown=30)),
]


def k1_farming(sc, budget=0.01, delta=0.002, group_col="g_functional_unit", detectors=("GBM", "AE", "IF", "LR"),
               alphas=(0.5, 1.0), n_farm=(3,), release="r4.2"):
    """Farming insiders make the benign days before their attack resemble it, hoping that these
    days are alerted and cleared, so that the attack falls under the cooldown or the discount."""
    rows = []
    for det in [d for d in detectors if d in sc]:
        sd = sc[det]
        for nf in n_farm:
            for alpha in alphas:
                sdf, farmers = fi.discount_farming(sd, n_farm=nf, alpha=alpha)
                if not farmers:
                    continue
                for pol, kw in SUPPRESSION:
                    res, inv = {}, 0
                    for lab, s_ in (("no farming", sd), ("farming", sdf)):
                        out = h5.simulate_h(s_, budget, delta, group_col, **kw)
                        sub = out[out["user"].isin(farmers) & (out["y"] == 1)]
                        res[lab] = sub.groupby("user")["alert"].any()
                        if lab == "farming":
                            inv = int(out[out["user"].isin(farmers) & (out["y"] == 0)]["alert"].sum())
                    a, b = res["no farming"].align(res["farming"], join="inner")
                    rows.append(dict(release=release, detector=det, n_farm=nf, alpha=alpha, policy=pol,
                                     farmers=len(farmers), detected_no_farming=int(a.sum()),
                                     detected_farming=int(b.sum()), lost=int((a & ~b).sum()),
                                     gained=int((~a & b).sum()), farmer_benign_alerts=inv))
            log(f"K1 {release} {det} done")
    return pd.DataFrame(rows)


def k3_attacks(sc, budget=0.01, delta=0.002, group_col="g_functional_unit", detectors=("GBM", "LSTM"), release="r4.2"):
    pols = [("B11c + cooldown 30 days", dict(allocator="deficit", history="cleared", cooldown=30)),
            ("B4c + cooldown 30 days", dict(history="cleared", cooldown=30))]
    df = h5.i7_attacks(sc, budget, delta, group_col, detectors=detectors, det_draws=3, rand_draws=3, policies=pols)
    df["release"] = release
    return df

In [ ]:
%%writefile fairirs_rev8.py
"""Experiments for the security review (revision 8).

L0  Check that rescoring reproduces the cached scores, and compare score-preserving with
    behavioral relocation under the global policy (random timing)
L1  Timing attacks against parity with behavioral relocation and rescoring
L2  Multiplicity: prespecified primary attacker, descriptive worst case, joint bootstrap of the maximum
L3  Realistic farming: attacker knowledge, number of probes, clearance probability, feasibility cap
L4  Score- and group-dependent verdict errors, and inconclusive verdicts
L5  Social cost of burden: break-even ratio of repeat-alert cost to first-alert cost
"""
import time
import numpy as np
import pandas as pd
import fairirs as fi
import fairirs_rev as fr
import fairirs_rev5 as h5
import fairirs_rev6 as j6
import fairirs_rev7 as k7

CF = fi.COUNT_FEATURES
VF = fi.VOLUME_FEATURES


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


# =============================================================================
# Rescoring modified behavior with the saved models
# =============================================================================

def training_context(features_path, split_day):
    """Training-period feature rows (all users), needed so that the LSTM sees the same
    history before the test period as when the cached scores were computed."""
    F = pd.read_pickle(features_path)
    F = F[(F["exposure"] > 0) & (F["day"] < split_day)]
    return F[["user", "day"] + CF + ["weekend"]].copy()


class Rescorer:
    """Rescores test rows whose features were changed, with the detector that produced the
    cached scores. Unchanged rows keep their raw score; all rows are then put back on the
    causal quantile scale of the benign training-period scores."""

    def __init__(self, sd, context=None):
        self.sd = sd
        self.seq = bool(getattr(sd.model, "sequence", False))
        if isinstance(sd.model, fi.PerGroupDetector) or sd.model is None:
            raise ValueError(f"{sd.detector}: rescoring needs a single saved model")
        if self.seq and context is None:
            raise ValueError(f"{sd.detector}: the LSTM needs the training-period features")
        self.context = context

    def raw(self, T, rows):
        rows = np.asarray(sorted(set(int(r) for r in rows)), int)
        out = T["raw_score"].to_numpy(np.float64).copy()
        if len(rows) == 0:
            return out
        tf, model = self.sd.transform, self.sd.model
        if not self.seq:
            out[rows] = model.score(tf.transform(T.iloc[rows]))
            return out
        users = T["user"].iloc[rows].unique()
        Tu = T[T["user"].isin(users)]
        C = self.context[self.context["user"].isin(users)]
        cols = ["user", "day"] + CF + ["weekend"]
        Fu = pd.concat([C[cols].assign(_t=-1), Tu[cols].assign(_t=Tu.index.to_numpy())], ignore_index=True)
        r = model.score_seq(Fu, tf.transform(Fu))
        t = Fu["_t"].to_numpy()
        m = t >= 0
        out[t[m]] = r[m]
        return out

    def __call__(self, T, rows, name=None):
        T = T.copy()
        rows = sorted(set(int(r) for r in rows))
        if rows:
            T.loc[rows, "exposure"] = T.loc[rows, VF].sum(axis=1)
        raw = self.raw(T, rows)
        T["raw_score"] = raw
        T["score"] = fi.to_quantile_ref(raw, self.sd.raw_ref)
        phi = self.sd.phi
        if phi is not None and rows:
            phi = phi.copy()
            tf = self.sd.transform
            phi[rows] = phi[rows] + (tf.transform(T.loc[rows]) - tf.transform(self.sd.test.loc[rows]))
        return fi.ScoredData(name or self.sd.detector, T, phi, self.sd.model, self.sd.transform,
                             self.sd.raw_ref, self.sd.train_ref)


def _float_features(T):
    T = T.copy()
    for c in CF + ["exposure", "raw_score"]:
        if c in T:
            T[c] = T[c].astype(np.float64)
    return T


def relocate_behavior(sd, group_col, chooser, seed, rescorer):
    """Behavioral timing attack. Each insider keeps the malicious actions of the episode but
    performs them on other days: the malicious increment over the insider's typical benign
    day, (x_malicious - median benign)+, is added to the benign activity of the destination
    days, and the vacated days become typical benign days. Exposure is recomputed and the
    saved detector rescores the modified history."""
    rng = np.random.default_rng(seed)
    T = _float_features(sd.test)
    changed = []
    for u, mal_idx in fr._episodes(T).items():
        rows = T.index[T["user"] == u].to_numpy()
        L = len(mal_idx)
        if len(rows) <= L:
            continue
        st = chooser(u, rows, T, L, rng)
        st = int(min(max(st, 0), len(rows) - L))
        new = rows[st:st + L]
        ben = T.loc[rows][T.loc[rows, "y"] == 0]
        base = np.round(ben[CF].median().to_numpy(np.float64)) if len(ben) else np.zeros(len(CF))
        inc = np.clip(T.loc[mal_idx, CF].to_numpy(np.float64) - base, 0, None)
        scen = T.loc[mal_idx[0], "scenario"]
        T.loc[mal_idx, CF] = base
        T.loc[mal_idx, "y"] = 0
        T.loc[mal_idx, "scenario"] = ""
        T.loc[new, CF] = T.loc[new, CF].to_numpy(np.float64) + inc
        T.loc[new, "y"] = 1
        T.loc[new, "scenario"] = scen
        changed.extend(list(mal_idx) + list(new))
    T["y"] = T["y"].astype(np.int8)
    return rescorer(T, changed)


def make_shifter(rescorers):
    def shifter(det, sd0, group_col, fn, dr):
        return relocate_behavior(sd0, group_col, fn, dr, rescorers[det])
    return shifter


# =============================================================================
# L0  Checks
# =============================================================================

def l0_checks(sc, rescorers, budget=0.01, group_col="g_functional_unit", draws=5, release="r4.2"):
    rows = []
    for det, R in rescorers.items():
        sd = sc[det]
        same = R(_float_features(sd.test), [])
        moved_rows = sorted(set(int(i) for i in sd.test.index[sd.test["y"] == 1]))[:200]
        again = R(_float_features(sd.test), moved_rows)
        out0 = h5.simulate_h(sd, budget, 1.0, group_col, history="global")
        for dr in range(draws):
            rand = lambda u, rows_, T, L, rng: int(rng.integers(0, len(rows_) - L + 1))
            sp = fr.shift_to(sd, group_col, rand, dr)
            bh = relocate_behavior(sd, group_col, rand, dr, R)
            o_sp = h5.simulate_h(sp, budget, 1.0, group_col, history="global")
            o_bh = h5.simulate_h(bh, budget, 1.0, group_col, history="global")
            rows.append(dict(release=release, detector=det, draw=dr,
                             max_abs_diff_no_change=float(np.abs(same.test["score"] - sd.test["score"]).max()),
                             max_abs_diff_rescore_unchanged=float(np.abs(again.test["raw_score"] - sd.test["raw_score"]).max()),
                             insiders_detected_original=int(h5.insider_hits(out0).sum()),
                             insiders_detected_score_preserving=int(h5.insider_hits(o_sp).sum()),
                             insiders_detected_behavioral=int(h5.insider_hits(o_bh).sum()),
                             mean_malicious_score_original=float(sd.test.loc[sd.test["y"] == 1, "score"].mean()),
                             mean_malicious_score_behavioral=float(bh.test.loc[bh.test["y"] == 1, "score"].mean()),
                             share_malicious_above_0999_original=float((sd.test.loc[sd.test["y"] == 1, "score"] >= 0.999).mean()),
                             share_malicious_above_0999_behavioral=float((bh.test.loc[bh.test["y"] == 1, "score"] >= 0.999).mean())))
        log(f"L0 {release} {det} done")
    return pd.DataFrame(rows)


# =============================================================================
# L1/L2  Timing attacks with rescoring, and multiplicity
# =============================================================================

L1_POLICIES = [("B4 label-free", dict(history="alerts")),
               ("B4c outcome-aware", dict(history="cleared")),
               ("B4c outcome-aware, 7-day verdict delay", dict(history="cleared", delay=7)),
               j6.DEFAULT]
PRIMARY = "estimates deficit"


def l1_attacks(sc, rescorers, budget=0.01, delta=0.002, group_col="g_functional_unit",
               detectors=("GBM", "LSTM", "AE"), det_draws=5, rand_draws=10, n_boot=5000, release="r4.2"):
    collect = []
    dets = tuple(d for d in detectors if d in sc and d in rescorers)
    df = h5.i7_attacks(sc, budget, delta, group_col, detectors=dets, det_draws=det_draws, rand_draws=rand_draws,
                       n_boot=n_boot, policies=L1_POLICIES, shifter=make_shifter(rescorers), collect=collect)
    df["release"] = release
    per = pd.DataFrame(collect)
    per["release"] = release
    return df, per


def l2_multiplicity(per, primary=PRIMARY, n_boot=5000, alpha=0.05, seed=0):
    """per: one row per (detector, policy, attack, draw, insider) with d = allocation effect under
    attack minus without. Excess loss of an attacker = -sum over insiders of d averaged over draws."""
    rows = []
    rng = np.random.default_rng(seed)
    for (rel, det, pol), D in per.groupby(["release", "detector", "policy"], sort=False):
        M = D.groupby(["user", "attack"])["d"].mean().unstack("attack").fillna(0.0)
        atk = list(M.columns)
        X = -M.to_numpy()
        n, A = X.shape
        est = X.sum(0)
        idx = rng.integers(0, n, size=(n_boot, n))
        B = X[idx].sum(1)                                   # n_boot x A, same resample for every attacker
        mx = B.max(1)
        a_lo, a_hi = 100 * alpha / 2, 100 * (1 - alpha / 2)
        bon = alpha / max(A, 1)
        w = int(np.argmax(est))
        r = dict(release=rel, detector=det, policy=pol, insiders=n, attackers=A,
                 worst_attacker=atk[w], worst_excess=float(est[w]),
                 joint_max_lo=float(np.percentile(mx, a_lo)), joint_max_hi=float(np.percentile(mx, a_hi)),
                 joint_max_one_sided_lo=float(np.percentile(mx, 100 * alpha)))
        if primary in atk:
            p = atk.index(primary)
            r.update(primary_attacker=primary, primary_excess=float(est[p]),
                     primary_lo=float(np.percentile(B[:, p], a_lo)), primary_hi=float(np.percentile(B[:, p], a_hi)))
        for j, a in enumerate(atk):
            r[f"{a} | excess"] = float(est[j])
            r[f"{a} | bonferroni_lo"] = float(np.percentile(B[:, j], 100 * bon / 2))
            r[f"{a} | bonferroni_hi"] = float(np.percentile(B[:, j], 100 * (1 - bon / 2)))
        rows.append(r)
    return pd.DataFrame(rows)


# =============================================================================
# L3  Realistic farming
# =============================================================================

FARM_POLICIES = [("B0 global", dict(history="global")),
                 ("B0 + scheduling", dict(history="global", schedule=True)),
                 ("B0 + cooldown 30 days", dict(history="global", cooldown=30)),
                 ("B0 + cooldown 30 days, no override", dict(history="global", cooldown=30, tau_ov=1.01)),
                 ("B11c + cooldown 30 days", dict(allocator="deficit", history="cleared", cooldown=30))]
FARM_GRID = ([(k, n, 1.0) for k in ("own", "scenario", "generic") for n in (1, 3, 5)] +
             [(k, 3, p) for k in ("own", "scenario", "generic") for p in (0.5, 0.8)])


def farm(sd, rescorer, knowledge="own", n_probe=3, p_clear=1.0, alpha=1.0, cap_q=0.99, role_col="role", min_rows=50):
    """Each insider makes the n_probe benign days before the attack resemble it, so that they
    are alerted and cleared first. knowledge: 'own' (the insider's own planned increment),
    'scenario' (mean increment of the other insiders of the same scenario), 'generic' (mean
    increment of all other insiders). Each probe feature is capped at the 99th percentile of
    benign days of non-insiders in the same role (a probe beyond what colleagues do would be
    implausible); a probe alert is confirmed (not cleared) with probability 1 - p_clear."""
    T = _float_features(sd.test)
    mal = T[T["y"] == 1]
    ins = list(mal["user"].unique())
    inc, scen = {}, {}
    for u, D in mal.groupby("user"):
        ub = T[(T["user"] == u) & (T["y"] == 0)]
        b = ub[CF].median() if len(ub) else pd.Series(0.0, index=CF)
        inc[u] = (D[CF].mean() - b).clip(lower=0).to_numpy(np.float64)
        scen[u] = D["scenario"].iloc[0]
    ben = T[(T["y"] == 0) & ~T["user"].isin(ins)]
    gcap = ben[CF].quantile(cap_q).to_numpy(np.float64)
    caps = {}
    if role_col in T:
        for r_, D in ben.groupby(role_col):
            caps[r_] = D[CF].quantile(cap_q).to_numpy(np.float64) if len(D) >= min_rows else gcap
    T["fc"] = np.nan
    changed, farmers, capped = [], [], 0.0
    for u in ins:
        others = [v for v in ins if v != u]
        if knowledge == "own":
            d = inc[u]
        elif knowledge == "scenario":
            same = [v for v in others if scen[v] == scen[u]]
            d = np.mean([inc[v] for v in (same or others)], axis=0) if others else inc[u]
        else:
            d = np.mean([inc[v] for v in others], axis=0) if others else inc[u]
        d0 = mal.loc[mal["user"] == u, "day"].min()
        prior = T[(T["user"] == u) & (T["y"] == 0) & (T["day"] < d0)].sort_values("day").tail(n_probe)
        if prior.empty:
            continue
        idx = prior.index
        cap = caps.get(T.loc[idx[0], role_col], gcap) if role_col in T else gcap
        x = T.loc[idx, CF].to_numpy(np.float64)
        want = np.round(x + alpha * d)
        got = np.maximum(x, np.minimum(want, cap))
        capped += float((got < want).mean())
        T.loc[idx, CF] = got
        T.loc[idx, "fc"] = 1.0 - p_clear
        changed.extend(idx)
        farmers.append(u)
    sdf = rescorer(T, changed, name=sd.detector + "-farm")
    return sdf, farmers, list(changed), capped / max(len(farmers), 1)


def l3_farming(sc, rescorers, budget=0.01, delta=0.002, group_col="g_functional_unit",
               detectors=("GBM", "AE", "LR", "IF"), grid=FARM_GRID, policies=FARM_POLICIES, seeds=3, release="r4.2"):
    rows = []
    for det in [d for d in detectors if d in sc and d in rescorers]:
        sd = sc[det]
        base = {}
        for pol, kw in policies:
            if kw.get("schedule") and sd.phi is None:
                continue
            o = h5.simulate_h(sd, budget, delta, group_col, **kw)
            base[pol] = o[o["y"] == 1].groupby("user")["alert"].any()
        for knowledge, n_probe, p_clear in grid:
            sdf, farmers, probes, capped = farm(sd, rescorers[det], knowledge, n_probe, p_clear)
            if not farmers:
                continue
            probe_set = np.zeros(len(sdf.test), bool)
            probe_set[probes] = True
            for pol, kw in policies:
                if pol not in base:
                    continue
                for s in range(seeds if p_clear < 1 else 1):
                    o = h5.simulate_h(sdf, budget, delta, group_col, fc_col="fc", seed=s, **kw)
                    att = o[(o["y"] == 1) & o["user"].isin(farmers)].groupby("user")["alert"].any()
                    pr = o[probe_set]
                    probed = pr[pr["alert"]].groupby("user").size()
                    exposed = pr[pr["alert"] & ~pr["cleared"]].groupby("user").size()
                    a0 = base[pol].reindex(farmers).fillna(False).astype(bool)
                    a1 = att.reindex(farmers).fillna(False).astype(bool)
                    ex = pd.Series(False, index=farmers)
                    ex[ex.index.isin(exposed.index)] = True
                    caught = a1 | ex
                    rows.append(dict(release=release, detector=det, knowledge=knowledge, n_probe=n_probe,
                                     p_clear=p_clear, policy=pol, seed=s, farmers=len(farmers),
                                     probe_features_capped=capped,
                                     probes_alerted_insiders=int(len(probed)),
                                     detected_no_farming=int(a0.sum()), attack_detected_farming=int(a1.sum()),
                                     exposed_by_probe=int(ex.sum()), caught_farming=int(caught.sum()),
                                     evaded=int((a0 & ~caught).sum()), gained=int((~a0 & caught).sum())))
        log(f"L3 {release} {det} done")
    return pd.DataFrame(rows)


def l3_summary(df):
    keys = ["release", "detector", "knowledge", "n_probe", "p_clear", "policy"]
    return df.groupby(keys, sort=False)[["farmers", "probe_features_capped", "probes_alerted_insiders",
                                          "detected_no_farming", "attack_detected_farming", "exposed_by_probe",
                                          "caught_farming", "evaded", "gained"]].mean().reset_index()


# =============================================================================
# L4  Verdict errors and inconclusive verdicts
# =============================================================================

ERR_POLICIES = [("B0 global", dict(history="global")),
                ("B0 + scheduling", dict(history="global", schedule=True)),
                ("B0 + cooldown 30 days", dict(history="global", cooldown=30)),
                ("B0 + cooldown 30 days, no override", dict(history="global", cooldown=30, tau_ov=1.01)),
                ("B4c outcome-aware", dict(history="cleared")),
                j6.DEFAULT,
                ("B11c + cooldown 30 days", dict(allocator="deficit", history="cleared", cooldown=30))]


def error_conditions(sd, group_col):
    sizes = sd.test.groupby(group_col)["user"].nunique().sort_values()
    small, large = sizes.index[0], sizes.index[-1]
    return [("no errors", {}),
            ("score-dependent misses (0.30 below 0.999, 0.05 above)", dict(p_miss=0.05, p_miss_low=0.30)),
            ("score-dependent false confirmations (0.01 below 0.999, 0.10 above)", dict(p_fc=0.01, p_fc_high=0.10)),
            (f"group-dependent: smallest unit ({small}) worse", dict(p_miss=0.05, p_fc=0.01, err_group=small, p_miss_g=0.30, p_fc_g=0.05)),
            (f"group-dependent: largest unit ({large}) worse", dict(p_miss=0.05, p_fc=0.01, err_group=large, p_miss_g=0.30, p_fc_g=0.05)),
            ("inconclusive 20%", dict(p_inconclusive=0.2)),
            ("inconclusive 50%", dict(p_inconclusive=0.5)),
            ("score-dependent errors + inconclusive 50%", dict(p_miss=0.05, p_miss_low=0.30, p_fc=0.01, p_fc_high=0.10, p_inconclusive=0.5))]


def l4_errors(sc, budget=0.01, delta=0.002, group_col="g_functional_unit", detectors=j6.DETS, seeds=2, release="r4.2"):
    rows = []
    for det in [d for d in detectors if d in sc]:
        sd = sc[det]
        for cond, ekw in error_conditions(sd, group_col):
            for pol, kw in ERR_POLICIES:
                if kw.get("schedule") and sd.phi is None:
                    continue
                for s in range(seeds if ekw else 1):
                    out = h5.simulate_h(sd, budget, delta, group_col, seed=s, **kw, **ekw)
                    m = h5.metrics(out, group_col, release=release, detector=det, policy=pol, condition=cond, seed=s)
                    m["confirmed_alerts_on_insiders"] = int((out["alert"] & ~out["cleared"] & (out["y"] == 1)).sum())
                    rows.append(m)
        log(f"L4 {release} {det} done")
    return pd.DataFrame(rows)


# =============================================================================
# L5  Social cost
# =============================================================================

COST_POLICIES = list(k7.SUPPRESSION) + [j6.DEFAULT, ("B4c outcome-aware", dict(history="cleared"))]


def l5_social_cost(sc, budget=0.01, delta=0.002, group_col="g_functional_unit", detectors=j6.DETS,
                   ratios=(0.1, 0.25, 0.5, 1.0, 2.0), release="r4.2"):
    """C = sum_i [c1 * 1(F_i > 0) + c2 * (F_i - 1)+] over benign employees (c1 = 1).
    Relative to the global policy, C_P - C_G = dN + r dR with r = c2/c1, so the policy is
    cheaper for every r above the break-even r* = -dN/dR when dR < 0 (and so on)."""
    rows = []
    for det in [d for d in detectors if d in sc]:
        sd = sc[det]
        ins = set(sd.test.loc[sd.test["y"] == 1, "user"])
        base = None
        for pol, kw in COST_POLICIES:
            if kw.get("schedule") and sd.phi is None:
                continue
            out = h5.simulate_h(sd, budget, delta, group_col, **kw)
            b = out[(out["y"] == 0) & ~out["user"].isin(ins)]
            F = b.groupby("user")["alert"].sum()
            N, R = int((F > 0).sum()), int((F - 1).clip(lower=0).sum())
            hits = int(h5.insider_hits(out).sum())
            r = dict(release=release, detector=det, policy=pol, alerted_employees=N, repeat_alerts=R,
                     false_alerts=int(F.sum()), max_F=int(F.max()) if len(F) else 0, insiders_detected=hits)
            if pol == "B0 global":
                base = r
            if base is not None:
                dN, dR = N - base["alerted_employees"], R - base["repeat_alerts"]
                r["d_alerted_employees"], r["d_repeat_alerts"] = dN, dR
                r["d_insiders_detected"] = hits - base["insiders_detected"]
                if dR != 0:
                    rs = -dN / dR
                    r["break_even_ratio"] = rs
                    r["cheaper_when"] = "never" if (dR > 0 and dN >= 0) else ("always" if (dR < 0 and dN <= 0) else
                                        (f"c2/c1 > {rs:.3g}" if dR < 0 else f"c2/c1 < {rs:.3g}"))
                else:
                    r["break_even_ratio"] = np.nan
                    r["cheaper_when"] = "always" if dN < 0 else ("never" if dN > 0 else "equal")
                for q in ratios:
                    r[f"cost_vs_global_r{q}"] = dN + q * dR
            rows.append(r)
        log(f"L5 {release} {det} done")
    return pd.DataFrame(rows)

In [ ]:
%%writefile fairirs_rev9.py
"""Experiments on attack realism (revision 9).

M1  Online timing attackers: the insider moves forward day by day, sees only the past, and
    starts the episode the first time its signal crosses a threshold set from the values it
    has seen so far. Hindsight attackers are kept as upper bounds. Behavior is relocated and
    rescored, and each relocated attack is scored under the parity rule and the global policy.
M2  Plausibility of the modified behavior: distance to benign days of the same role and
    structural consistency of the counts, for relocated attack days and farming probes;
    excess loss and evasions as plausibility requirements become stricter.
"""
import time
import numpy as np
import pandas as pd
import fairirs as fi
import fairirs_rev as fr
import fairirs_rev5 as h5
import fairirs_rev6 as j6
import fairirs_rev8 as l8

CF = fi.COUNT_FEATURES


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


# =============================================================================
# Plausibility
# =============================================================================

SUBTOTALS = [("logon_ah", ["logon_n"]), ("dev_ah", ["dev_n"]), ("file_ah", ["file_n"]), ("file_usb", ["file_n"]),
             ("email_ah", ["email_n"]), ("email_ext", ["email_n"]), ("email_bcc", ["email_n"]),
             ("http_ah", ["http_n"]), ("http_job", ["http_n"]), ("http_leak", ["http_n"]),
             ("http_cloud", ["http_n"]), ("http_upload", ["http_n"])]
EXT = ["file_exe", "file_zip", "file_doc", "file_pdf", "file_txt", "file_jpg"]


def violations(F):
    """Number of structural violations per row: a subtotal above its total, or file
    extension classes summing to more than the file events."""
    v = np.zeros(len(F), int)
    for sub, tot in SUBTOTALS:
        if sub in F and tot[0] in F:
            v += (F[sub].to_numpy(np.float64) > F[tot[0]].to_numpy(np.float64) + 1e-9).astype(int)
    if all(c in F for c in EXT + ["file_n"]):
        v += (F[EXT].sum(axis=1).to_numpy(np.float64) > F["file_n"].to_numpy(np.float64) + 1e-9).astype(int)
    return v


def repair(T, rows):
    """Make modified rows structurally consistent: each total is raised to at least its
    largest subtotal (and file events to at least the sum of extension classes), as the
    actions that produce a subtotal also produce the total."""
    rows = list(rows)
    if not rows:
        return T
    for sub, tot in SUBTOTALS:
        if sub in T and tot[0] in T:
            T.loc[rows, tot[0]] = np.maximum(T.loc[rows, tot[0]].to_numpy(np.float64), T.loc[rows, sub].to_numpy(np.float64))
    if all(c in T for c in EXT + ["file_n"]):
        T.loc[rows, "file_n"] = np.maximum(T.loc[rows, "file_n"].to_numpy(np.float64), T.loc[rows, EXT].sum(axis=1).to_numpy(np.float64))
    return T


class RepairingRescorer:
    """Repairs structural consistency of the changed rows, then rescores them."""

    def __init__(self, rescorer):
        self.r = rescorer
        self.sd = rescorer.sd

    def __call__(self, T, rows, name=None):
        return self.r(repair(T, rows), rows, name=name)


class Plausibility:
    """Distance of a user-day to its k-th nearest benign day of a non-insider in the same
    role, in the detector's standardized feature space, expressed as the percentile of that
    distance among the role's benign days themselves (leave-one-out)."""

    def __init__(self, sd, k=5, max_ref=4000, role_col="role", min_rows=50, seed=0):
        from sklearn.neighbors import NearestNeighbors
        rng = np.random.default_rng(seed)
        T = sd.test
        ins = set(T.loc[T["y"] == 1, "user"])
        ben = T[(T["y"] == 0) & ~T["user"].isin(ins)]
        self.tf, self.k, self.role_col = sd.transform, k, role_col
        self.models, self.ref = {}, {}
        groups = list(ben.groupby(role_col)) if role_col in T else []
        allb = ben.sample(min(len(ben), max_ref), random_state=seed)
        for key, D in groups + [("__all__", allb)]:
            if key != "__all__" and len(D) < min_rows:
                continue
            D = D.sample(min(len(D), max_ref), random_state=int(rng.integers(1 << 30)))
            X = self.tf.transform(D)
            nn = NearestNeighbors(n_neighbors=min(k + 1, len(D))).fit(X)
            d, _ = nn.kneighbors(X)
            self.models[key] = nn
            self.ref[key] = np.sort(d[:, -1])               # leave-one-out: first neighbour is the point itself

    def percentile(self, F, raw=False):
        out = np.full(len(F), np.nan)
        keys = F[self.role_col].to_numpy() if self.role_col in F else np.array(["__all__"] * len(F))
        X = self.tf.transform(F)
        for key in np.unique(keys):
            m = keys == key
            kk = key if key in self.models else "__all__"
            nn, ref = self.models[kk], self.ref[kk]
            d, _ = nn.kneighbors(X[m], n_neighbors=min(self.k, len(ref)))
            out[m] = d[:, -1] / np.median(ref) if raw else np.searchsorted(ref, d[:, -1], side="right") / len(ref)
        return out

    def distance(self, F):
        """k-NN distance relative to the median k-NN distance of the role's benign days."""
        return self.percentile(F, raw=True)


# =============================================================================
# M1  Online timing attackers
# =============================================================================

def _online_start(sig, L, burn, q, require_positive=True):
    """First position j >= burn whose signal is at least the q-quantile of the signals seen
    at positions before j (and positive). If the signal never triggers, the insider acts at
    the last possible position. Returns (position, triggered)."""
    last = len(sig) - L
    for j in range(min(burn, last), last + 1):
        h = sig[:j]
        if len(h) == 0:
            continue
        if (not require_positive or sig[j] > 0) and sig[j] >= np.quantile(h, q):
            return j, True
    return last, False


M1_POLICIES = l8.L1_POLICIES
PRIMARY_ONLINE = "online deficit, q=0.9"


def m1_online(sc, rescorers, budget=0.01, delta=0.002, group_col="g_functional_unit",
              detectors=("GBM", "LSTM", "AE"), policies=M1_POLICIES, qs=(0.8, 0.9, 0.95), burn=30,
              noisy_draws=5, rand_draws=10, window=30, eps=1.0, n_boot=5000, plaus=None, release="r4.2"):
    """Returns (summary rows per draw, per-insider rows). Per insider: P0, G0 (detected under
    the parity rule and the global policy without attack), PA, GA (with the attack), the
    relocated days' plausibility percentile and structural violations, and whether the online
    rule triggered."""
    rows, per = [], []
    for det in [d for d in detectors if d in sc and d in rescorers]:
        sd0, R = sc[det], RepairingRescorer(rescorers[det])
        out_g0 = h5.simulate_h(sd0, budget, 1.0, group_col, history="global")
        hit_g0 = h5.insider_hits(out_g0)
        P = plaus.get(det) if plaus else None
        for pol, kw in policies:
            out0, rec = h5.simulate_h(sd0, budget, delta, group_col, record=True, seed=0, **kw)
            hit0 = h5.insider_hits(out0)
            r = rec.copy()
            r["cut"] = (r["K"] - r["k"]).clip(lower=0)
            cut = r.set_index(["day", "group"])["cut"].to_dict()
            obs = {}
            for lab, D in r.sort_values("day").groupby("group"):
                a = D["k"].rolling(window, min_periods=1).sum().shift(1).fillna(0)
                n = D["n"].rolling(window, min_periods=1).sum().shift(1).fillna(1)
                for dd, v in zip(D["day"], (a / n).to_numpy()):
                    obs[(dd, lab)] = v
            rep = fr.report_targets(out0, group_col)
            allm = sorted(rep.index)
            prev_month = {m: (allm[i - 1] if i > 0 else None) for i, m in enumerate(allm)}

            def deficit_map(seed):
                M = h5._monthly_noisy(rec, eps, np.random.default_rng(500 + seed))
                months = sorted(M["month"].unique())
                cumk = M.pivot(index="month", columns="group", values="k").fillna(0).cumsum()
                cumn = M.pivot(index="month", columns="group", values="n").fillna(0).cumsum()
                overall = cumk.sum(axis=1) / cumn.sum(axis=1)
                Rr = r.copy()
                Rr["month"] = pd.to_datetime(Rr["day"]).dt.strftime("%Y-%m")
                est = {}
                for lab, D in Rr.sort_values("day").groupby("group"):
                    D = D.copy()
                    D["prev"] = D["month"].map(lambda mth: months[months.index(mth) - 1] if months.index(mth) > 0 else None)
                    first = D["month"] != D["month"].shift(1)
                    D["k_since"] = D.groupby("month")["k"].cumsum().shift(1).fillna(0)
                    D.loc[first, "k_since"] = 0
                    D["n_since"] = D.groupby("month")["n"].cumsum().shift(1).fillna(0)
                    D.loc[first, "n_since"] = 0
                    for dd, pm, ks, ns in zip(D["day"], D["prev"], D["k_since"], D["n_since"]):
                        if pm is None or pd.isna(pm) or lab not in cumk.columns:
                            est[(dd, lab)] = 0.0
                            continue
                        ck, cn = cumk.loc[pm, lab] + ks, cumn.loc[pm, lab] + ns
                        est[(dd, lab)] = ck / max(cn, 1) - overall.loc[pm]
                return est

            def signal(kind, rows_, T, est=None):
                gu = T.loc[rows_, group_col].to_numpy()
                dd = T.loc[rows_, "day"].to_numpy()
                if kind == "deficit":
                    return np.array([est.get((dd[j], gu[j]), 0.0) for j in range(len(rows_))], float)
                if kind == "colleagues":
                    return np.array([obs.get((dd[j], gu[j]), 0.0) for j in range(len(rows_))], float)
                if kind == "reports":
                    mon = pd.to_datetime(T.loc[rows_, "day"]).dt.strftime("%Y-%m").to_numpy()
                    g = gu[0]
                    out = np.zeros(len(rows_))
                    if g in rep.columns:
                        for j, m in enumerate(mon):
                            pm = prev_month.get(m)
                            if pm is not None and not pd.isna(rep.loc[pm, g]):
                                out[j] = rep.loc[pm, g]
                    return out
                if kind == "cut":
                    return np.array([cut.get((dd[j], gu[j]), 0) for j in range(len(rows_))], float)
                raise ValueError(kind)

            trig = {}

            def make(kind, mode, q=None, est=None, name=None):
                def fn(u, rows_, T, L, rng):
                    s = signal(kind, rows_, T, est)
                    if mode == "hindsight":
                        if kind == "cut":
                            s = np.convolve(s, np.ones(L), "valid")
                        else:
                            s = s[:len(rows_) - L + 1]
                        trig[u] = True
                        return int(rng.choice(np.flatnonzero(s == s.max())))
                    j, ok = _online_start(s, L, burn, q)
                    trig[u] = ok
                    return j
                return fn

            def rand(u, rows_, T, L, rng):
                trig[u] = True
                return int(rng.integers(0, len(rows_) - L + 1))

            attackers = [("oracle: knows allocation (hindsight)", lambda dr: make("cut", "hindsight"), 3),
                         ("hindsight deficit", lambda dr: make("deficit", "hindsight", est=deficit_map(dr)), noisy_draws)]
            for q in qs:
                attackers.append((f"online deficit, q={q}", lambda dr, q=q: make("deficit", "online", q, est=deficit_map(dr)), noisy_draws))
            attackers += [("online colleagues, q=0.9", lambda dr: make("colleagues", "online", 0.9), 1),
                          ("online reports, q=0.9", lambda dr: make("reports", "online", 0.9), 1),
                          ("random", lambda dr: rand, rand_draws)]
            for name, factory, nd in attackers:
                for dr in range(nd):
                    trig.clear()
                    fn = factory(dr)
                    sd = l8.relocate_behavior(sd0, group_col, fn, dr, R)
                    outP = h5.simulate_h(sd, budget, delta, group_col, seed=dr + 1, **kw)
                    outG = h5.simulate_h(sd, budget, 1.0, group_col, history="global")
                    hP, hG = h5.insider_hits(outP), h5.insider_hits(outG)
                    D = pd.concat([hit0.rename("P0"), hit_g0.rename("G0"), hP.rename("PA"), hG.rename("GA")], axis=1).fillna(0)
                    d_ = ((D["PA"] - D["GA"]) - (D["P0"] - D["G0"])).to_numpy()
                    mal = sd.test[sd.test["y"] == 1]
                    pl = pd.Series(P.distance(mal), index=mal["user"].to_numpy()).groupby(level=0).max() if P else pd.Series(dtype=float)
                    vi = pd.Series(violations(mal), index=mal["user"].to_numpy()).groupby(level=0).sum()
                    for u, row in D.iterrows():
                        per.append(dict(release=release, detector=det, policy=pol, attack=name, draw=dr, user=u,
                                        P0=int(row.P0), G0=int(row.G0), PA=int(row.PA), GA=int(row.GA),
                                        d=float((row.PA - row.GA) - (row.P0 - row.G0)),
                                        triggered=bool(trig.get(u, True)),
                                        plaus_max=float(pl.get(u, np.nan)), violations=int(vi.get(u, 0))))
                    bs = h5._boot_mean(d_, n_boot, seed=dr)
                    rows.append(dict(release=release, detector=det, policy=pol, attack=name, draw=dr, insiders=len(D),
                                     none_P=int(D["P0"].sum()), none_G=int(D["G0"].sum()),
                                     attacked_P=int(D["PA"].sum()), attacked_G=int(D["GA"].sum()),
                                     excess_loss=float(-d_.sum()), excess_lo=float(-np.percentile(bs, 97.5) * len(D)),
                                     excess_hi=float(-np.percentile(bs, 2.5) * len(D)),
                                     triggered_share=float(np.mean(list(trig.values()))) if trig else np.nan))
            log(f"M1 {release} {det} {pol} done")
    return pd.DataFrame(rows), pd.DataFrame(per)


def original_plausibility(sc, plaus, release="r4.2"):
    """Plausibility percentile and violations of the original (unmoved) malicious days, per
    insider, the reference for relocated attacks."""
    rows = []
    for det, P in plaus.items():
        T = sc[det].test
        mal = T[T["y"] == 1]
        pl = pd.Series(P.distance(mal), index=mal["user"].to_numpy()).groupby(level=0).max()
        vi = pd.Series(violations(mal), index=mal["user"].to_numpy()).groupby(level=0).sum()
        ben = T[T["y"] == 0]
        for u in pl.index:
            rows.append(dict(release=release, detector=det, user=u, plaus_max=float(pl[u]), violations=int(vi[u])))
        rows.append(dict(release=release, detector=det, user="__benign_violation_rate__",
                         plaus_max=np.nan, violations=float((violations(ben) > 0).mean())))
    return pd.DataFrame(rows)


def m1_constrained(per, orig, levels=("none", "no violations beyond original", "within original 100%",
                                      "within original 90%", "within original 75%", "within original 50%"),
                   n_boot=5000, seed=0):
    """Excess loss when only insiders whose relocated attack is plausible may move: an
    insider whose relocation fails the requirement keeps the original timing (contribution 0).
    'within original x%': the relocated days' largest plausibility percentile does not exceed
    the x-th percentile of the same quantity over the original attacks (same release and
    detector); violations: no more structural violations than the insider's original days."""
    rng = np.random.default_rng(seed)
    out = []
    O = orig[orig["user"] != "__benign_violation_rate__"]
    for (rel, det, pol, atk), D in per.groupby(["release", "detector", "policy", "attack"], sort=False):
        o = O[(O["release"] == rel) & (O["detector"] == det)].set_index("user")
        if o.empty:
            continue
        for lev in levels:
            ok = np.ones(len(D), bool)
            if lev == "no violations beyond original":
                ok = D["violations"].to_numpy() <= o["violations"].reindex(D["user"]).fillna(0).to_numpy()
            elif lev.startswith("within original"):
                x = float(lev.split()[-1].rstrip("%")) / 100
                thr = np.nanquantile(o["plaus_max"].to_numpy(), x)
                ok = (D["plaus_max"].to_numpy() <= thr) & (D["violations"].to_numpy() <= o["violations"].reindex(D["user"]).fillna(0).to_numpy())
            dd = np.where(ok, D["d"].to_numpy(), 0.0)
            M = pd.DataFrame(dict(user=D["user"].to_numpy(), d=dd)).groupby("user")["d"].mean()
            X = -M.to_numpy()
            idx = rng.integers(0, len(X), size=(n_boot, len(X)))
            B = X[idx].sum(1)
            out.append(dict(release=rel, detector=det, policy=pol, attack=atk, constraint=lev,
                            admissible_share=float(ok.mean()), excess=float(X.sum()),
                            lo=float(np.percentile(B, 2.5)), hi=float(np.percentile(B, 97.5))))
    return pd.DataFrame(out)


def m1_decomposition(per):
    """2 x 2 components and per-insider outcome classes, averaged over draws."""
    g = per.groupby(["release", "detector", "policy", "attack", "draw"], sort=False)
    S = g[["P0", "G0", "PA", "GA"]].sum()
    lost_mech = ((per["GA"] == 1) & (per["PA"] == 0)).astype(int)
    lost_both = ((per["GA"] == 0) & (per["PA"] == 0) & ((per["G0"] == 1) | (per["P0"] == 1))).astype(int)
    tmp = per.assign(lost_by_mechanism_only=lost_mech, lost_by_both=lost_both,
                     gained_by_mechanism_only=((per["GA"] == 0) & (per["PA"] == 1)).astype(int))
    S2 = tmp.groupby(["release", "detector", "policy", "attack", "draw"], sort=False)[
        ["lost_by_mechanism_only", "lost_by_both", "gained_by_mechanism_only"]].sum()
    S = S.join(S2)
    S["relocation_effect_global"] = S["GA"] - S["G0"]
    S["relocation_effect_policy"] = S["PA"] - S["P0"]
    S["excess_loss"] = S["relocation_effect_global"] - S["relocation_effect_policy"]
    return S.groupby(level=[0, 1, 2, 3], sort=False).mean().reset_index()


def m1_multiplicity(per, primary=PRIMARY_ONLINE, family_prefix="online", n_boot=5000, alpha=0.05, seed=0):
    """Primary online attacker with its own interval; Bonferroni over the online family."""
    rng = np.random.default_rng(seed)
    out = []
    for (rel, det, pol), D in per.groupby(["release", "detector", "policy"], sort=False):
        M = D.groupby(["user", "attack"])["d"].mean().unstack("attack").fillna(0.0)
        atk = list(M.columns)
        X = -M.to_numpy()
        idx = rng.integers(0, X.shape[0], size=(n_boot, X.shape[0]))
        B = X[idx].sum(1)
        fam = [j for j, a in enumerate(atk) if a.startswith(family_prefix)]
        bon = alpha / max(len(fam), 1)
        r = dict(release=rel, detector=det, policy=pol, insiders=X.shape[0])
        for j, a in enumerate(atk):
            r[f"{a} | excess"] = float(X[:, j].sum())
            r[f"{a} | lo"] = float(np.percentile(B[:, j], 100 * alpha / 2))
            r[f"{a} | hi"] = float(np.percentile(B[:, j], 100 * (1 - alpha / 2)))
            if j in fam:
                r[f"{a} | bonferroni_lo"] = float(np.percentile(B[:, j], 100 * bon / 2))
        out.append(r)
    return pd.DataFrame(out)


# =============================================================================
# M2  Plausibility of farming probes
# =============================================================================

M2_POLICIES = [("B0 global", dict(history="global")),
               ("B0 + scheduling", dict(history="global", schedule=True)),
               ("B0 + cooldown 30 days", dict(history="global", cooldown=30)),
               ("B0 + cooldown 30 days, no override", dict(history="global", cooldown=30, tau_ov=1.01))]


def m2_farming(sc, rescorers, plaus, budget=0.01, delta=0.002, group_col="g_functional_unit",
               detectors=("GBM", "AE", "LR", "IF"), grid=(("own", 1), ("own", 3), ("generic", 1), ("generic", 3)),
               levels=(1.0, 0.9, 0.75, 0.5), release="r4.2"):
    """Per farming insider: evaded or not, the largest distance of its probes to benign days of
    its role, and structural violations. A probe must be alerted to work, so it is compared with
    the benign days that the detector itself ranks in its top 1% (the unusual but innocent days
    analysts already see and clear): at level x, a farmer may farm only if its probes are no
    farther from ordinary benign days than the x-quantile of those innocent alerted days and
    violate no count constraint; otherwise it does not farm (evasion counted as 0). Level 'own':
    probes no farther than the insider's own most unusual benign day."""
    per, summ, alerted_ref_by = [], [], {}
    for det in [d for d in detectors if d in sc and d in rescorers and d in plaus]:
        sd, R, P = sc[det], RepairingRescorer(rescorers[det]), plaus[det]
        T0 = sd.test
        ins0 = set(T0.loc[T0["y"] == 1, "user"])
        Bn = T0[(T0["y"] == 0) & ~T0["user"].isin(ins0)]
        Bn = Bn[Bn["score"] >= Bn["score"].quantile(0.99)]
        Bn = Bn.sample(min(len(Bn), 3000), random_state=0)
        alerted_ref = np.sort(P.distance(Bn))
        alerted_ref_by[det] = alerted_ref
        base = {}
        for pol, kw in M2_POLICIES:
            if kw.get("schedule") and sd.phi is None:
                continue
            o = h5.simulate_h(sd, budget, delta, group_col, **kw)
            base[pol] = o[o["y"] == 1].groupby("user")["alert"].any()
        for knowledge, n in grid:
            sdf, farmers, probes, _ = l8.farm(sd, R, knowledge, n, 1.0)
            if not farmers:
                continue
            Tp = sdf.test.loc[probes]
            pp = pd.Series(P.percentile(Tp), index=Tp["user"].to_numpy()).groupby(level=0).max()
            pd_ = pd.Series(P.distance(Tp), index=Tp["user"].to_numpy()).groupby(level=0).max()
            own = sdf.test[sdf.test["user"].isin(farmers) & (sdf.test["y"] == 0)].drop(index=probes)
            od = pd.Series(P.distance(own), index=own["user"].to_numpy()).groupby(level=0).max()
            vv = pd.Series(violations(Tp), index=Tp["user"].to_numpy()).groupby(level=0).sum()
            for pol, kw in M2_POLICIES:
                if pol not in base:
                    continue
                o = h5.simulate_h(sdf, budget, delta, group_col, fc_col="fc", **kw)
                att = o[(o["y"] == 1) & o["user"].isin(farmers)].groupby("user")["alert"].any()
                pr = o.loc[probes]
                exposed = set(pr.loc[pr["alert"] & ~pr["cleared"], "user"])
                for u in farmers:
                    a0 = bool(base[pol].get(u, False))
                    a1 = bool(att.get(u, False)) or (u in exposed)
                    per.append(dict(release=release, detector=det, knowledge=knowledge, n_probe=n, policy=pol, user=u,
                                    detected_no_farming=a0, evaded=a0 and not a1,
                                    probe_plaus_max=float(pp.get(u, np.nan)), probe_distance=float(pd_.get(u, np.nan)),
                                    own_benign_max_distance=float(od.get(u, np.nan)), probe_violations=int(vv.get(u, 0))))
        log(f"M2 {release} {det} done")
    per = pd.DataFrame(per)
    for (rel, det, k, n, pol), D in per.groupby(["release", "detector", "knowledge", "n_probe", "policy"], sort=False):
        for lev in list(levels) + ["own"]:
            if lev == "own":
                ok = (D["probe_distance"] <= D["own_benign_max_distance"]) & (D["probe_violations"] == 0)
            else:
                thr = np.quantile(alerted_ref_by[det], lev)
                ok = (D["probe_distance"] <= thr) & (D["probe_violations"] == 0)
            summ.append(dict(release=rel, detector=det, knowledge=k, n_probe=n, policy=pol, level=str(lev),
                             farmers=len(D), admissible=int(np.sum(ok)), detected_no_farming=int(D["detected_no_farming"].sum()),
                             evaded=int((D["evaded"] & ok).sum()), evaded_unconstrained=int(D["evaded"].sum())))
    return per, pd.DataFrame(summ)

In [ ]:
%%writefile fairirs_rev10.py
"""Experiments for the referee reports (revision 10).

N1  Burden and detection for every retrained seed (seed means for the main tables), with a
    randomized global policy, convex costs of repeated suspicion and the daily budget k
N2  Timing attacks as unilateral deviations: one insider relocates while every other employee
    keeps the original behavior; every feasible start day is evaluated against the allocation
    the others produce, which gives an oracle that maximizes the excess loss; over seeds; a
    sample is checked with full simulations
N3  Farming over seeds with delayed and insider-level verdicts, earlier probes, a cooldown
    without backfill, and three defenses
N4  Per-employee-baseline detectors: benefit, farming and unilateral timing
N5  Power of the permutation test and a size-weighted statistic
N6  Insiders whose episodes span the training/test split
"""
import time
import numpy as np
import pandas as pd
import fairirs as fi
import fairirs_rev as fr
import fairirs_rev5 as h5
import fairirs_rev6 as j6
import fairirs_rev8 as l8
import fairirs_rev9 as m9

CF = fi.COUNT_FEATURES


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def gini(x):
    x = np.sort(np.asarray(x, float))
    if x.sum() == 0:
        return 0.0
    n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1).dot(x) / (n * x.sum()))


# =============================================================================
# N1  Burden and detection over seeds, randomized global policy, convex costs
# =============================================================================

def randomized_global(sd, budget=0.01, rho=0.2, pool=5, seed=0):
    """Memoryless randomized policy: each day the top (1 - rho) k scores are alerted, and rho k
    more are drawn uniformly from the next (pool - 1) k ranks."""
    rng = np.random.default_rng(seed)
    T = sd.test
    days = T["day"].to_numpy()
    s = T["score"].to_numpy()
    alert = np.zeros(len(T), bool)
    for d in np.unique(days):
        idx = np.flatnonzero(days == d)
        k = max(1, int(round(budget * len(idx))))
        order = idx[np.argsort(-s[idx], kind="stable")]
        k1 = int(round((1 - rho) * k))
        alert[order[:k1]] = True
        cand = order[k1:min(len(order), pool * k)]
        if len(cand) and k - k1 > 0:
            alert[rng.choice(cand, size=min(k - k1, len(cand)), replace=False)] = True
    out = T[["user", "day", "y", "score", "exposure", "scenario"] + [c for c in T.columns if c.startswith("g_")]].copy()
    out["alert"] = alert
    out["cleared"] = alert & (T["y"].to_numpy() == 0)
    return out


N1_POLICIES = [("B0 global", dict(history="global")),
               ("B0 + cooldown 30 days", dict(history="global", cooldown=30)),
               ("B0 + cooldown 30 days, no override", dict(history="global", cooldown=30, tau_ov=1.01)),
               ("B0 + cooldown 30 days, no backfill", dict(history="global", cooldown=30, cd_backfill=False)),
               ("B0 + scheduling", dict(history="global", schedule=True)),
               ("B4c outcome-aware", dict(history="cleared")),
               j6.DEFAULT]


def burden_row(out, group_col, **extra):
    m = h5.metrics(out, group_col, **extra)
    ins = set(out.loc[out["y"] == 1, "user"])
    b = out[(out["y"] == 0) & ~out["user"].isin(ins)]
    F = b.groupby("user")["alert"].sum()
    Fp = F[F > 0].to_numpy(float)
    m.update(alerted_employees=int((F > 0).sum()), repeat_alerts=int((F - 1).clip(lower=0).sum()),
             false_alerts=int(F.sum()), max_F_benign=int(F.max()) if len(F) else 0,
             gini_F=gini(F.to_numpy()), top1_share=float(np.sort(F.to_numpy())[::-1][:max(1, len(F) // 100)].sum() / max(F.sum(), 1)),
             cost_g1_5=float((Fp ** 1.5).sum()), cost_g2=float((Fp ** 2).sum()))
    return m


def n1_seeds(runs, budget=0.01, delta=0.002, group_col="g_functional_unit", release="r4.2"):
    """runs: {detector: {seed: ScoredData}}."""
    rows = []
    for det, by_seed in runs.items():
        for seed, sd in by_seed.items():
            for pol, kw in N1_POLICIES:
                if kw.get("schedule") and sd.phi is None:
                    continue
                out = h5.simulate_h(sd, budget, delta, group_col, **kw)
                rows.append(burden_row(out, group_col, release=release, detector=det, seed=seed, policy=pol))
            for rho in (0.2, 0.5):
                out = randomized_global(sd, budget, rho=rho, seed=seed)
                rows.append(burden_row(out, group_col, release=release, detector=det, seed=seed, policy=f"randomized global, rho={rho}"))
            log(f"N1 {release} {det} seed {seed} done")
    return pd.DataFrame(rows)


def daily_budget(sd, budget=0.01, release="r4.2"):
    n = sd.test.groupby("day").size()
    k = np.maximum(1, np.round(budget * n)).astype(int)
    return pd.DataFrame(dict(release=release, day=n.index, active=n.to_numpy(), k=k.to_numpy()))


# =============================================================================
# Signals of the timing attackers (as in revision 9)
# =============================================================================

def timing_signals(out0, rec, group_col, window=30):
    r = rec.copy()
    r["cut"] = (r["K"] - r["k"]).clip(lower=0)
    cut = r.set_index(["day", "group"])["cut"].to_dict()
    obs = {}
    for lab, D in r.sort_values("day").groupby("group"):
        a = D["k"].rolling(window, min_periods=1).sum().shift(1).fillna(0)
        n = D["n"].rolling(window, min_periods=1).sum().shift(1).fillna(1)
        for dd, v in zip(D["day"], (a / n).to_numpy()):
            obs[(dd, lab)] = v
    return cut, obs, r


def deficit_map(rec_r, rec, eps, seed):
    M = h5._monthly_noisy(rec, eps, np.random.default_rng(500 + seed))
    months = sorted(M["month"].unique())
    cumk = M.pivot(index="month", columns="group", values="k").fillna(0).cumsum()
    cumn = M.pivot(index="month", columns="group", values="n").fillna(0).cumsum()
    overall = cumk.sum(axis=1) / cumn.sum(axis=1)
    Rr = rec_r.copy()
    Rr["month"] = pd.to_datetime(Rr["day"]).dt.strftime("%Y-%m")
    est = {}
    for lab, D in Rr.sort_values("day").groupby("group"):
        D = D.copy()
        D["prev"] = D["month"].map(lambda mth: months[months.index(mth) - 1] if mth in months and months.index(mth) > 0 else None)
        first = D["month"] != D["month"].shift(1)
        D["k_since"] = D.groupby("month")["k"].cumsum().shift(1).fillna(0)
        D.loc[first, "k_since"] = 0
        D["n_since"] = D.groupby("month")["n"].cumsum().shift(1).fillna(0)
        D.loc[first, "n_since"] = 0
        for dd, pm, ks, ns in zip(D["day"], D["prev"], D["k_since"], D["n_since"]):
            if pm is None or (isinstance(pm, float) and np.isnan(pm)) or lab not in cumk.columns:
                est[(dd, lab)] = 0.0
                continue
            ck, cn = cumk.loc[pm, lab] + ks, cumn.loc[pm, lab] + ns
            est[(dd, lab)] = ck / max(cn, 1) - overall.loc[pm]
    return est


# =============================================================================
# N2  Unilateral timing attacks
# =============================================================================

class DayAllocator:
    """Recomputes one day's parity allocation with one row's score changed, from the history
    the allocator saw that day in the unattacked run (unilateral deviation)."""

    def __init__(self, sd, kw, delta, group_col, state):
        T = sd.test
        self.s = T["score"].to_numpy(np.float64)
        self.g = T[group_col].to_numpy()
        self.state, self.kw, self.delta = state, kw, delta
        self.rows_by_day = T.groupby("day").indices
        self.raw_sorted = np.sort(T["raw_score"].to_numpy(np.float64))
        self.ref, self.N = sd.raw_ref, len(T)

    def score_of(self, raw):
        q = np.searchsorted(self.ref, raw, side="right") / len(self.ref)
        return q + 1e-6 * (np.searchsorted(self.raw_sorted, raw, side="right") + 1) / (self.N + 1)

    def alerted(self, d, own_row, raw):
        st = self.state.get(d)
        if st is None:
            return None
        A0, C0, k, use_global = st
        idx = self.rows_by_day[d]
        sc = self.s[idx].copy()
        pos = np.flatnonzero(idx == own_row)
        if len(pos) == 0:
            return None
        sc[pos[0]] = self.score_of(raw)
        gi = self.g[idx]
        if self.kw.get("allocator") == "deficit":
            m = h5._deficit(sc, gi, k, A0, C0)
        else:
            m = fi.fair_allocate_cum(sc, gi, k, self.delta, A0, C0)[0]
        return bool(m[pos[0]])


class FixedState:
    """Alert thresholds of a completed run: on each day, the k-th largest raw score of each
    group (parity) or overall (global), with the insider's own row removed. An insider's
    relocated user-day is alerted if its raw score exceeds the threshold of its day."""

    def __init__(self, T, out, group_col):
        self.raw = T["raw_score"].to_numpy(np.float64)
        self.day = T["day"].to_numpy()
        self.grp = T[group_col].to_numpy()
        a = out["alert"].to_numpy()
        self.tab = {}
        self.agree = 0.0
        recomputed = np.zeros(len(T), bool)
        key = pd.Series(list(zip(self.day, self.grp)))
        for (d, g), idx in key.groupby(key).groups.items():
            idx = np.asarray(idx)
            r = np.sort(self.raw[idx])[::-1]
            k = int(a[idx].sum())
            self.tab[(d, g)] = (r, k)
            if k > 0:
                recomputed[idx[np.argsort(-self.raw[idx], kind="stable")[:k]]] = True
        self.agree = float((recomputed == a).mean())

    def near(self, d, g, own_raw, extra=5):
        """Score the insider would need to be within `extra` ranks of the group's cut."""
        r, k = self.tab.get((d, g), (np.array([]), 0))
        j = k + extra - 1
        return r[j] if len(r) > j else -np.inf

    def threshold(self, d, g, own_raw):
        r, k = self.tab.get((d, g), (np.array([]), 0))
        if k <= 0:
            return np.inf
        if own_raw is not None and len(r) >= k and own_raw >= r[k - 1]:
            return r[k] if len(r) > k else -np.inf
        return r[k - 1] if len(r) >= k else -np.inf


def _insider_candidates(sd, R, u, group_col, seq_ctx=None, stride=1):
    """Raw scores of the insider's relocated malicious days for every feasible start.
    Returns (rows, mal_idx, L, starts, raw[starts, L], days[starts, L])."""
    T = sd.test
    rows = T.index[T["user"] == u].to_numpy()
    mal_idx = T.index[(T["user"] == u) & (T["y"] == 1)].to_numpy()
    L, M = len(mal_idx), len(rows)
    if L == 0 or M <= L:
        return None
    ben = T.loc[rows][T.loc[rows, "y"] == 0]
    base = np.round(ben[CF].median().to_numpy(np.float64)) if len(ben) else np.zeros(len(CF))
    inc = np.clip(T.loc[mal_idx, CF].to_numpy(np.float64) - base, 0, None)
    X = T.loc[rows, CF].to_numpy(np.float64).copy()
    is_mal = np.isin(rows, mal_idx)
    X[is_mal] = base
    starts = np.arange(0, M - L + 1, stride)
    tf, model = sd.transform, sd.model
    seq = bool(getattr(model, "sequence", False))
    dayv = T.loc[rows, "day"].to_numpy()
    days = np.stack([dayv[s:s + L] for s in starts])
    if not seq:
        blocks = []
        for s in starts:
            Z = X[s:s + L] + inc
            blocks.append(Z)
        Z = np.concatenate(blocks)
        D = pd.DataFrame(Z, columns=CF)
        D["weekend"] = T.loc[rows, "weekend"].to_numpy()[np.concatenate([np.arange(s, s + L) for s in starts])]
        for c in T.columns:
            if c not in D and c in ("user", "role") or c.startswith("g_"):
                D[c] = T.loc[rows[0], c]
        D = m9.repair(D, D.index)
        raw = model.score(tf.transform(D)).reshape(len(starts), L)
        return rows, mal_idx, L, starts, raw, days
    # sequence model: score each candidate history as its own pseudo-employee
    C = seq_ctx[seq_ctx["user"] == u]
    cols = ["user", "day"] + CF + ["weekend"]
    Tu = T.loc[rows, cols].copy()
    for c in CF:
        Tu[c] = Tu[c].astype(np.float64)
    frames, tpos = [], []
    for s in starts:
        Z = X.copy()
        Z[s:s + L] = Z[s:s + L] + inc
        Tz = Tu.copy()
        Tz[CF] = Z
        Tz = m9.repair(Tz, Tz.index[s:s + L])
        uid = f"{u}#{s}"
        frames.append(C[cols].assign(user=uid, _p=-1))
        frames.append(Tz.assign(user=uid, _p=np.arange(len(rows))))
    Fu = pd.concat(frames, ignore_index=True)
    r = model.score_seq(Fu, tf.transform(Fu))
    raw = np.empty((len(starts), L))
    Fu["_r"] = r
    for j, s in enumerate(starts):
        sub = Fu[(Fu["user"] == f"{u}#{s}") & (Fu["_p"] >= 0)].sort_values("_p")
        raw[j] = sub["_r"].to_numpy()[s:s + L]
    return rows, mal_idx, L, starts, raw, days


N2_POLICIES = [("B4 label-free", dict(history="alerts")),
               ("B4c outcome-aware", dict(history="cleared")),
               ("B4c outcome-aware, 7-day verdict delay", dict(history="cleared", delay=7)),
               j6.DEFAULT]


def n2_unilateral(runs, rescorers, budget=0.01, delta=0.002, group_col="g_functional_unit", policies=N2_POLICIES,
                  burn=30, q=0.9, noisy_draws=5, eps=1.0, seq_ctx=None, release="r4.2", seq_stride=1):
    """runs: {detector: {seed: ScoredData}}; rescorers: same shape. Returns per-insider rows:
    for each attacker, the insider's excess contribution (G1 - P1) - (G0 - P0), its predicted
    P1 and G1, and the chosen start."""
    out_rows, agree_rows = [], []
    for det, by_seed in runs.items():
        for seed, sd in by_seed.items():
            T = sd.test
            outG = h5.simulate_h(sd, budget, 1.0, group_col, history="global")
            Tg = T.assign(_all="all")
            outG2 = outG.assign(_all="all")
            FG = FixedState(Tg, outG2, "_all")
            hitG0 = h5.insider_hits(outG)
            ctx = seq_ctx if getattr(sd.model, "sequence", False) else None
            cand = {}
            for u in T.loc[T["y"] == 1, "user"].unique():
                c = _insider_candidates(sd, None, u, group_col, ctx, stride=seq_stride if ctx is not None else 1)
                if c is not None:
                    cand[u] = c
            for pol, kw in policies:
                state = {}
                out0, rec = h5.simulate_h(sd, budget, delta, group_col, record=True, seed=0, state_log=state, **kw)
                FP = FixedState(T, out0, group_col)
                DA = DayAllocator(sd, kw, delta, group_col, state)
                agree_rows.append(dict(release=release, detector=det, seed=seed, policy=pol, agree_parity=FP.agree, agree_global=FG.agree))
                hitP0 = h5.insider_hits(out0)
                cut, obs, rr = timing_signals(out0, rec, group_col)
                defs = [deficit_map(rr, rec, eps, dr) for dr in range(noisy_draws)]
                for u, (rows, mal_idx, L, starts, raw, days) in cand.items():
                    g = T.loc[rows[0], group_col]
                    own = T["raw_score"].to_numpy()
                    own_by_day = dict(zip(T.loc[rows, "day"], own[rows]))
                    row_by_day = dict(zip(T.loc[rows, "day"], rows))
                    n_exact = 0
                    dvec = T.loc[rows, "day"].to_numpy()
                    tP = np.array([FP.threshold(x, g, own_by_day.get(x)) for x in dvec])
                    tG = np.array([FG.threshold(x, "all", own_by_day.get(x)) for x in dvec])
                    tN = np.array([FP.near(x, g, own_by_day.get(x)) for x in dvec])
                    pos = starts[:, None] + np.arange(L)[None, :]
                    aP = raw > tP[pos]
                    aG = raw > tG[pos]
                    P1 = aP.any(1).astype(int)
                    G1 = aG.any(1).astype(int)
                    cand_exact = (~aP) & ((raw > tG[pos]) | (raw > tN[pos]))
                    for j in np.flatnonzero((P1 == 0) & cand_exact.any(1)):
                        for l in np.flatnonzero(cand_exact[j]):
                            d = days[j, l]
                            n_exact += 1
                            if DA.alerted(d, row_by_day[d], raw[j, l]):
                                P1[j] = 1
                                break
                    p0, g0 = int(hitP0.get(u, 0)), int(hitG0.get(u, 0))
                    val = (G1 - P1) - (g0 - p0)
                    dd = T.loc[rows, "day"].to_numpy()
                    gcut = np.array([cut.get((x, g), 0) for x in dd], float)
                    sig_cut = np.convolve(gcut, np.ones(L), "valid")[starts]
                    choice = {"oracle: maximizes excess loss": [int(np.argmax(val))],
                              "oracle: evades the parity rule if possible": [int(np.argmax(1 - P1 + 1e-6 * val))],
                              "knows allocation: largest cut": [int(np.argmax(sig_cut))]}
                    hs, on = [], []
                    for est in defs:
                        s_ = np.array([est.get((x, g), 0.0) for x in dd])
                        hs.append(int(np.argmax(s_[starts])))
                        j_, _ = m9._online_start(s_, L, burn, q)
                        on.append(int(np.argmin(np.abs(starts - j_))))
                    choice["hindsight deficit"] = hs
                    choice[f"online deficit, q={q}"] = on
                    for name, js in choice.items():
                        out_rows.append(dict(release=release, detector=det, seed=seed, policy=pol, user=u, attack=name,
                                             d=float(np.mean(val[js])), P1=float(np.mean(P1[js])), G1=float(np.mean(G1[js])),
                                             P0=p0, G0=g0, start=int(starts[js[0]])))
                    out_rows.append(dict(release=release, detector=det, seed=seed, policy=pol, user=u, attack="random",
                                         d=float(val.mean()), P1=float(P1.mean()), G1=float(G1.mean()), P0=p0, G0=g0, start=-1))
                log(f"N2 {release} {det} seed {seed} {pol} done ({len(cand)} insiders)")
    return pd.DataFrame(out_rows), pd.DataFrame(agree_rows)


def relocate_one(sd, u, start, rescorer):
    """Relocate only insider u to the given start; every other row is unchanged."""
    T = l8._float_features(sd.test)
    rows = T.index[T["user"] == u].to_numpy()
    mal_idx = T.index[(T["user"] == u) & (T["y"] == 1)].to_numpy()
    L = len(mal_idx)
    ben = T.loc[rows][T.loc[rows, "y"] == 0]
    base = np.round(ben[CF].median().to_numpy(np.float64))
    inc = np.clip(T.loc[mal_idx, CF].to_numpy(np.float64) - base, 0, None)
    scen = T.loc[mal_idx[0], "scenario"]
    new = rows[start:start + L]
    T.loc[mal_idx, CF] = base
    T.loc[mal_idx, "y"] = 0
    T.loc[mal_idx, "scenario"] = ""
    T.loc[new, CF] = T.loc[new, CF].to_numpy(np.float64) + inc
    T.loc[new, "y"] = 1
    T.loc[new, "scenario"] = scen
    T["y"] = T["y"].astype(np.int8)
    return m9.RepairingRescorer(rescorer)(T, list(mal_idx) + list(new))


def n2_validate(per, runs, rescorers, budget=0.01, delta=0.002, group_col="g_functional_unit", policies=N2_POLICIES,
                n_insiders=12, attack="oracle: maximizes excess loss", seed_only=0, release="r4.2"):
    """Full simulations for a sample of insiders: relocate one insider at the chosen start and
    rerun the parity rule and the global policy."""
    rows = []
    rng = np.random.default_rng(0)
    for det, by_seed in runs.items():
        if seed_only not in by_seed:
            continue
        sd = by_seed[seed_only]
        for pol, kw in policies:
            D = per[(per["detector"] == det) & (per["seed"] == seed_only) & (per["policy"] == pol) & (per["attack"] == attack)]
            if D.empty:
                continue
            pick = D.sample(min(n_insiders, len(D)), random_state=int(rng.integers(1 << 30)))
            for _, r in pick.iterrows():
                sdx = relocate_one(sd, r["user"], int(r["start"]), rescorers[det][seed_only])
                oP = h5.simulate_h(sdx, budget, delta, group_col, seed=0, **kw)
                oG = h5.simulate_h(sdx, budget, 1.0, group_col, history="global")
                hp, hg = int(h5.insider_hits(oP).get(r["user"], 0)), int(h5.insider_hits(oG).get(r["user"], 0))
                rows.append(dict(release=release, detector=det, policy=pol, user=r["user"], start=int(r["start"]),
                                 P1_pred=r["P1"], G1_pred=r["G1"], P1_full=hp, G1_full=hg,
                                 others_P_changed=int((h5.insider_hits(oP).drop(r["user"], errors="ignore") !=
                                                       h5.insider_hits(h5.simulate_h(sd, budget, delta, group_col, seed=0, **kw)).drop(r["user"], errors="ignore")).sum())))
            log(f"N2 check {release} {det} {pol} done")
    return pd.DataFrame(rows)


def n2_summary(per, n_boot=5000, seed=0):
    """Excess loss per setting (seed mean), with an interval that resamples seeds and insiders."""
    rng = np.random.default_rng(seed)
    out = []
    for (rel, det, pol, atk), D in per.groupby(["release", "detector", "policy", "attack"], sort=False):
        M = D.pivot_table(index="user", columns="seed", values="d", aggfunc="mean").fillna(0.0)
        X = M.to_numpy()
        n, S = X.shape
        est = X.sum(0).mean()
        bi = rng.integers(0, n, size=(n_boot, n))
        bs = rng.integers(0, S, size=(n_boot, S))
        B = np.array([X[bi[b]][:, bs[b]].sum(0).mean() for b in range(n_boot)])
        out.append(dict(release=rel, detector=det, policy=pol, attack=atk, seeds=S, insiders=n,
                        excess=float(est), lo=float(np.percentile(B, 2.5)), hi=float(np.percentile(B, 97.5)),
                        per_seed_min=float(X.sum(0).min()), per_seed_max=float(X.sum(0).max()),
                        positive_insiders=float((X.mean(1) > 0).sum()), negative_insiders=float((X.mean(1) < 0).sum())))
    return pd.DataFrame(out)


# =============================================================================
# N3  Farming over seeds, verdict models and defenses
# =============================================================================

N3_VARIANTS = [("cooldown 30 d, override 0.999", dict(history="global", cooldown=30)),
               ("no override", dict(history="global", cooldown=30, tau_ov=1.01)),
               ("verdicts 7 days late", dict(history="global", cooldown=30, cd_delay=7)),
               ("verdicts 14 days late", dict(history="global", cooldown=30, cd_delay=14)),
               ("no backfill", dict(history="global", cooldown=30, cd_backfill=False)),
               ("defense: re-check 10% of suppressed alerts", dict(history="global", cooldown=30, cd_recheck=0.1)),
               ("defense: farming-signature monitor (score >= 0.99 within 7 days of a clearance)",
                dict(history="global", cooldown=30, cd_recent_days=7, cd_recent_tau=0.99)),
               ("defense: 3-day cooldown after clearing a score >= 0.99", dict(history="global", cooldown=30, cd_high_days=3)),
               ("global (no cooldown)", dict(history="global"))]


def farm_gap(sd, rescorer, knowledge="own", n_probe=3, gap=0, role_col="role", cap_q=0.99, min_rows=50):
    """As fairirs_rev8.farm with every probe cleared, but the probes are the n benign days ending
    `gap` days before the first malicious day, and probes are made consistent."""
    T = l8._float_features(sd.test)
    mal = T[T["y"] == 1]
    ins = list(mal["user"].unique())
    inc, scen = {}, {}
    for u, D in mal.groupby("user"):
        ub = T[(T["user"] == u) & (T["y"] == 0)]
        b = ub[CF].median() if len(ub) else pd.Series(0.0, index=CF)
        inc[u] = (D[CF].mean() - b).clip(lower=0).to_numpy(np.float64)
        scen[u] = D["scenario"].iloc[0]
    ben = T[(T["y"] == 0) & ~T["user"].isin(ins)]
    gcap = ben[CF].quantile(cap_q).to_numpy(np.float64)
    caps = {r_: (D[CF].quantile(cap_q).to_numpy(np.float64) if len(D) >= min_rows else gcap)
            for r_, D in ben.groupby(role_col)} if role_col in T else {}
    T["fc"] = np.nan
    changed, farmers = [], []
    for u in ins:
        others = [v for v in ins if v != u]
        d = inc[u] if knowledge == "own" else (np.mean([inc[v] for v in others], axis=0) if others else inc[u])
        d0 = mal.loc[mal["user"] == u, "day"].min()
        prior = T[(T["user"] == u) & (T["y"] == 0) & (T["day"] < d0 - pd.Timedelta(days=gap))].sort_values("day").tail(n_probe)
        if prior.empty:
            continue
        idx = prior.index
        cap = caps.get(T.loc[idx[0], role_col], gcap) if role_col in T else gcap
        x = T.loc[idx, CF].to_numpy(np.float64)
        T.loc[idx, CF] = np.maximum(x, np.minimum(np.round(x + d), cap))
        T.loc[idx, "fc"] = 0.0
        changed.extend(idx)
        farmers.append(u)
    sdf = m9.RepairingRescorer(rescorer)(T, changed, name=sd.detector + "-farm")
    return sdf, farmers, list(changed)


def insider_level_fc(T):
    """Benign user-days of an insider between the first and last malicious day are confirmed
    (an investigation of the insider during the episode finds it), not cleared."""
    fc = np.full(len(T), np.nan)
    mal = T[T["y"] == 1].groupby("user")["day"].agg(["min", "max"])
    for u, r in mal.iterrows():
        m = (T["user"] == u).to_numpy() & (T["y"] == 0).to_numpy() & (T["day"] >= r["min"]).to_numpy() & (T["day"] <= r["max"]).to_numpy()
        fc[m] = 1.0
    return fc


def n3_farming(runs, rescorers, budget=0.01, delta=0.002, group_col="g_functional_unit",
               grid=(("own", 3, 0), ("generic", 3, 0), ("generic", 1, 0), ("own", 3, 14)), release="r4.2"):
    rows = []
    for det, by_seed in runs.items():
        for seed, sd in by_seed.items():
            R = rescorers[det][seed]
            base_T = sd.test
            fc_ins = insider_level_fc(base_T)
            farms = {}
            for knowledge, n, gap in grid:
                farms[(knowledge, n, gap)] = farm_gap(sd, R, knowledge, n, gap)
            for vname, kw in N3_VARIANTS + [("insider-level verdicts", dict(history="global", cooldown=30))]:
                ins_level = vname == "insider-level verdicts"
                sd0 = sd
                if ins_level:
                    sd0 = fi.ScoredData(sd.detector, base_T.assign(fc=fc_ins), sd.phi, sd.model, sd.transform, sd.raw_ref, sd.train_ref)
                o = h5.simulate_h(sd0, budget, delta, group_col, fc_col="fc" if ins_level else None, **kw)
                b = burden_row(o, group_col)
                a0 = o[o["y"] == 1].groupby("user")["alert"].any()
                for (knowledge, n, gap), (sdf, farmers, probes) in farms.items():
                    Tf = sdf.test
                    if ins_level:
                        f2 = Tf["fc"].to_numpy().copy()
                        m = np.isnan(f2)
                        f2[m] = fc_ins[m]
                        Tf = Tf.assign(fc=f2)
                    sdx = fi.ScoredData(sdf.detector, Tf, sdf.phi, sdf.model, sdf.transform, sdf.raw_ref, sdf.train_ref)
                    of = h5.simulate_h(sdx, budget, delta, group_col, fc_col="fc", **kw)
                    a1 = of[(of["y"] == 1) & of["user"].isin(farmers)].groupby("user")["alert"].any()
                    det0 = a0.reindex(farmers).fillna(False).astype(bool)
                    det1 = a1.reindex(farmers).fillna(False).astype(bool)
                    probed = of.loc[probes]
                    rows.append(dict(release=release, detector=det, seed=seed, variant=vname, knowledge=knowledge, n_probe=n, gap=gap,
                                     farmers=len(farmers), detected_no_farming=int(det0.sum()), evaded=int((det0 & ~det1).sum()),
                                     gained=int((~det0 & det1).sum()), probes_alerted=int(probed["alert"].sum()),
                                     insiders_detected_variant=b["insiders_detected"], max_F_variant=b["max_F_benign"],
                                     alerted_employees_variant=b["alerted_employees"]))
            log(f"N3 {release} {det} seed {seed} done")
    return pd.DataFrame(rows)


# =============================================================================
# N4  Per-employee-baseline detectors
# =============================================================================

class UserTransform:
    """Standardized log counts minus the employee's own training-period mean."""

    def __init__(self, ft, base):
        self.ft, self.base = ft, base

    def transform(self, F):
        X = self.ft.transform(F)
        B = self.base.reindex(F["user"].astype(str).str.split("#").str[0].values).fillna(0.0).to_numpy(np.float32)
        return (X - B).astype(np.float32)


def run_detector_user2(ds, det):
    F = ds.df
    tr = F["train"].to_numpy()
    y = F["y"].to_numpy()
    fit = tr & (y == 0) if not det.supervised else tr
    ft = fi.FeatureTransform("raw").fit(F[fit])
    X0 = ft.transform(F)
    base = fi.user_baseline(F, X0, tr & (y == 0))
    utf = UserTransform(ft, base)
    X = utf.transform(F)
    det.fit(X[fit], y[fit])
    raw = det.score(X[~tr])
    ref = np.sort(det.score(X[tr & (y == 0)]))
    test = F[~tr].copy().reset_index(drop=True)
    test["raw_score"] = raw
    test["score"] = fi.to_quantile_ref(raw, ref)
    return fi.ScoredData(det.name + "-user", test, X[~tr].copy(), det, utf, ref, None)


# =============================================================================
# N5  Power of the permutation test
# =============================================================================

def _group_stats(F, n, g, labels):
    sF = np.bincount(g, weights=F, minlength=len(labels))
    sn = np.bincount(g, weights=n, minlength=len(labels))
    rate = sF / np.maximum(sn, 1)
    overall = sF.sum() / sn.sum()
    return rate.max() - rate.min(), float((sn * (rate - overall) ** 2).sum() / sn.sum())


def permutation_p(F, n, g, labels, n_perm, rng):
    obs = _group_stats(F, n, g, labels)
    cnt = np.zeros(2)
    for _ in range(n_perm):
        gp = rng.permutation(g)
        st = _group_stats(F, n, gp, labels)
        cnt += np.array([st[0] >= obs[0] - 1e-15, st[1] >= obs[1] - 1e-15])
    return (1 + cnt) / (1 + n_perm), obs


def n5_power(runs, budget=0.01, group_col="g_functional_unit", effects=(0.25, 0.5, 1.0, 2.0, 4.0),
             n_inject=50, n_perm=200, release="r4.2", seed=0):
    """Observed p-values for both statistics, and power: the false alerts of one group (the
    smallest or the largest) are raised by a relative effect r through Poisson additions."""
    rng = np.random.default_rng(seed)
    rows = []
    for det, by_seed in runs.items():
        sd = by_seed.get(0) or next(iter(by_seed.values()))
        out = h5.simulate_h(sd, budget, 1.0, group_col, history="global")
        ins = set(out.loc[out["y"] == 1, "user"])
        b = out[(out["y"] == 0) & ~out["user"].isin(ins)]
        E = b.groupby("user").agg(F=("alert", "sum"), n=("alert", "size"), g=(group_col, lambda s: s.mode().iloc[0]))
        labels = sorted(E["g"].unique())
        gi = E["g"].map({l: i for i, l in enumerate(labels)}).to_numpy()
        F, n = E["F"].to_numpy(float), E["n"].to_numpy(float)
        p, obs = permutation_p(F, n, gi, labels, 2000, rng)
        rows.append(dict(release=release, detector=det, kind="observed", group="", effect=0.0,
                         p_range=float(p[0]), p_weighted=float(p[1]), stat_range=obs[0], stat_weighted=obs[1]))
        sizes = pd.Series(gi).value_counts()
        for which, gsel in (("smallest", sizes.idxmin()), ("largest", sizes.idxmax())):
            mem = gi == gsel
            rate_g = F[mem].sum() / n[mem].sum()
            for r in effects:
                hits = np.zeros(2)
                for _ in range(n_inject):
                    F2 = F.copy()
                    F2[mem] = F2[mem] + rng.poisson(r * rate_g * n[mem])
                    pp, _ = permutation_p(F2, n, gi, labels, n_perm, rng)
                    hits += pp < 0.05
                rows.append(dict(release=release, detector=det, kind="power", group=f"{which} ({labels[gsel]}, {int(mem.sum())} employees)",
                                 effect=r, p_range=float(hits[0] / n_inject), p_weighted=float(hits[1] / n_inject)))
        log(f"N5 {release} {det} done")
    return pd.DataFrame(rows)


# =============================================================================
# N6  Insiders whose episodes span the split
# =============================================================================

def n6_spanning(ds, release="r4.2"):
    F = ds.df
    mal = F[F["y"] == 1]
    tr_ins = set(mal.loc[mal["train"], "user"])
    te_ins = set(mal.loc[~mal["train"], "user"])
    span = sorted(tr_ins & te_ins)
    return pd.DataFrame(dict(release=release, user=span)), dict(release=release, train_insiders=len(tr_ins),
                                                                test_insiders=len(te_ins), spanning=len(span))


def n6_detection(runs, spanning, budget=0.01, delta=0.002, group_col="g_functional_unit", release="r4.2"):
    """Insiders detected with and without the insiders whose episodes cross the split."""
    rows = []
    span = set(spanning)
    for det, by_seed in runs.items():
        for seed, sd in by_seed.items():
            for pol, kw in N1_POLICIES:
                if kw.get("schedule") and sd.phi is None:
                    continue
                h = h5.insider_hits(h5.simulate_h(sd, budget, delta, group_col, **kw))
                keep = h[~h.index.isin(span)]
                rows.append(dict(release=release, detector=det, seed=seed, policy=pol, insiders=len(h), detected=int(h.sum()),
                                 insiders_excluding_spanning=len(keep), detected_excluding_spanning=int(keep.sum())))
    return pd.DataFrame(rows)

In [ ]:
# ===== Load the cached scores =====
import importlib, pickle, warnings, traceback, json
import numpy as np, pandas as pd
import fairirs as fi, fairirs_rev as fr, fairirs_rev3 as g3, fairirs_rev5 as h5, fairirs_rev6 as j6, fairirs_rev7 as k7, fairirs_rev8 as l8, fairirs_rev9 as m9, fairirs_rev10 as n10
for m in (fi, fr, g3, h5, j6, k7, l8, m9, n10):
    importlib.reload(m)
warnings.filterwarnings('ignore')
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40); pd.set_option('display.max_rows', 300)
try:
    display
except NameError:
    display = print
def load(name):
    p = os.path.join(CACHE, name)
    if not os.path.exists(p):
        print('not found (skipped):', name); return None
    with open(p, 'rb') as f:
        return pickle.load(f)
SC = load('scored_r4.2_v2.pkl'); SC5 = load('scored_r5.2_v2.pkl')
print('r4.2:', list(SC), '| r5.2:', list(SC5))
def run_cached(name, fn):
    p = os.path.join(REV, name + '.csv')
    if os.path.exists(p):
        print('reloading finished part:', p); return pd.read_csv(p)
    df = fn(); df.to_csv(p, index=False); print('saved', p); return df
FAILED = []

## Seeds, datasets and rescorers

In [ ]:
SEEDS4 = {s: load(f'scored_{MAIN}_seed{s}_v2.pkl') for s in range(5)}
LSTM4 = {s: load(f'scored_{MAIN}_lstm_seed{s}_v2.pkl') for s in (1, 2)}
SEEDS5 = {1: load(f'scored_{NEW}_seed1_v2.pkl'), 2: load(f'scored_{NEW}_seed2_v2.pkl')}
def runs_for(sc, seeds, lstm):
    R = {}
    for det in ('IF', 'AE', 'LR', 'GBM', 'LSTM'):
        R[det] = {}
        if sc is not None and det in sc:
            R[det][0] = sc[det]
        for s, v in seeds.items():
            if v is not None and det in v and s != 0:
                R[det][s] = v[det]
        if det == 'LSTM':
            for s, v in lstm.items():
                if v is not None:
                    R[det][s] = v
        if not R[det]:
            del R[det]
    return R
RUNS = {'r4.2': runs_for(SC, SEEDS4, LSTM4), 'r5.2': runs_for(SC5, SEEDS5, {})}
for rel, R in RUNS.items():
    print(rel, {d: sorted(v) for d, v in R.items()})
DSS, CTX, RESC = {}, {}, {}
for rel, ver in (('r4.2', VERSION), ('r5.2', NEW_VERSION)):
    try:
        meta = json.load(open(os.path.join(CACHE, f'meta_{rel}.json')))
        DSS[rel] = fi.build_dataset(rel, os.path.join(RAW, rel), ANSWERS, ver, CACHE, use_http=meta['web_logs'],
                                    train_frac=0.5, min_group=MIN_GROUP, seed=0)
    except Exception:
        traceback.print_exc(); print(rel, ': dataset not loaded, N4 and N6 skipped for this release')
    split = min(v.test['day'].min() for d in RUNS[rel].values() for v in d.values())
    fp = os.path.join(CACHE, f'features_{rel}.pkl')
    CTX[rel] = l8.training_context(fp, split) if os.path.exists(fp) else None
    RESC[rel] = {}
    for det, by in RUNS[rel].items():
        RESC[rel][det] = {}
        for s, sd in by.items():
            try:
                RESC[rel][det][s] = l8.Rescorer(sd, CTX[rel])
            except Exception as e:
                print(rel, det, s, 'not rescorable:', e)

## N6: Insiders whose episodes cross the split

In [ ]:
SPAN = []
for rel, ds in DSS.items():
    try:
        lst, info = n10.n6_spanning(ds, rel); SPAN.append(lst); print(info)
    except Exception:
        traceback.print_exc(); FAILED.append(f'N6 {rel}')
if SPAN:
    pd.concat(SPAN).to_csv(os.path.join(REV, 'n6_spanning_insiders.csv'), index=False)
for L_ in SPAN:
    rel = L_['release'].iloc[0] if len(L_) else None
    if rel is None:
        continue
    tag = rel.replace('.', '')
    try:
        D6 = run_cached(f'n6_detection_{tag}', lambda: n10.n6_detection(RUNS[rel], L_['user'].tolist(), BUDGET, DELTA, MAIN_GROUP, release=rel))
        display(D6.groupby(['detector', 'policy'], sort=False)[['insiders', 'detected', 'insiders_excluding_spanning', 'detected_excluding_spanning']].mean().round(1))
    except Exception:
        traceback.print_exc(); FAILED.append(f'N6 detection {rel}')

## N1: Burden and detection over seeds

In [ ]:
for rel in RUNS:
    tag = rel.replace('.', '')
    try:
        N1 = run_cached(f'n1_seeds_{tag}', lambda rel=rel: n10.n1_seeds(RUNS[rel], BUDGET, DELTA, MAIN_GROUP, release=rel))
        display(N1.groupby(['detector', 'policy'], sort=False)[['insiders_detected', 'max_F_benign', 'alerted_employees', 'gini_F', 'delta_FAR']].mean().round(3))
        sd0 = next(iter(RUNS[rel]['GBM'].values()))
        run_cached(f'n1_daily_budget_{tag}', lambda: n10.daily_budget(sd0, BUDGET, rel))
    except Exception:
        traceback.print_exc(); FAILED.append(f'N1 {rel}')

## N2: Timing attacks as unilateral deviations

In [ ]:
for rel in RUNS:
    tag = rel.replace('.', '')
    try:
        sub = {d: dict(sorted(v.items())[:3]) if d == 'LSTM' else v for d, v in RUNS[rel].items() if d in ('GBM', 'AE', 'LSTM')}
        sub = {d: {s: sd for s, sd in v.items() if s in RESC[rel].get(d, {})} for d, v in sub.items()}
        p_per, p_ag = os.path.join(REV, f'n2_unilateral_{tag}_per_insider.csv'), os.path.join(REV, f'n2_agreement_{tag}.csv')
        if os.path.exists(p_per) and os.path.exists(p_ag):
            print('reloading finished part:', p_per); per = pd.read_csv(p_per)
        else:
            per, ag = n10.n2_unilateral(sub, RESC[rel], BUDGET, DELTA, MAIN_GROUP, seq_ctx=CTX[rel], release=rel, seq_stride=3)
            per.to_csv(p_per, index=False); ag.to_csv(p_ag, index=False)
        S = n10.n2_summary(per); S.to_csv(os.path.join(REV, f'n2_summary_{tag}.csv'), index=False)
        display(S[['detector', 'policy', 'attack', 'seeds', 'excess', 'lo', 'hi']].round(2))
        V = run_cached(f'n2_validation_{tag}', lambda: n10.n2_validate(per, sub, RESC[rel], BUDGET, DELTA, MAIN_GROUP, release=rel))
        print('agreement with full simulations: parity', (V.P1_pred == V.P1_full).mean(), '| global', (V.G1_pred == V.G1_full).mean())
    except Exception:
        traceback.print_exc(); FAILED.append(f'N2 {rel}')

## N3: Farming over seeds, verdict models and defenses

In [ ]:
for rel in RUNS:
    tag = rel.replace('.', '')
    try:
        sub = {d: {s: sd for s, sd in v.items() if s in RESC[rel].get(d, {})} for d, v in RUNS[rel].items() if d in ('GBM', 'AE', 'LR', 'IF')}
        N3 = run_cached(f'n3_farming_{tag}', lambda: n10.n3_farming(sub, RESC[rel], BUDGET, DELTA, MAIN_GROUP, release=rel))
        display(N3.groupby(['detector', 'variant'], sort=False)[['detected_no_farming', 'evaded', 'insiders_detected_variant', 'max_F_variant']].mean().round(1))
    except Exception:
        traceback.print_exc(); FAILED.append(f'N3 {rel}')

## N4: Per-employee-baseline detectors

In [ ]:
USER = {}
for rel, ds in DSS.items():
    tag = rel.replace('.', '')
    try:
        p = os.path.join(CACHE, f'scored_{rel}_user_v3.pkl')
        if os.path.exists(p):
            USER[rel] = pickle.load(open(p, 'rb'))
        else:
            USER[rel] = {}
            for det in (fi.IFDetector(0), fi.AEDetector(0), fi.GBMDetector(0)):
                sd = n10.run_detector_user2(ds, det); USER[rel][sd.detector] = sd
                print(rel, sd.detector, 'trained')
            pickle.dump(USER[rel], open(p, 'wb'))
        runs = {d: {0: sd} for d, sd in USER[rel].items()}
        resc = {d: {0: l8.Rescorer(sd)} for d, sd in USER[rel].items()}
        run_cached(f'n4_user_burden_{tag}', lambda: n10.n1_seeds(runs, BUDGET, DELTA, MAIN_GROUP, release=rel))
        run_cached(f'n4_user_farming_{tag}', lambda: n10.n3_farming(runs, resc, BUDGET, DELTA, MAIN_GROUP, grid=(("own", 3, 0), ("generic", 3, 0)), release=rel))
        p_per = os.path.join(REV, f'n4_user_timing_{tag}_per_insider.csv')
        if not os.path.exists(p_per):
            per, ag = n10.n2_unilateral(runs, resc, BUDGET, DELTA, MAIN_GROUP, release=rel)
            per.to_csv(p_per, index=False); ag.to_csv(os.path.join(REV, f'n4_user_agreement_{tag}.csv'), index=False)
        per = pd.read_csv(p_per)
        S = n10.n2_summary(per); S.to_csv(os.path.join(REV, f'n4_user_timing_{tag}_summary.csv'), index=False)
        display(S[['detector', 'policy', 'attack', 'excess', 'lo', 'hi']].round(2))
    except Exception:
        traceback.print_exc(); FAILED.append(f'N4 {rel}')

## N5: Power of the permutation test

In [ ]:
for rel in RUNS:
    tag = rel.replace('.', '')
    try:
        sub = {d: v for d, v in RUNS[rel].items() if d in ('IF', 'AE', 'GBM', 'LR', 'LSTM')}
        N5 = run_cached(f'n5_power_{tag}', lambda: n10.n5_power(sub, BUDGET, MAIN_GROUP, release=rel))
        display(N5.round(3))
    except Exception:
        traceback.print_exc(); FAILED.append(f'N5 {rel}')

In [ ]:
import zipfile
zip_path = os.path.join(REV, 'fairirs_revision10_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(REV)):
        if fn.endswith('.csv'):
            z.write(os.path.join(REV, fn), fn)
print('Results zipped at', zip_path)
print('files:', sorted(f for f in os.listdir(REV) if f.endswith('.csv')))
print('ALL EXPERIMENTS FINISHED' if not FAILED else f'FAILED: {FAILED} (send the error messages above)')